### 標註 + 轉成user-level
##### 執行完接 python dataset.py 得到 ContextVecNet_Instagram
##### 再用 filter_dataset_no_blank_image.py 過續最低貼文數 得到 ContextVecNet_Instagram_flitered_new

In [3]:
#套用殘差修正 ，引入 Dropout 保護 ECR 模組，並將EPOCHS 從 10 增加到20
import os
import re
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, cohen_kappa_score, roc_auc_score
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

# ============================================
# 1. 全域設定與種子固定
# ============================================
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 路徑設定
DER_CKPT_PATH = "/content/drive/MyDrive/DECEN/best_sentence_level_bert_focal.pt"
POST_CSV = "/content/drive/MyDrive/DECEN/post_level_dataset_800_plus_syn.csv"
SAVE_DIR = "/content/drive/MyDrive/DECEN/"
PRECOMPUTED_PATH = os.path.join(SAVE_DIR, "precomputed_features.pt")
MODEL_NAME = "bert-base-chinese"

# 超參數
MAX_SENT_LEN = 128
MAX_SENTS_PER_POST = 20
EMO_DIM = 128
DD_HIDDEN = 128
BATCH_SIZE = 16 # 預算完後 Batch Size 可以調大
# LR = 1e-5 #2e-5
EPOCHS = 10

# ============================================
# 修改後的 Optimizer 設定 (分層學習率)
# ============================================


# ============================================
# 2. 損失函數：Focal Loss (對齊論文 alpha=0.5)
# ============================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.5, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt)**self.gamma * ce_loss
        return focal_loss.mean()

# ============================================
# 3. 模型定義 (DER 與修正後的 DECEN)
# ============================================

class DER_Inference(nn.Module):
    def __init__(self, n):
        super().__init__()
        self.bert = AutoModel.from_pretrained(MODEL_NAME)
        self.classifier = nn.Linear(768, n)
    def forward(self, ids, mask):
        out = self.bert(ids, mask)
        return self.classifier(out.last_hidden_state[:, 0])

class ECR_Module(nn.Module):
    """
    對齊 DECEN 論文的 Emotion-Context Relationship 模組
    公式: alpha = softmax(v^T * tanh(W_e * Re + W_c * Rc + b))
    """
    def __init__(self, emo_dim, ctx_dim):
        super().__init__()
        self.W_e = nn.Linear(emo_dim, emo_dim)
        self.W_c = nn.Linear(ctx_dim, emo_dim)
        self.v = nn.Linear(emo_dim, 1, bias=False)
        # self.dropout = nn.Dropout(0.1) # 加入這行

    def forward(self, Re, Rc, mask):
        # Re: (B, T, Emo_dim), Rc: (B, T, Ctx_dim)
        combined = torch.tanh(self.W_e(Re) + self.W_c(Rc)) # (B, T, Emo_dim)
        # combined = self.dropout(combined) # 加入這行
        score = self.v(combined).squeeze(-1) # (B, T)
        score = score.masked_fill(~mask, -1e9)
        alpha = torch.softmax(score, dim=1) # (B, T)
        return alpha

class DECEN_Model(nn.Module):
    def __init__(self, emo_class_num, emo_dim, dd_hidden):
        super().__init__()
        self.emo_proj = nn.Linear(emo_class_num, emo_dim)
        self.ctx_down = nn.Linear(768, emo_dim)

        self.ecr = ECR_Module(emo_dim, 768)
        self.bilstm = nn.LSTM(emo_dim, dd_hidden, batch_first=True, bidirectional=True)

        # 增加一個 LayerNorm 穩定特徵分布
        self.norm = nn.LayerNorm(emo_dim)
        self.classifier = nn.Linear(dd_hidden * 2, 2)

    def forward(self, Rc_pad, Re_probs_pad, lengths, mask, ablation="full"):
        Re_pad = self.emo_proj(Re_probs_pad) # (B, T, Emo_dim)
        Rc_low = self.ctx_down(Rc_pad)       # (B, T, Emo_dim)

        if ablation == "full":
            alpha = self.ecr(Re_pad, Rc_pad, mask) # (B, T)
            # 修正：殘差結構。Rc_low 是主幹，alpha 調節 Re_pad 的注入
            # 這樣即便 alpha 學不好，Rc_low 的訊息也能完整保留
            R_tilde = self.norm(Rc_low + (alpha.unsqueeze(-1) * Re_pad))

        elif ablation == "no_ecr":
            R_tilde = self.norm(Rc_low + Re_pad)
        elif ablation == "no_emotion":
            R_tilde = Rc_low
        elif ablation == "no_context":
            R_tilde = Re_pad
        else:
            R_tilde = Rc_low

        packed = nn.utils.rnn.pack_padded_sequence(R_tilde, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (hn, _) = self.bilstm(packed)
        feat = torch.cat((hn[-2], hn[-1]), dim=-1)
        return self.classifier(feat)

# ============================================
# 4. 預處理階段：提取特徵並儲存
# ============================================

SENT_SPLIT_RE = re.compile(r'([。！？!?\n]+)')

def split_sentences(text):
    text = str(text).strip()
    if not text: return [" "]
    parts = SENT_SPLIT_RE.split(text)
    sents, buf = [], ""
    for part in parts:
        if not part: continue
        buf += part
        if SENT_SPLIT_RE.fullmatch(part):
            sents.append(buf.strip()); buf = ""
    if buf.strip(): sents.append(buf.strip())
    return sents[:MAX_SENTS_PER_POST]

def precompute_features():
    if os.path.exists(PRECOMPUTED_PATH):
        print("偵測到預算特徵，跳過預處理...")
        return torch.load(PRECOMPUTED_PATH)

    print("開始預處理：提取 BERT 與 情緒向量 (此過程僅執行一次)...")
    df = pd.read_csv(POST_CSV)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    # 載入 DER 模型
    ckpt = torch.load(DER_CKPT_PATH, map_location=device)
    der_model = DER_Inference(len(ckpt["label2id"])).to(device)
    der_model.load_state_dict(ckpt["model_state"], strict=False)
    der_model.eval()

    # 載入基礎 BERT
    base_bert = AutoModel.from_pretrained(MODEL_NAME).to(device)
    base_bert.eval()

    all_data = []

    with torch.no_grad():
        for i, row in tqdm(df.iterrows(), total=len(df)):
            sents = split_sentences(row["caption"])
            num_s = len(sents)

            # Tokenization
            inputs = tokenizer(sents, truncation=True, padding=True, max_length=MAX_SENT_LEN, return_tensors="pt").to(device)

            # 1. 提取 Context Embedding (Rc)
            outputs = base_bert(**inputs)
            Rc = outputs.last_hidden_state[:, 0, :].cpu() # (num_s, 768)

            # 2. 提取 Emotion Probs (Re)
            der_logits = der_model(inputs["input_ids"], inputs["attention_mask"])
            Re_probs = torch.softmax(der_logits, dim=-1).cpu() # (num_s, emo_class_num)

            all_data.append({
                "Rc": Rc,
                "Re_probs": Re_probs,
                "y": int(row["y"]),
                "num_s": num_s
            })

    torch.save(all_data, PRECOMPUTED_PATH)
    return all_data

# ============================================
# 5. 訓練用 Dataset 與 DataLoader
# ============================================

class PrecomputedDataset(Dataset):
    def __init__(self, data_list):
        self.data = data_list
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx]

def precomputed_collate_fn(batch):
    B = len(batch)
    lengths = torch.tensor([b["num_s"] for b in batch], dtype=torch.long)
    T_max = lengths.max().item()

    ctx_dim = batch[0]["Rc"].shape[1]
    emo_dim = batch[0]["Re_probs"].shape[1]

    Rc_pad = torch.zeros((B, T_max, ctx_dim))
    Re_probs_pad = torch.zeros((B, T_max, emo_dim))
    mask = torch.zeros((B, T_max), dtype=torch.bool)
    ys = torch.tensor([b["y"] for b in batch], dtype=torch.long)

    for i, b in enumerate(batch):
        l = b["num_s"]
        Rc_pad[i, :l] = b["Rc"]
        Re_probs_pad[i, :l] = b["Re_probs"]
        mask[i, :l] = True

    return {
        "Rc": Rc_pad.to(device),
        "Re_probs": Re_probs_pad.to(device),
        "lengths": lengths.to(device),
        "mask": mask.to(device),
        "ys": ys.to(device)
    }


In [4]:
# 設定你想使用的特定模型路徑
BEST_MODEL_PATH = "D:\時間序列\DECEN\decen_v2_evolved.pt"

# 1. 初始化模型架構 (參數需與訓練時一致)
# emo_class_num 應根據你 precomputed_features 產出的維度設定
model = DECEN_Model(emo_class_num=5, emo_dim=128, dd_hidden=128).to(device)

# 2. 載入權重
if os.path.exists(BEST_MODEL_PATH):
    model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
    model.eval() # 切換至評估模式
    print(f"✅ 已成功載入模型！")
else:
    print(f"❌ 找不到檔案，請檢查路徑是否正確：{BEST_MODEL_PATH}")

✅ 已成功載入模型！


<ipython-input-4-b797d2ec2ace>:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))


In [6]:
import os, re, json, math, random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

# =========================
# 0) 基本設定
# =========================
SEED = 42
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# ====== 你要改的路徑 ======
MODEL_NAME = "bert-base-chinese"
DER_CKPT_PATH = "D:\\時間序列\\DECEN\\best_sentence_level_bert_focal.pt"
DECEN_WEIGHTS_PATH = "D:\\時間序列\\DECEN\\decen_v2_evolved.pt"   # 你進化後模型
INPUT_CSV = "D:/時間序列/depress_dataset/貼文資料.csv"  # 全量資料
OUT_DIR = "D:\\時間序列\\DECEN_TS"
os.makedirs(OUT_DIR, exist_ok=True)

# ====== 超參數（需與 DECEN 權重一致）======
MAX_SENT_LEN = 128
MAX_SENTS_PER_POST = 20
EMO_DIM = 128
DD_HIDDEN = 128

# ====== 推論設定 ======
T_CALIB = 0.1          # 你要保存的溫度版
BATCH_SENT = 32        # 一篇貼文內句子分批
USE_FP16 = True        # 有 GPU 建議 True；CPU 會自動忽略

# ====== pseudo-label 門檻（給高置信資料用）======
LABEL_SCORE = "p_tcal"   # "p_tcal" / "p_t1" / "margin"
TH_POS = 0.9
TH_NEG = 0.1

# ====== 要輸出的序列長度（一次產生多個 L）======
L_LIST = [20, 50, 100, 200]

# ====== 多變量是否加入 p_t1（會輸出兩份：有/無）======
EXPORT_MULTIVAR_WITH_P = True

# =========================
# 1) 斷句
# =========================
SENT_SPLIT_RE = re.compile(r'([。！？!?\n]+)')

def split_sentences(text):
    text = str(text).strip()
    if not text:
        return [" "]
    parts = SENT_SPLIT_RE.split(text)
    sents, buf = [], ""
    for part in parts:
        if not part:
            continue
        buf += part
        if SENT_SPLIT_RE.fullmatch(part):
            sents.append(buf.strip())
            buf = ""
    if buf.strip():
        sents.append(buf.strip())
    return sents[:MAX_SENTS_PER_POST]

# =========================
# 2) 模型：DER / ECR / DECEN（支援 return_alpha）
# =========================
class DER_Inference(nn.Module):
    def __init__(self, model_name, n_class):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.classifier = nn.Linear(768, n_class)

    def forward(self, ids, mask):
        out = self.bert(input_ids=ids, attention_mask=mask)
        return self.classifier(out.last_hidden_state[:, 0, :])  # (B, n_class)

class ECR_Module(nn.Module):
    def __init__(self, emo_dim, ctx_dim):
        super().__init__()
        self.W_e = nn.Linear(emo_dim, emo_dim)
        self.W_c = nn.Linear(ctx_dim, emo_dim)
        self.v = nn.Linear(emo_dim, 1, bias=False)

    def forward(self, Re, Rc, mask):
        combined = torch.tanh(self.W_e(Re) + self.W_c(Rc))
        score = self.v(combined).squeeze(-1)
        score = score.masked_fill(~mask, -1e9)
        alpha = torch.softmax(score, dim=1)
        return alpha

class DECEN_Model(nn.Module):
    def __init__(self, emo_class_num, emo_dim, dd_hidden):
        super().__init__()
        self.emo_proj = nn.Linear(emo_class_num, emo_dim)
        self.ctx_down = nn.Linear(768, emo_dim)

        self.ecr = ECR_Module(emo_dim, 768)
        self.bilstm = nn.LSTM(emo_dim, dd_hidden, batch_first=True, bidirectional=True)

        self.norm = nn.LayerNorm(emo_dim)
        self.classifier = nn.Linear(dd_hidden * 2, 2)

    def forward(self, Rc_pad, Re_probs_pad, lengths, mask, ablation="full", return_alpha=False):
        Re_pad = self.emo_proj(Re_probs_pad)  # (B,T,emo_dim)
        Rc_low = self.ctx_down(Rc_pad)        # (B,T,emo_dim)

        alpha = None
        if ablation == "full":
            alpha = self.ecr(Re_pad, Rc_pad, mask)  # (B,T)
            R_tilde = self.norm(Rc_low + alpha.unsqueeze(-1) * Re_pad)
        elif ablation == "no_ecr":
            R_tilde = self.norm(Rc_low + Re_pad)
        elif ablation == "no_emotion":
            R_tilde = Rc_low
        elif ablation == "no_context":
            R_tilde = Re_pad
        else:
            R_tilde = Rc_low

        packed = nn.utils.rnn.pack_padded_sequence(R_tilde, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (hn, _) = self.bilstm(packed)
        feat = torch.cat((hn[-2], hn[-1]), dim=-1)
        logits = self.classifier(feat)

        if return_alpha:
            return logits, alpha
        return logits

# =========================
# 3) 載入模型（不重訓，只推論）
# =========================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_bert = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()

ckpt = torch.load(DER_CKPT_PATH, map_location="cpu")
emo_class_num = len(ckpt["label2id"])
der_model = DER_Inference(MODEL_NAME, emo_class_num).to(device)
der_model.load_state_dict(ckpt["model_state"], strict=False)
der_model.eval()

decen_model = DECEN_Model(emo_class_num, EMO_DIM, DD_HIDDEN).to(device)
decen_model.load_state_dict(torch.load(DECEN_WEIGHTS_PATH, map_location=device), strict=True)
decen_model.eval()

print("emo_class_num:", emo_class_num)
print("✅ models loaded.")

# =========================
# 4) 單篇貼文 -> (三種分數 + multivar timestep 向量)
# =========================
@torch.no_grad()
def infer_one_post(caption: str):
    sents = split_sentences(caption)
    if len(sents) == 0:
        sents = [" "]
    T = len(sents)

    Rc_list, Re_list = [], []
    # autocast（GPU fp16）可減顯存/加速；CPU 會被關閉
    autocast_ctx = torch.cuda.amp.autocast(enabled=(USE_FP16 and device.type == "cuda"))

    for i in range(0, T, BATCH_SENT):
        chunk = sents[i:i+BATCH_SENT]
        inputs = tokenizer(
            chunk, truncation=True, padding=True,
            max_length=MAX_SENT_LEN, return_tensors="pt"
        ).to(device)

        with autocast_ctx:
            out = base_bert(**inputs)
            Rc = out.last_hidden_state[:, 0, :]  # (k,768)
            der_logits = der_model(inputs["input_ids"], inputs["attention_mask"])
            Re_probs = torch.softmax(der_logits, dim=-1)  # (k,C)

        Rc_list.append(Rc.float())
        Re_list.append(Re_probs.float())

    Rc_sent = torch.cat(Rc_list, dim=0)  # (T,768)
    Re_sent = torch.cat(Re_list, dim=0)  # (T,C)

    # 組 batch=1 給 DECEN 取 logits 與 alpha
    Rc_pad = Rc_sent.unsqueeze(0)                           # (1,T,768)
    Re_pad = Re_sent.unsqueeze(0)                           # (1,T,C)
    lengths = torch.tensor([T], dtype=torch.long).to(device)
    mask = torch.ones((1, T), dtype=torch.bool).to(device)

    logits, alpha = decen_model(Rc_pad, Re_pad, lengths, mask, ablation="full", return_alpha=True)
    # logits: (1,2), alpha: (1,T)
    alpha = alpha.squeeze(0)                                # (T,)
    alpha_norm = alpha / (alpha.sum() + 1e-8)

    # 三種分數
    margin = (logits[:, 1] - logits[:, 0]).item()
    p_t1 = torch.softmax(logits, dim=-1)[:, 1].item()
    p_tcal = torch.softmax(logits / T_CALIB, dim=-1)[:, 1].item()

    # 句子層 -> 貼文層聚合
    Rc_post = (alpha_norm.unsqueeze(-1) * Rc_sent).sum(dim=0, keepdim=True)   # (1,768)
    Re_post = (alpha_norm.unsqueeze(-1) * Re_sent).sum(dim=0, keepdim=True)   # (1,C)

    Rc_down = decen_model.ctx_down(Rc_post)                # (1,128)
    alpha_max = alpha.max().view(1, 1)                     # (1,1)

    x_no_p = torch.cat([Rc_down, Re_post, alpha_max], dim=-1).squeeze(0).cpu().numpy().astype(np.float32)
    x_with_p = np.concatenate([x_no_p, np.array([p_t1], dtype=np.float32)], axis=0)

    return p_t1, p_tcal, margin, x_no_p, x_with_p

# =========================
# 5) 跑全量推論：同時輸出單變量分數 + 多變量 timestep
# =========================
df = pd.read_csv(INPUT_CSV)

need_cols = ["username", "taken_at", "caption", "post_id"]
for c in need_cols:
    if c not in df.columns:
        raise ValueError(f"缺欄位: {c}")

df["taken_at"] = pd.to_datetime(df["taken_at"], errors="coerce")
df = df.sort_values(["username", "taken_at"]).reset_index(drop=True)

all_p_t1, all_p_tcal, all_margin = [], [], []
all_x_no_p, all_x_with_p = [], []

print(f"🚀 開始推論並抽特徵：共 {len(df):,} 筆")
for _, r in tqdm(df.iterrows(), total=len(df)):
    p_t1, p_tcal, margin, x_no_p, x_with_p = infer_one_post(r["caption"])
    all_p_t1.append(p_t1)
    all_p_tcal.append(p_tcal)
    all_margin.append(margin)
    all_x_no_p.append(x_no_p)
    all_x_with_p.append(x_with_p)

df["p_depression_t1"] = all_p_t1
df["p_depression_tcal"] = all_p_tcal
df["logit_margin"] = all_margin

# pseudo_label：用你指定的 score
if LABEL_SCORE == "p_tcal":
    score = df["p_depression_tcal"].astype(float)
elif LABEL_SCORE == "p_t1":
    score = df["p_depression_t1"].astype(float)
else:
    score = df["logit_margin"].astype(float)

df["pseudo_label"] = -1
df.loc[score > TH_POS, "pseudo_label"] = 1
df.loc[score < TH_NEG, "pseudo_label"] = 0

# 存一份完整 CSV（含三分數）
CSV_OUT = os.path.join(OUT_DIR, "auto_labeled_with_scores.csv")
df.to_csv(CSV_OUT, index=False, encoding="utf-8-sig")
print("✅ saved:", CSV_OUT)



device: cuda


c:\Users\Angle\Anaconda3\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
<ipython-input-6-a1965ca8e2d9>:148: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode u

emo_class_num: 5
✅ models loaded.
🚀 開始推論並抽特徵：共 43,525 筆


  0%|          | 0/43525 [00:00<?, ?it/s]<ipython-input-6-a1965ca8e2d9>:173: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  autocast_ctx = torch.cuda.amp.autocast(enabled=(USE_FP16 and device.type == "cuda"))
c:\Users\Angle\Anaconda3\lib\site-packages\transformers\models\bert\modeling_bert.py:439: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
100%|██████████| 43525/43525 [21:31<00:00, 33.70it/s]  


✅ saved: D:\時間序列\DECEN_TS\auto_labeled_with_scores.csv


In [25]:
import os, re, json, math, random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

# =========================
# 0) 基本設定
# =========================
SEED = 42
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# ====== 你要改的路徑 ======
MODEL_NAME = "bert-base-chinese"
DER_CKPT_PATH = "D:\\時間序列\\DECEN\\best_sentence_level_bert_focal.pt"
DECEN_WEIGHTS_PATH = "D:\\時間序列\\DECEN\\decen_v2_evolved.pt"   # 你進化後模型
INPUT_CSV = "D:\\時間序列\\depress_dataset\\非憂鬱標籤貼文.csv"  # 全量資料
OUT_DIR = "D:\\時間序列\\DECEN_TS"
os.makedirs(OUT_DIR, exist_ok=True)

# ====== 超參數（需與 DECEN 權重一致）======
MAX_SENT_LEN = 128
MAX_SENTS_PER_POST = 20
EMO_DIM = 128
DD_HIDDEN = 128

# ====== 推論設定 ======
T_CALIB = 0.1          # 你要保存的溫度版
BATCH_SENT = 32        # 一篇貼文內句子分批
USE_FP16 = True        # 有 GPU 建議 True；CPU 會自動忽略

# ====== pseudo-label 門檻（給高置信資料用）======
LABEL_SCORE = "p_tcal"   # "p_tcal" / "p_t1" / "margin"
TH_POS = 0.9
TH_NEG = 0.1

# ====== 要輸出的序列長度（一次產生多個 L）======
L_LIST = [20, 50, 100, 200]

# ====== 多變量是否加入 p_t1（會輸出兩份：有/無）======
EXPORT_MULTIVAR_WITH_P = True

# =========================
# 1) 斷句
# =========================
SENT_SPLIT_RE = re.compile(r'([。！？!?\n]+)')

def split_sentences(text):
    text = str(text).strip()
    if not text:
        return [" "]
    parts = SENT_SPLIT_RE.split(text)
    sents, buf = [], ""
    for part in parts:
        if not part:
            continue
        buf += part
        if SENT_SPLIT_RE.fullmatch(part):
            sents.append(buf.strip())
            buf = ""
    if buf.strip():
        sents.append(buf.strip())
    return sents[:MAX_SENTS_PER_POST]

# =========================
# 2) 模型：DER / ECR / DECEN（支援 return_alpha）
# =========================
class DER_Inference(nn.Module):
    def __init__(self, model_name, n_class):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.classifier = nn.Linear(768, n_class)

    def forward(self, ids, mask):
        out = self.bert(input_ids=ids, attention_mask=mask)
        return self.classifier(out.last_hidden_state[:, 0, :])  # (B, n_class)

class ECR_Module(nn.Module):
    def __init__(self, emo_dim, ctx_dim):
        super().__init__()
        self.W_e = nn.Linear(emo_dim, emo_dim)
        self.W_c = nn.Linear(ctx_dim, emo_dim)
        self.v = nn.Linear(emo_dim, 1, bias=False)

    def forward(self, Re, Rc, mask):
        combined = torch.tanh(self.W_e(Re) + self.W_c(Rc))
        score = self.v(combined).squeeze(-1)
        score = score.masked_fill(~mask, -1e9)
        alpha = torch.softmax(score, dim=1)
        return alpha

class DECEN_Model(nn.Module):
    def __init__(self, emo_class_num, emo_dim, dd_hidden):
        super().__init__()
        self.emo_proj = nn.Linear(emo_class_num, emo_dim)
        self.ctx_down = nn.Linear(768, emo_dim)

        self.ecr = ECR_Module(emo_dim, 768)
        self.bilstm = nn.LSTM(emo_dim, dd_hidden, batch_first=True, bidirectional=True)

        self.norm = nn.LayerNorm(emo_dim)
        self.classifier = nn.Linear(dd_hidden * 2, 2)

    def forward(self, Rc_pad, Re_probs_pad, lengths, mask, ablation="full", return_alpha=False):
        Re_pad = self.emo_proj(Re_probs_pad)  # (B,T,emo_dim)
        Rc_low = self.ctx_down(Rc_pad)        # (B,T,emo_dim)

        alpha = None
        if ablation == "full":
            alpha = self.ecr(Re_pad, Rc_pad, mask)  # (B,T)
            R_tilde = self.norm(Rc_low + alpha.unsqueeze(-1) * Re_pad)
        elif ablation == "no_ecr":
            R_tilde = self.norm(Rc_low + Re_pad)
        elif ablation == "no_emotion":
            R_tilde = Rc_low
        elif ablation == "no_context":
            R_tilde = Re_pad
        else:
            R_tilde = Rc_low

        packed = nn.utils.rnn.pack_padded_sequence(R_tilde, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (hn, _) = self.bilstm(packed)
        feat = torch.cat((hn[-2], hn[-1]), dim=-1)
        logits = self.classifier(feat)

        if return_alpha:
            return logits, alpha
        return logits

# =========================
# 3) 載入模型（不重訓，只推論）
# =========================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_bert = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()

ckpt = torch.load(DER_CKPT_PATH, map_location="cpu")
emo_class_num = len(ckpt["label2id"])
der_model = DER_Inference(MODEL_NAME, emo_class_num).to(device)
der_model.load_state_dict(ckpt["model_state"], strict=False)
der_model.eval()

decen_model = DECEN_Model(emo_class_num, EMO_DIM, DD_HIDDEN).to(device)
decen_model.load_state_dict(torch.load(DECEN_WEIGHTS_PATH, map_location=device), strict=True)
decen_model.eval()

print("emo_class_num:", emo_class_num)
print("✅ models loaded.")

# =========================
# 4) 單篇貼文 -> (三種分數 + multivar timestep 向量)
# =========================
@torch.no_grad()
def infer_one_post(caption: str):
    sents = split_sentences(caption)
    if len(sents) == 0:
        sents = [" "]
    T = len(sents)

    Rc_list, Re_list = [], []
    # autocast（GPU fp16）可減顯存/加速；CPU 會被關閉
    autocast_ctx = torch.cuda.amp.autocast(enabled=(USE_FP16 and device.type == "cuda"))

    for i in range(0, T, BATCH_SENT):
        chunk = sents[i:i+BATCH_SENT]
        inputs = tokenizer(
            chunk, truncation=True, padding=True,
            max_length=MAX_SENT_LEN, return_tensors="pt"
        ).to(device)

        with autocast_ctx:
            out = base_bert(**inputs)
            Rc = out.last_hidden_state[:, 0, :]  # (k,768)
            der_logits = der_model(inputs["input_ids"], inputs["attention_mask"])
            Re_probs = torch.softmax(der_logits, dim=-1)  # (k,C)

        Rc_list.append(Rc.float())
        Re_list.append(Re_probs.float())

    Rc_sent = torch.cat(Rc_list, dim=0)  # (T,768)
    Re_sent = torch.cat(Re_list, dim=0)  # (T,C)

    # 組 batch=1 給 DECEN 取 logits 與 alpha
    Rc_pad = Rc_sent.unsqueeze(0)                           # (1,T,768)
    Re_pad = Re_sent.unsqueeze(0)                           # (1,T,C)
    lengths = torch.tensor([T], dtype=torch.long).to(device)
    mask = torch.ones((1, T), dtype=torch.bool).to(device)

    logits, alpha = decen_model(Rc_pad, Re_pad, lengths, mask, ablation="full", return_alpha=True)
    # logits: (1,2), alpha: (1,T)
    alpha = alpha.squeeze(0)                                # (T,)
    alpha_norm = alpha / (alpha.sum() + 1e-8)

    # 三種分數
    margin = (logits[:, 1] - logits[:, 0]).item()
    p_t1 = torch.softmax(logits, dim=-1)[:, 1].item()
    p_tcal = torch.softmax(logits / T_CALIB, dim=-1)[:, 1].item()

    # 句子層 -> 貼文層聚合
    Rc_post = (alpha_norm.unsqueeze(-1) * Rc_sent).sum(dim=0, keepdim=True)   # (1,768)
    Re_post = (alpha_norm.unsqueeze(-1) * Re_sent).sum(dim=0, keepdim=True)   # (1,C)

    Rc_down = decen_model.ctx_down(Rc_post)                # (1,128)
    alpha_max = alpha.max().view(1, 1)                     # (1,1)

    x_no_p = torch.cat([Rc_down, Re_post, alpha_max], dim=-1).squeeze(0).cpu().numpy().astype(np.float32)
    x_with_p = np.concatenate([x_no_p, np.array([p_t1], dtype=np.float32)], axis=0)

    return p_t1, p_tcal, margin, x_no_p, x_with_p

# =========================
# 5) 跑全量推論：同時輸出單變量分數 + 多變量 timestep
# =========================
df = pd.read_csv(INPUT_CSV)

need_cols = ["username", "taken_at", "caption", "post_id"]
for c in need_cols:
    if c not in df.columns:
        raise ValueError(f"缺欄位: {c}")

df["taken_at"] = pd.to_datetime(df["taken_at"], errors="coerce")
df = df.sort_values(["username", "taken_at"]).reset_index(drop=True)

all_p_t1, all_p_tcal, all_margin = [], [], []
all_x_no_p, all_x_with_p = [], []

print(f"🚀 開始推論並抽特徵：共 {len(df):,} 筆")
for _, r in tqdm(df.iterrows(), total=len(df)):
    p_t1, p_tcal, margin, x_no_p, x_with_p = infer_one_post(r["caption"])
    all_p_t1.append(p_t1)
    all_p_tcal.append(p_tcal)
    all_margin.append(margin)
    all_x_no_p.append(x_no_p)
    all_x_with_p.append(x_with_p)

df["p_depression_t1"] = all_p_t1
df["p_depression_tcal"] = all_p_tcal
df["logit_margin"] = all_margin

# pseudo_label：用你指定的 score
if LABEL_SCORE == "p_tcal":
    score = df["p_depression_tcal"].astype(float)
elif LABEL_SCORE == "p_t1":
    score = df["p_depression_t1"].astype(float)
else:
    score = df["logit_margin"].astype(float)

df["pseudo_label"] = -1
df.loc[score > TH_POS, "pseudo_label"] = 1
df.loc[score < TH_NEG, "pseudo_label"] = 0

# 存一份完整 CSV（含三分數）
CSV_OUT = os.path.join(OUT_DIR, "auto_labeled_with_scores_non.csv")
df.to_csv(CSV_OUT, index=False, encoding="utf-8-sig")
print("✅ saved:", CSV_OUT)



device: cuda


<ipython-input-25-8299f14644ac>:148: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(DER_CKPT_PATH, map_location="cpu")
<ipython-input-25-8299f14644ac>:155: 

emo_class_num: 5
✅ models loaded.
🚀 開始推論並抽特徵：共 81,236 筆


  0%|          | 0/81236 [00:00<?, ?it/s]<ipython-input-25-8299f14644ac>:173: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  autocast_ctx = torch.cuda.amp.autocast(enabled=(USE_FP16 and device.type == "cuda"))
100%|██████████| 81236/81236 [38:18<00:00, 35.34it/s] 


✅ saved: D:\時間序列\DECEN_TS\auto_labeled_with_scores_non.csv


In [10]:
import pandas as pd
import numpy as np

# =========================================================
# 1. 參數設定
# =========================================================
df_non = pd.read_csv("D:\\時間序列\\DECEN_TS\\auto_labeled_with_scores_non.csv")
df_dep = pd.read_csv("D:\\時間序列\\DECEN_TS\\auto_labeled_with_scores.csv")

#合併兩份資料

df = pd.concat([df_non, df_dep], ignore_index=True)
#保存合併後的資料
df.to_csv("D:\\時間序列\\DECEN_TS\\auto_labeled_with_scores_combined.csv", index=False)

INPUT_CSV = "D:\\時間序列\\DECEN_TS\\auto_labeled_with_scores_combined.csv"
OUTPUT_POST_WITH_USER_LABEL = "D:\\時間序列\\labeled\\post_with_user_label.csv"
OUTPUT_USER_SUMMARY = "D:\\時間序列\\labeled\\user_summary_14d.csv"
OUTPUT_POSITIVE_WINDOWS = "D:\\時間序列\\labeled\\positive_windows_14d.csv"

# 時間範圍
START_DATE = "2021-01-01 00:00:00+00:00"
END_DATE   = "2026-06-01 23:59:59+00:00"

# 視窗設定
WINDOW_DAYS = 14

# 陽性視窗條件（主方案）
MIN_POSTS_IN_WINDOW = 3
MIN_HIGH_RISK_POSTS = 2
MEAN_SCORE_THRESHOLD = 0.6

# 高風險貼文定義
# 若 pseudo_label == 1 則視為高風險
# 或 p_depression_tcal >= HIGH_RISK_SCORE_THRESHOLD 也視為高風險
HIGH_RISK_SCORE_THRESHOLD = 0.7

# 若某視窗中沒有 pseudo_label，可直接依 score 判斷
USE_PSEUDO_LABEL_IF_AVAILABLE = True

# =========================================================
# 2. 讀檔與基本清理
# =========================================================
df = pd.read_csv(INPUT_CSV)

df.columns = [c.strip() for c in df.columns]

required_cols = ["username", "post_id", "taken_at", "p_depression_tcal"]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"缺少必要欄位: {col}")

# 時間轉 datetime
df["taken_at"] = pd.to_datetime(df["taken_at"], utc=True, errors="coerce")

# 數值欄位轉型
if "pseudo_label" in df.columns:
    df["pseudo_label"] = pd.to_numeric(df["pseudo_label"], errors="coerce")

df["p_depression_tcal"] = pd.to_numeric(df["p_depression_tcal"], errors="coerce")

# 去除關鍵缺失
df = df.dropna(subset=["username", "post_id", "taken_at", "p_depression_tcal"]).copy()

# 去重
df = df.drop_duplicates(subset=["username", "post_id"]).copy()

# 限定時間範圍
start_ts = pd.Timestamp(START_DATE)
end_ts = pd.Timestamp(END_DATE)
df = df[(df["taken_at"] >= start_ts) & (df["taken_at"] <= end_ts)].copy()

# 排序
df = df.sort_values(["username", "taken_at", "post_id"]).reset_index(drop=True)

# =========================================================
# 3. 定義高風險貼文
# =========================================================
def is_high_risk_post(row,
                      use_pseudo_label_if_available=True,
                      high_risk_score_threshold=0.7):
    """
    高風險貼文判定：
    1) 若 pseudo_label 可用且 use_pseudo_label_if_available=True，則 pseudo_label == 1 視為高風險
    2) 否則 p_depression_tcal >= threshold 視為高風險
    """
    if use_pseudo_label_if_available and "pseudo_label" in row.index and pd.notna(row["pseudo_label"]):
        return int(row["pseudo_label"] == 1)
    return int(pd.notna(row["p_depression_tcal"]) and row["p_depression_tcal"] >= high_risk_score_threshold)

df["is_high_risk"] = df.apply(
    lambda r: is_high_risk_post(
        r,
        use_pseudo_label_if_available=USE_PSEUDO_LABEL_IF_AVAILABLE,
        high_risk_score_threshold=HIGH_RISK_SCORE_THRESHOLD
    ),
    axis=1
)

# =========================================================
# 4. 14 天滑動視窗標註函式
# =========================================================
def label_users_by_sliding_window(
    df_user,
    window_days=14,
    min_posts=3,
    min_high_risk=2,
    mean_score_threshold=0.6
):
    """
    對單一 user 的貼文做 14 天滑動視窗標註

    回傳：
    - user_label: 0/1
    - user_summary: dict
    - positive_windows: list[dict]
    """
    g = df_user.sort_values("taken_at").copy().reset_index(drop=True)

    if len(g) == 0:
        return 0, None, []

    positive_windows = []
    n = len(g)

    # 逐篇貼文當視窗起點
    for i in range(n):
        window_start = g.loc[i, "taken_at"]
        window_end = window_start + pd.Timedelta(days=window_days)

        window_df = g[(g["taken_at"] >= window_start) & (g["taken_at"] < window_end)].copy()

        n_posts = len(window_df)
        n_high = int(window_df["is_high_risk"].sum())
        mean_score = float(window_df["p_depression_tcal"].mean()) if n_posts > 0 else np.nan

        is_positive_window = (
            (n_posts >= min_posts) and
            (n_high >= min_high_risk) and
            (mean_score >= mean_score_threshold)
        )

        if is_positive_window:
            positive_windows.append({
                "username": g.loc[0, "username"],
                "window_start": window_start.isoformat(),
                "window_end": window_end.isoformat(),
                "n_posts": int(n_posts),
                "n_high_risk_posts": int(n_high),
                "mean_p_depression_tcal": float(mean_score),
                "post_ids": list(window_df["post_id"].astype(str)),
            })

    user_label = int(len(positive_windows) > 0)

    user_summary = {
        "username": g.loc[0, "username"],
        "user_label": user_label,
        "num_posts_total": int(len(g)),
        "num_high_risk_posts_total": int(g["is_high_risk"].sum()),
        "mean_p_depression_tcal_total": float(g["p_depression_tcal"].mean()),
        "num_positive_windows": int(len(positive_windows)),
        "first_post_time": g["taken_at"].min().isoformat(),
        "last_post_time": g["taken_at"].max().isoformat(),
    }

    return user_label, user_summary, positive_windows

# =========================================================
# 5. 套用到所有使用者
# =========================================================
user_summaries = []
all_positive_windows = []
user_label_map = {}

for username, g in df.groupby("username"):
    user_label, user_summary, positive_windows = label_users_by_sliding_window(
        g,
        window_days=WINDOW_DAYS,
        min_posts=MIN_POSTS_IN_WINDOW,
        min_high_risk=MIN_HIGH_RISK_POSTS,
        mean_score_threshold=MEAN_SCORE_THRESHOLD
    )

    user_label_map[username] = user_label
    user_summaries.append(user_summary)

    if len(positive_windows) > 0:
        all_positive_windows.extend(positive_windows)

# =========================================================
# 6. 回寫 user_label 到原始 post-level 資料
# =========================================================
df["user_label"] = df["username"].map(user_label_map)

user_summary_df = pd.DataFrame(user_summaries)
positive_windows_df = pd.DataFrame(all_positive_windows)

# =========================================================
# 7. 輸出結果
# =========================================================
df.to_csv(OUTPUT_POST_WITH_USER_LABEL, index=False, encoding="utf-8-sig")
user_summary_df.to_csv(OUTPUT_USER_SUMMARY, index=False, encoding="utf-8-sig")

if len(positive_windows_df) > 0:
    positive_windows_df.to_csv(OUTPUT_POSITIVE_WINDOWS, index=False, encoding="utf-8-sig")
else:
    # 若沒有陽性視窗，也輸出空檔方便檢查
    pd.DataFrame(columns=[
        "username", "window_start", "window_end",
        "n_posts", "n_high_risk_posts", "mean_p_depression_tcal", "post_ids"
    ]).to_csv(OUTPUT_POSITIVE_WINDOWS, index=False, encoding="utf-8-sig")

# =========================================================
# 8. 顯示摘要
# =========================================================
print("=== 完成 14 天滑動視窗 user-level 標註 ===")
print(f"貼文總數: {len(df)}")
print(f"使用者總數: {df['username'].nunique()}")
print(f"陽性使用者數: {int(user_summary_df['user_label'].sum())}")
print(f"陰性使用者數: {int((user_summary_df['user_label'] == 0).sum())}")

print("\n=== user_summary 前幾列 ===")
print(user_summary_df.head())

print("\n=== positive_windows 前幾列 ===")
print(positive_windows_df.head())

print("\n已輸出：")
print(OUTPUT_POST_WITH_USER_LABEL)
print(OUTPUT_USER_SUMMARY)
print(OUTPUT_POSITIVE_WINDOWS)

c:\Users\Angle\Anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3165: DtypeWarning: Columns (10,15) have mixed types.Specify dtype option on import or set low_memory=False.
  has_raised = await self.run_ast_nodes(code_ast.body, cell_name,


=== 完成 14 天滑動視窗 user-level 標註 ===
貼文總數: 100526
使用者總數: 362
陽性使用者數: 150
陰性使用者數: 212

=== user_summary 前幾列 ===
            username  user_label  num_posts_total  num_high_risk_posts_total  \
0         156.melody           0              935                          0   
1  163______________           1              200                        107   
2            30.yuna           0              200                          7   
3       617_lyc_0126           0               18                          4   
4         9.willis.1           0              200                          6   

   mean_p_depression_tcal_total  num_positive_windows  \
0                  6.032290e-09                     0   
1                  5.664943e-01                    89   
2                  4.398552e-02                     0   
3                  2.256549e-01                     0   
4                  5.114039e-02                     0   

             first_post_time             last_post_time  
0  2022-10

In [32]:
from pathlib import Path
import pandas as pd

root = Path(r"D:\時間序列\ContextVecNet_Instagram_filtered")

rows = []

for split in ["positive", "negative"]:
    for user_dir in (root / split).iterdir():
        if not user_dir.is_dir():
            continue

        meta_path = user_dir / "metadata.csv"
        if not meta_path.exists():
            continue

        df = pd.read_csv(meta_path)
        n_posts = len(df)
        n_img = df["image_path"].fillna("").astype(str).str.len().gt(0).sum()

        rows.append({
            "split": split,
            "user": user_dir.name,
            "n_posts": n_posts,
            "n_posts_with_image_path": n_img,
            "image_coverage": n_img / n_posts if n_posts else 0
        })

summary = pd.DataFrame(rows)
print(summary.describe())
print(summary.sort_values("image_coverage").head(20))

summary.to_csv(root / "image_coverage_summary.csv", index=False, encoding="utf-8-sig")

           n_posts  n_posts_with_image_path  image_coverage
count   252.000000               252.000000           252.0
mean    239.123016               239.123016             1.0
std     346.975327               346.975327             0.0
min      20.000000                20.000000             1.0
25%      61.750000                61.750000             1.0
50%     144.000000               144.000000             1.0
75%     207.750000               207.750000             1.0
max    2751.000000              2751.000000             1.0
        split                    user  n_posts  n_posts_with_image_path  \
0    positive       163______________      198                      198   
160  negative              hao_emo520      496                      496   
161  negative             hellotuday_     1603                     1603   
162  negative             hi_lo_memes     2751                     2751   
163  negative             hongpeishuo       21                       21   
164  negat

In [34]:
from pathlib import Path
import pandas as pd

DST_ROOT = Path(r"D:\時間序列\ContextVecNet_Instagram_filtered_new")

results = []
for split in ["positive", "negative"]:
    split_dir = DST_ROOT / split
    if not split_dir.exists():
        print(f"Warning: {split_dir} not found")
        continue

    user_dirs = [p for p in split_dir.iterdir() if p.is_dir()]
    user_counts = []
    for user_dir in user_dirs:
        metadata_path = user_dir / "metadata.csv"
        if metadata_path.exists():
            df = pd.read_csv(metadata_path)
            user_counts.append(len(df))
        else:
            # 如果沒有 metadata.csv，改用 timeline.txt 估算
            timeline_path = user_dir / "timeline.txt"
            if timeline_path.exists():
                with open(timeline_path, "r", encoding="utf-8", errors="ignore") as f:
                    user_counts.append(sum(1 for _ in f))
            else:
                user_counts.append(0)

    user_count = len(user_counts)
    total_posts = sum(user_counts)
    avg_posts = total_posts / user_count if user_count > 0 else 0.0
    results.append({
        "split": split,
        "user_count": user_count,
        "total_posts": total_posts,
        "avg_posts_per_user": avg_posts,
        "min_posts_per_user": min(user_counts) if user_counts else 0,
        "max_posts_per_user": max(user_counts) if user_counts else 0,
        "median_posts_per_user": pd.Series(user_counts).median() if user_counts else 0,
    })

summary_df = pd.DataFrame(results)
print(summary_df.to_string(index=False))

overall = {
    "split": "all",
    "user_count": int(summary_df["user_count"].sum()),
    "total_posts": int(summary_df["total_posts"].sum()),
    "avg_posts_per_user": float(summary_df["total_posts"].sum() / summary_df["user_count"].sum())
    if summary_df["user_count"].sum() > 0 else 0.0,
}
print("\nOverall:")
print(overall)

   split  user_count  total_posts  avg_posts_per_user  min_posts_per_user  max_posts_per_user  median_posts_per_user
positive         130        23367          179.746154                  20                 908                  135.5
negative         165        44201          267.884848                  20                2751                  176.0

Overall:
{'split': 'all', 'user_count': 295, 'total_posts': 67568, 'avg_posts_per_user': 229.04406779661016}


#### 計算資料集padding

In [1]:
from pathlib import Path
import pandas as pd

root = Path(r"D:\時間序列\ContextVecNet_Instagram_filtered_new")
output_path = root / "image_coverage_summary.csv"

rows = []
for split_dir in ["positive", "negative"]:
    split_path = root / split_dir
    if not split_path.exists():
        continue

    for user_dir in sorted(split_path.iterdir()):
        if not user_dir.is_dir():
            continue

        metadata_path = user_dir / "metadata.csv"
        if not metadata_path.exists():
            continue

        df = pd.read_csv(metadata_path)
        df.columns = df.columns.str.strip()

        n_posts = len(df)
        if n_posts == 0:
            continue

        def has_image_path(x):
            if pd.isna(x):
                return False
            s = str(x).strip()
            return len(s) > 0

        n_posts_with_image_path = df["image_path"].apply(has_image_path).sum()
        image_coverage = n_posts_with_image_path / n_posts

        rows.append({
            "split": split_dir,
            "user": user_dir.name,
            "n_posts": n_posts,
            "n_posts_with_image_path": n_posts_with_image_path,
            "image_coverage": image_coverage,
        })

summary_df = pd.DataFrame(rows)
summary_df.to_csv(output_path, index=False)
print("Saved:", output_path)
print(summary_df.head())

Saved: D:\時間序列\ContextVecNet_Instagram_filtered_new\image_coverage_summary.csv
      split               user  n_posts  n_posts_with_image_path  \
0  positive  163______________      200                      200   
1  positive     _feifei_words_       98                       98   
2  positive       _kana.words_       71                       71   
3  positive         _midleave_      741                      741   
4  positive      _t_depression      106                      106   

   image_coverage  
0             1.0  
1             1.0  
2             1.0  
3             1.0  
4             1.0  


In [1]:
import pandas as pd

summary_path = r"D:\時間序列\ContextVecNet_Instagram_filtered_new\image_coverage_summary.csv"
df = pd.read_csv(summary_path)

window_sizes = [16, 32, 64, 128]

total_users = df["user"].nunique()

for w in window_sizes:
    valid_users = df[df["n_posts_with_image_path"] >= w]["user"].nunique()
    ratio = valid_users / total_users
    avg_coverage = (df["n_posts_with_image_path"] / w).clip(upper=1).mean()
    print(f"window_size={w}:")
    print(f"  valid users = {valid_users} / {total_users} ({ratio:.2%})")
    print(f"  average coverage ratio = {avg_coverage:.2%}")
    print()

window_size=16:
  valid users = 292 / 292 (100.00%)
  average coverage ratio = 100.00%

window_size=32:
  valid users = 270 / 292 (92.47%)
  average coverage ratio = 98.41%

window_size=64:
  valid users = 220 / 292 (75.34%)
  average coverage ratio = 90.91%

window_size=128:
  valid users = 162 / 292 (55.48%)
  average coverage ratio = 77.38%



In [2]:
for split in df["split"].unique():
    df_split = df[df["split"] == split]
    total_split_users = df_split["user"].nunique()

    print(f"split={split}")
    for w in window_sizes:
        valid_users = df_split[df_split["n_posts_with_image_path"] >= w]["user"].nunique()
        ratio = valid_users / total_split_users
        print(f"  window_size={w}: {valid_users} / {total_split_users} ({ratio:.2%})")
    print()

split=positive
  window_size=16: 130 / 130 (100.00%)
  window_size=32: 123 / 130 (94.62%)
  window_size=64: 100 / 130 (76.92%)
  window_size=128: 66 / 130 (50.77%)

split=negative
  window_size=16: 162 / 162 (100.00%)
  window_size=32: 147 / 162 (90.74%)
  window_size=64: 120 / 162 (74.07%)
  window_size=128: 96 / 162 (59.26%)



#### 轉成使用者標籤的敏感度分析

In [12]:
# =========================================================
# 14 天滑動視窗主規則 + 一次跑多組敏感度分析 + 輸出比較表
# =========================================================
# 功能：
# 1. 保留 2021-01-01 ~ 2025-12-31 的所有貼文
# 2. 以 username 為單位做 14 天滑動視窗標註
# 3. 一次跑多組參數組合（敏感度分析）
# 4. 輸出：
#    - sensitivity_summary.csv         : 每組參數的整體比較表
#    - user_labels_all_schemes.csv     : 每位使用者在每組方案下的 label
#    - positive_windows_<scheme>.csv   : 各方案的陽性視窗明細
#    - post_with_user_label_<scheme>.csv : 各方案下回寫 user_label 的 post-level 資料
# =========================================================

import pandas as pd
import numpy as np
from pathlib import Path

# =========================
# 1. 路徑設定
# =========================
INPUT_CSV = "D:\\時間序列\\DECEN_TS\\auto_labeled_with_scores_combined.csv"  # 改成你的檔案路徑
OUTPUT_DIR = Path("D:\\時間序列\\labeled\\label_sensitivity_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# 2. 基本參數
# =========================
START_DATE = "2021-01-01 00:00:00+00:00"
END_DATE   = "2026-06-01 23:59:59+00:00"
WINDOW_DAYS = 14

# 高風險貼文定義
HIGH_RISK_SCORE_THRESHOLD = 0.7
USE_PSEUDO_LABEL_IF_AVAILABLE = True

# 一次跑多組敏感度分析
# scheme_name 可自行改
PARAM_GRID = [
    {"scheme_name": "A_loose",   "min_posts": 2, "min_high_risk": 1, "mean_score_threshold": 0.60},
    {"scheme_name": "B_main",    "min_posts": 3, "min_high_risk": 2, "mean_score_threshold": 0.60},
    {"scheme_name": "C_strict",  "min_posts": 4, "min_high_risk": 2, "mean_score_threshold": 0.65},
]

# =========================
# 3. 讀取資料
# =========================
df = pd.read_csv(INPUT_CSV)
df.columns = [c.strip() for c in df.columns]

required_cols = ["username", "post_id", "taken_at", "p_depression_tcal"]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"缺少必要欄位: {col}")

# 時間轉 datetime
df["taken_at"] = pd.to_datetime(df["taken_at"], utc=True, errors="coerce")

# 數值轉型
df["p_depression_tcal"] = pd.to_numeric(df["p_depression_tcal"], errors="coerce")
if "pseudo_label" in df.columns:
    df["pseudo_label"] = pd.to_numeric(df["pseudo_label"], errors="coerce")

# 去除關鍵缺失
df = df.dropna(subset=["username", "post_id", "taken_at", "p_depression_tcal"]).copy()

# 去重
df = df.drop_duplicates(subset=["username", "post_id"]).copy()

# 時間範圍過濾
start_ts = pd.Timestamp(START_DATE)
end_ts = pd.Timestamp(END_DATE)
df = df[(df["taken_at"] >= start_ts) & (df["taken_at"] <= end_ts)].copy()

# 排序
df = df.sort_values(["username", "taken_at", "post_id"]).reset_index(drop=True)

# =========================
# 4. 高風險貼文判定
# =========================
def is_high_risk_post(row,
                      use_pseudo_label_if_available=True,
                      high_risk_score_threshold=0.7):
    if use_pseudo_label_if_available and "pseudo_label" in row.index and pd.notna(row.get("pseudo_label", np.nan)):
        return int(row["pseudo_label"] == 1)
    return int(pd.notna(row["p_depression_tcal"]) and row["p_depression_tcal"] >= high_risk_score_threshold)

df["is_high_risk"] = df.apply(
    lambda r: is_high_risk_post(
        r,
        use_pseudo_label_if_available=USE_PSEUDO_LABEL_IF_AVAILABLE,
        high_risk_score_threshold=HIGH_RISK_SCORE_THRESHOLD
    ),
    axis=1
)

# =========================
# 5. 單一使用者：14 天滑動視窗主規則
# =========================
def label_single_user_by_window(df_user,
                                window_days=14,
                                min_posts=3,
                                min_high_risk=2,
                                mean_score_threshold=0.60):
    g = df_user.sort_values("taken_at").copy().reset_index(drop=True)
    if len(g) == 0:
        return 0, None, []

    positive_windows = []
    n = len(g)

    for i in range(n):
        window_start = g.loc[i, "taken_at"]
        window_end = window_start + pd.Timedelta(days=window_days)

        w = g[(g["taken_at"] >= window_start) & (g["taken_at"] < window_end)].copy()

        n_posts = len(w)
        n_high = int(w["is_high_risk"].sum())
        mean_score = float(w["p_depression_tcal"].mean()) if n_posts > 0 else np.nan

        is_positive_window = (
            (n_posts >= min_posts) and
            (n_high >= min_high_risk) and
            (mean_score >= mean_score_threshold)
        )

        if is_positive_window:
            positive_windows.append({
                "username": g.loc[0, "username"],
                "window_start": window_start.isoformat(),
                "window_end": window_end.isoformat(),
                "n_posts": int(n_posts),
                "n_high_risk_posts": int(n_high),
                "mean_p_depression_tcal": float(mean_score),
                "post_ids": "|".join(w["post_id"].astype(str).tolist()),
            })

    user_label = int(len(positive_windows) > 0)

    user_summary = {
        "username": g.loc[0, "username"],
        "num_posts_total": int(len(g)),
        "num_high_risk_posts_total": int(g["is_high_risk"].sum()),
        "mean_p_depression_tcal_total": float(g["p_depression_tcal"].mean()),
        "user_label": int(user_label),
        "num_positive_windows": int(len(positive_windows)),
        "first_post_time": g["taken_at"].min().isoformat(),
        "last_post_time": g["taken_at"].max().isoformat(),
    }

    return user_label, user_summary, positive_windows

# =========================
# 6. 跑單一方案
# =========================
def run_one_scheme(df, scheme_name, window_days, min_posts, min_high_risk, mean_score_threshold, output_dir):
    user_summaries = []
    all_positive_windows = []
    user_label_map = {}

    for username, g in df.groupby("username"):
        user_label, user_summary, positive_windows = label_single_user_by_window(
            g,
            window_days=window_days,
            min_posts=min_posts,
            min_high_risk=min_high_risk,
            mean_score_threshold=mean_score_threshold
        )

        user_label_map[username] = user_label

        user_summary["scheme_name"] = scheme_name
        user_summary["window_days"] = window_days
        user_summary["min_posts"] = min_posts
        user_summary["min_high_risk"] = min_high_risk
        user_summary["mean_score_threshold"] = mean_score_threshold
        user_summaries.append(user_summary)

        for pw in positive_windows:
            pw["scheme_name"] = scheme_name
            pw["window_days"] = window_days
            pw["min_posts"] = min_posts
            pw["min_high_risk"] = min_high_risk
            pw["mean_score_threshold"] = mean_score_threshold
            all_positive_windows.append(pw)

    user_summary_df = pd.DataFrame(user_summaries)
    positive_windows_df = pd.DataFrame(all_positive_windows)

    # 回寫到 post-level
    df_scheme = df.copy()
    df_scheme["user_label"] = df_scheme["username"].map(user_label_map)
    df_scheme["scheme_name"] = scheme_name

    # 輸出檔案
    user_summary_path = output_dir / f"user_summary_{scheme_name}.csv"
    positive_windows_path = output_dir / f"positive_windows_{scheme_name}.csv"
    post_with_label_path = output_dir / f"post_with_user_label_{scheme_name}.csv"

    user_summary_df.to_csv(user_summary_path, index=False, encoding="utf-8-sig")

    if len(positive_windows_df) > 0:
        positive_windows_df.to_csv(positive_windows_path, index=False, encoding="utf-8-sig")
    else:
        pd.DataFrame(columns=[
            "scheme_name", "username", "window_start", "window_end",
            "n_posts", "n_high_risk_posts", "mean_p_depression_tcal", "post_ids"
        ]).to_csv(positive_windows_path, index=False, encoding="utf-8-sig")

    df_scheme.to_csv(post_with_label_path, index=False, encoding="utf-8-sig")

    # 整體比較摘要
    n_users = user_summary_df["username"].nunique()
    n_positive_users = int(user_summary_df["user_label"].sum())
    n_negative_users = int((user_summary_df["user_label"] == 0).sum())
    positive_rate = n_positive_users / n_users if n_users > 0 else np.nan
    avg_positive_windows_per_positive_user = (
        user_summary_df.loc[user_summary_df["user_label"] == 1, "num_positive_windows"].mean()
        if n_positive_users > 0 else 0.0
    )

    scheme_summary = {
        "scheme_name": scheme_name,
        "window_days": window_days,
        "min_posts": min_posts,
        "min_high_risk": min_high_risk,
        "mean_score_threshold": mean_score_threshold,
        "n_users": int(n_users),
        "n_positive_users": int(n_positive_users),
        "n_negative_users": int(n_negative_users),
        "positive_rate": float(positive_rate),
        "avg_positive_windows_per_positive_user": float(avg_positive_windows_per_positive_user),
        "user_summary_path": str(user_summary_path),
        "positive_windows_path": str(positive_windows_path),
        "post_with_label_path": str(post_with_label_path),
    }

    return scheme_summary, user_summary_df, positive_windows_df, df_scheme

# =========================
# 7. 一次跑所有方案
# =========================
all_scheme_summaries = []
all_user_labels_wide = None

for params in PARAM_GRID:
    scheme_name = params["scheme_name"]
    min_posts = params["min_posts"]
    min_high_risk = params["min_high_risk"]
    mean_score_threshold = params["mean_score_threshold"]

    print(f"\n===== Running scheme: {scheme_name} =====")
    print(f"window_days={WINDOW_DAYS}, min_posts={min_posts}, min_high_risk={min_high_risk}, mean_score_threshold={mean_score_threshold}")

    scheme_summary, user_summary_df, positive_windows_df, df_scheme = run_one_scheme(
        df=df,
        scheme_name=scheme_name,
        window_days=WINDOW_DAYS,
        min_posts=min_posts,
        min_high_risk=min_high_risk,
        mean_score_threshold=mean_score_threshold,
        output_dir=OUTPUT_DIR
    )

    all_scheme_summaries.append(scheme_summary)

    # 整理每位使用者在不同方案下的 label（wide format）
    temp = user_summary_df[["username", "user_label"]].copy()
    temp = temp.rename(columns={"user_label": f"user_label_{scheme_name}"})

    if all_user_labels_wide is None:
        all_user_labels_wide = temp
    else:
        all_user_labels_wide = all_user_labels_wide.merge(temp, on="username", how="outer")

# =========================
# 8. 輸出整體敏感度比較表
# =========================
sensitivity_summary_df = pd.DataFrame(all_scheme_summaries)

sensitivity_summary_path = OUTPUT_DIR / "sensitivity_summary.csv"
user_labels_all_schemes_path = OUTPUT_DIR / "user_labels_all_schemes.csv"

sensitivity_summary_df.to_csv(sensitivity_summary_path, index=False, encoding="utf-8-sig")
all_user_labels_wide.to_csv(user_labels_all_schemes_path, index=False, encoding="utf-8-sig")

# =========================
# 9. 顯示結果
# =========================
print("\n==============================")
print("敏感度分析完成")
print("==============================")
print("\n=== sensitivity_summary ===")
print(sensitivity_summary_df)

print("\n=== user_labels_all_schemes 前幾列 ===")
print(all_user_labels_wide.head())

print("\n輸出檔案：")
print(sensitivity_summary_path)
print(user_labels_all_schemes_path)
print(f"其他各方案明細檔案都在：{OUTPUT_DIR}")

c:\Users\Angle\Anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3165: DtypeWarning: Columns (10,15) have mixed types.Specify dtype option on import or set low_memory=False.
  has_raised = await self.run_ast_nodes(code_ast.body, cell_name,



===== Running scheme: A_loose =====
window_days=14, min_posts=2, min_high_risk=1, mean_score_threshold=0.6

===== Running scheme: B_main =====
window_days=14, min_posts=3, min_high_risk=2, mean_score_threshold=0.6

===== Running scheme: C_strict =====
window_days=14, min_posts=4, min_high_risk=2, mean_score_threshold=0.65

敏感度分析完成

=== sensitivity_summary ===
  scheme_name  window_days  min_posts  min_high_risk  mean_score_threshold  \
0     A_loose           14          2              1                  0.60   
1      B_main           14          3              2                  0.60   
2    C_strict           14          4              2                  0.65   

   n_users  n_positive_users  n_negative_users  positive_rate  \
0      362               171               191       0.472376   
1      362               150               212       0.414365   
2      362               123               239       0.339779   

   avg_positive_windows_per_positive_user  \
0                 

In [22]:
import pandas as pd
import numpy as np

# =========================================================
# 1. 參數設定
# =========================================================
INPUT_CSV = "D:\\時間序列\\DECEN_TS\\auto_labeled_with_scores.csv"   # 改成你的檔案路徑

OUTPUT_POST_WITH_USER_LABEL = "D:\\時間序列\\labeled\\post_with_user_label.csv"
OUTPUT_USER_SUMMARY = "D:\\時間序列\\labeled\\user_summary_14d.csv"
OUTPUT_POSITIVE_WINDOWS = "D:\\時間序列\\labeled\\positive_windows_14d.csv"

# 時間範圍
START_DATE = "2021-01-01 00:00:00+00:00"
END_DATE   = "2025-12-31 23:59:59+00:00"

# 視窗設定
WINDOW_DAYS = 14

# 陽性視窗條件（主方案）
MIN_POSTS_IN_WINDOW = 3
MIN_HIGH_RISK_POSTS = 2
MEAN_SCORE_THRESHOLD = 0.6

# 高風險貼文定義
# 若 pseudo_label == 1 則視為高風險
# 或 p_depression_tcal >= HIGH_RISK_SCORE_THRESHOLD 也視為高風險
HIGH_RISK_SCORE_THRESHOLD = 0.7

# 若某視窗中沒有 pseudo_label，可直接依 score 判斷
USE_PSEUDO_LABEL_IF_AVAILABLE = True

# =========================================================
# 2. 讀檔與基本清理
# =========================================================
df = pd.read_csv(INPUT_CSV)

df.columns = [c.strip() for c in df.columns]

required_cols = ["username", "post_id", "taken_at", "p_depression_tcal"]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"缺少必要欄位: {col}")

# 時間轉 datetime
df["taken_at"] = pd.to_datetime(df["taken_at"], utc=True, errors="coerce")

# 數值欄位轉型
if "pseudo_label" in df.columns:
    df["pseudo_label"] = pd.to_numeric(df["pseudo_label"], errors="coerce")

df["p_depression_tcal"] = pd.to_numeric(df["p_depression_tcal"], errors="coerce")

# 去除關鍵缺失
df = df.dropna(subset=["username", "post_id", "taken_at", "p_depression_tcal"]).copy()

# 去重
df = df.drop_duplicates(subset=["username", "post_id"]).copy()

# 限定時間範圍
start_ts = pd.Timestamp(START_DATE)
end_ts = pd.Timestamp(END_DATE)
df = df[(df["taken_at"] >= start_ts) & (df["taken_at"] <= end_ts)].copy()

# 排序
df = df.sort_values(["username", "taken_at", "post_id"]).reset_index(drop=True)

# =========================================================
# 3. 定義高風險貼文
# =========================================================
def is_high_risk_post(row,
                      use_pseudo_label_if_available=True,
                      high_risk_score_threshold=0.7):
    """
    高風險貼文判定：
    1) 若 pseudo_label 可用且 use_pseudo_label_if_available=True，則 pseudo_label == 1 視為高風險
    2) 否則 p_depression_tcal >= threshold 視為高風險
    """
    if use_pseudo_label_if_available and "pseudo_label" in row.index and pd.notna(row["pseudo_label"]):
        return int(row["pseudo_label"] == 1)
    return int(pd.notna(row["p_depression_tcal"]) and row["p_depression_tcal"] >= high_risk_score_threshold)

df["is_high_risk"] = df.apply(
    lambda r: is_high_risk_post(
        r,
        use_pseudo_label_if_available=USE_PSEUDO_LABEL_IF_AVAILABLE,
        high_risk_score_threshold=HIGH_RISK_SCORE_THRESHOLD
    ),
    axis=1
)

# =========================================================
# 4. 14 天滑動視窗標註函式
# =========================================================
def label_users_by_sliding_window(
    df_user,
    window_days=14,
    min_posts=3,
    min_high_risk=2,
    mean_score_threshold=0.6
):
    """
    對單一 user 的貼文做 14 天滑動視窗標註

    回傳：
    - user_label: 0/1
    - user_summary: dict
    - positive_windows: list[dict]
    """
    g = df_user.sort_values("taken_at").copy().reset_index(drop=True)

    if len(g) == 0:
        return 0, None, []

    positive_windows = []
    n = len(g)

    # 逐篇貼文當視窗起點
    for i in range(n):
        window_start = g.loc[i, "taken_at"]
        window_end = window_start + pd.Timedelta(days=window_days)

        window_df = g[(g["taken_at"] >= window_start) & (g["taken_at"] < window_end)].copy()

        n_posts = len(window_df)
        n_high = int(window_df["is_high_risk"].sum())
        mean_score = float(window_df["p_depression_tcal"].mean()) if n_posts > 0 else np.nan

        is_positive_window = (
            (n_posts >= min_posts) and
            (n_high >= min_high_risk) and
            (mean_score >= mean_score_threshold)
        )

        if is_positive_window:
            positive_windows.append({
                "username": g.loc[0, "username"],
                "window_start": window_start.isoformat(),
                "window_end": window_end.isoformat(),
                "n_posts": int(n_posts),
                "n_high_risk_posts": int(n_high),
                "mean_p_depression_tcal": float(mean_score),
                "post_ids": list(window_df["post_id"].astype(str)),
            })

    user_label = int(len(positive_windows) > 0)

    user_summary = {
        "username": g.loc[0, "username"],
        "user_label": user_label,
        "num_posts_total": int(len(g)),
        "num_high_risk_posts_total": int(g["is_high_risk"].sum()),
        "mean_p_depression_tcal_total": float(g["p_depression_tcal"].mean()),
        "num_positive_windows": int(len(positive_windows)),
        "first_post_time": g["taken_at"].min().isoformat(),
        "last_post_time": g["taken_at"].max().isoformat(),
    }

    return user_label, user_summary, positive_windows

# =========================================================
# 5. 套用到所有使用者
# =========================================================
user_summaries = []
all_positive_windows = []
user_label_map = {}

for username, g in df.groupby("username"):
    user_label, user_summary, positive_windows = label_users_by_sliding_window(
        g,
        window_days=WINDOW_DAYS,
        min_posts=MIN_POSTS_IN_WINDOW,
        min_high_risk=MIN_HIGH_RISK_POSTS,
        mean_score_threshold=MEAN_SCORE_THRESHOLD
    )

    user_label_map[username] = user_label
    user_summaries.append(user_summary)

    if len(positive_windows) > 0:
        all_positive_windows.extend(positive_windows)

# =========================================================
# 6. 回寫 user_label 到原始 post-level 資料
# =========================================================
df["user_label"] = df["username"].map(user_label_map)

user_summary_df = pd.DataFrame(user_summaries)
positive_windows_df = pd.DataFrame(all_positive_windows)

# =========================================================
# 7. 輸出結果
# =========================================================
df.to_csv(OUTPUT_POST_WITH_USER_LABEL, index=False, encoding="utf-8-sig")
user_summary_df.to_csv(OUTPUT_USER_SUMMARY, index=False, encoding="utf-8-sig")

if len(positive_windows_df) > 0:
    positive_windows_df.to_csv(OUTPUT_POSITIVE_WINDOWS, index=False, encoding="utf-8-sig")
else:
    # 若沒有陽性視窗，也輸出空檔方便檢查
    pd.DataFrame(columns=[
        "username", "window_start", "window_end",
        "n_posts", "n_high_risk_posts", "mean_p_depression_tcal", "post_ids"
    ]).to_csv(OUTPUT_POSITIVE_WINDOWS, index=False, encoding="utf-8-sig")

# =========================================================
# 8. 顯示摘要
# =========================================================
print("=== 完成 14 天滑動視窗 user-level 標註 ===")
print(f"貼文總數: {len(df)}")
print(f"使用者總數: {df['username'].nunique()}")
print(f"陽性使用者數: {int(user_summary_df['user_label'].sum())}")
print(f"陰性使用者數: {int((user_summary_df['user_label'] == 0).sum())}")

print("\n=== user_summary 前幾列 ===")
print(user_summary_df.head())

print("\n=== positive_windows 前幾列 ===")
print(positive_windows_df.head())

print("\n已輸出：")
print(OUTPUT_POST_WITH_USER_LABEL)
print(OUTPUT_USER_SUMMARY)
print(OUTPUT_POSITIVE_WINDOWS)

=== 完成 14 天滑動視窗 user-level 標註 ===
貼文總數: 18670
使用者總數: 74
陽性使用者數: 46
陰性使用者數: 28

=== user_summary 前幾列 ===
        username  user_label  num_posts_total  num_high_risk_posts_total  \
0     914_planet           0               26                          0   
1   _kana.words_           1               71                         58   
2     _midleave_           1              740                         66   
3    a0962049992           0               15                          2   
4  adela.writing           1              486                        217   

   mean_p_depression_tcal_total  num_positive_windows  \
0                      0.061097                     0   
1                      0.838837                    36   
2                      0.124200                     1   
3                      0.148877                     0   
4                      0.496948                   164   

             first_post_time             last_post_time  
0  2021-04-03T16:13:33+00:00  2024-11-

In [2]:
# 確認總資料筆數

from pathlib import Path
import pandas as pd

# DST_ROOT = Path(r"D:\時間序列\ContextVecNet_Instagram_filtered")
DST_ROOT = Path(r"D:\時間序列\ContextVecNet_Instagram_242")
results = []
for split in ["positive", "negative"]:
    split_dir = DST_ROOT / split
    if not split_dir.exists():
        print(f"Warning: {split_dir} not found")
        continue

    user_dirs = [p for p in split_dir.iterdir() if p.is_dir()]
    user_counts = []
    for user_dir in user_dirs:
        metadata_path = user_dir / "metadata.csv"
        if metadata_path.exists():
            df = pd.read_csv(metadata_path)
            user_counts.append(len(df))
        else:
            # 如果沒有 metadata.csv，改用 timeline.txt 估算
            timeline_path = user_dir / "timeline.txt"
            if timeline_path.exists():
                with open(timeline_path, "r", encoding="utf-8", errors="ignore") as f:
                    user_counts.append(sum(1 for _ in f))
            else:
                user_counts.append(0)

    user_count = len(user_counts)
    total_posts = sum(user_counts)
    avg_posts = total_posts / user_count if user_count > 0 else 0.0
    results.append({
        "split": split,
        "user_count": user_count,
        "total_posts": total_posts,
        "avg_posts_per_user": avg_posts,
        "min_posts_per_user": min(user_counts) if user_counts else 0,
        "max_posts_per_user": max(user_counts) if user_counts else 0,
        "median_posts_per_user": pd.Series(user_counts).median() if user_counts else 0,
    })

summary_df = pd.DataFrame(results)
print(summary_df.to_string(index=False))

overall = {
    "split": "all",
    "user_count": int(summary_df["user_count"].sum()),
    "total_posts": int(summary_df["total_posts"].sum()),
    "avg_posts_per_user": float(summary_df["total_posts"].sum() / summary_df["user_count"].sum())
    if summary_df["user_count"].sum() > 0 else 0.0,
}
print("\nOverall:")
print(overall)


   split  user_count  total_posts  avg_posts_per_user  min_posts_per_user  max_posts_per_user  median_posts_per_user
positive          93        17687          190.182796                   8                 887                  126.0
negative         149        68936          462.657718                   1                3081                  197.0

Overall:
{'split': 'all', 'user_count': 242, 'total_posts': 86623, 'avg_posts_per_user': 357.94628099173553}


In [2]:
# 確認總資料筆數

from pathlib import Path
import pandas as pd

DST_ROOT = Path(r"D:\時間序列\ContextVecNet_Instagram_filtered_new")

results = []
for split in ["positive", "negative"]:
    split_dir = DST_ROOT / split
    if not split_dir.exists():
        print(f"Warning: {split_dir} not found")
        continue

    user_dirs = [p for p in split_dir.iterdir() if p.is_dir()]
    user_counts = []
    for user_dir in user_dirs:
        metadata_path = user_dir / "metadata.csv"
        if metadata_path.exists():
            df = pd.read_csv(metadata_path)
            user_counts.append(len(df))
        else:
            # 如果沒有 metadata.csv，改用 timeline.txt 估算
            timeline_path = user_dir / "timeline.txt"
            if timeline_path.exists():
                with open(timeline_path, "r", encoding="utf-8", errors="ignore") as f:
                    user_counts.append(sum(1 for _ in f))
            else:
                user_counts.append(0)

    user_count = len(user_counts)
    total_posts = sum(user_counts)
    avg_posts = total_posts / user_count if user_count > 0 else 0.0
    results.append({
        "split": split,
        "user_count": user_count,
        "total_posts": total_posts,
        "avg_posts_per_user": avg_posts,
        "min_posts_per_user": min(user_counts) if user_counts else 0,
        "max_posts_per_user": max(user_counts) if user_counts else 0,
        "median_posts_per_user": pd.Series(user_counts).median() if user_counts else 0,
    })

summary_df = pd.DataFrame(results)
print(summary_df.to_string(index=False))

overall = {
    "split": "all",
    "user_count": int(summary_df["user_count"].sum()),
    "total_posts": int(summary_df["total_posts"].sum()),
    "avg_posts_per_user": float(summary_df["total_posts"].sum() / summary_df["user_count"].sum())
    if summary_df["user_count"].sum() > 0 else 0.0,
}
print("\nOverall:")
print(overall)


   split  user_count  total_posts  avg_posts_per_user  min_posts_per_user  max_posts_per_user  median_posts_per_user
positive         130        23367          179.746154                  20                 908                  135.5
negative         162        43606          269.172840                  20                2751                  178.0

Overall:
{'split': 'all', 'user_count': 292, 'total_posts': 66973, 'avg_posts_per_user': 229.3595890410959}


#### 以下不用

In [ ]:
import pandas as pd
import numpy as np
import json
import ast
import re
from pathlib import Path

# =========================================
# 1. 路徑設定
# =========================================
INPUT_CSV = "D:/時間序列/labeled/label_sensitivity_outputs/post_with_user_label_B_main.csv"
# INPUT_CSV = "D:/時間序列/labeled/post_with_user_label.csv"
OUTPUT_DIR = Path("D:/時間序列/final_preprocessed_data")
# OUTPUT_DIR = Path("D:/時間序列/final_preprocessed_data_all")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_POST_LEVEL = OUTPUT_DIR / "model_input_post_level.csv"
OUTPUT_USER_JSON = OUTPUT_DIR / "user_level_data.json"
OUTPUT_USER_SUMMARY = OUTPUT_DIR / "user_level_summary.csv"

# =========================================
# 2. 基本參數
# =========================================
START_DATE = "2021-01-01 00:00:00+00:00"
END_DATE   = "2025-12-31 23:59:59+00:00"

MERGE_CAPTION_HASHTAGS = True
ADD_HASH_SYMBOL = True

# =========================================
# 3. 讀檔
# =========================================
df = pd.read_csv(INPUT_CSV)
df.columns = [c.strip() for c in df.columns]

required_cols = ["username", "post_id", "taken_at", "user_label"]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"缺少必要欄位: {col}")

# =========================================
# 4. 時間欄位處理
# =========================================
df["taken_at"] = pd.to_datetime(df["taken_at"], utc=True, errors="coerce")

start_ts = pd.Timestamp(START_DATE)
end_ts = pd.Timestamp(END_DATE)

df = df[(df["taken_at"] >= start_ts) & (df["taken_at"] <= end_ts)].copy()
df = df.dropna(subset=["username", "post_id", "taken_at", "user_label"]).copy()

# 去重
df = df.drop_duplicates(subset=["username", "post_id"]).copy()

# 排序
df = df.sort_values(["username", "taken_at", "post_id"]).reset_index(drop=True)

# UTC Unix timestamp（秒）
df["tau"] = df["taken_at"].astype("int64") // 10**9

# 依照論文時間處理：g(tau)=1/(tau+1)
df["tau_transformed"] = 1.0 / (df["tau"] + 1.0)

# 額外保留台灣時間特徵（若後續要做行為特徵可用）
df["taken_at_taipei"] = df["taken_at"].dt.tz_convert("Asia/Taipei")
df["hour"] = df["taken_at_taipei"].dt.hour
df["weekday"] = df["taken_at_taipei"].dt.weekday
df["is_night"] = df["hour"].apply(lambda x: 1 if x < 6 else 0)

# 相對時間（以每位使用者第一篇貼文為基準，單位：天）
df["user_first_time"] = df.groupby("username")["taken_at"].transform("min")
df["relative_days"] = (df["taken_at"] - df["user_first_time"]).dt.total_seconds() / 86400.0

# =========================================
# 5. hashtags 欄位整理
# =========================================
def parse_hashtags(x):
    if pd.isna(x):
        return []

    if isinstance(x, list):
        tags = x
    else:
        s = str(x).strip()
        if s == "":
            return []

        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, list):
                tags = parsed
            else:
                tags = [s]
        except Exception:
            tags = re.split(r"[\s,]+", s)

    clean_tags = []
    for t in tags:
        t = str(t).strip()
        if t == "":
            continue
        t = t.lstrip("#")
        if t != "":
            clean_tags.append(f"#{t}" if ADD_HASH_SYMBOL else t)
    return clean_tags

if "hashtags" in df.columns:
    df["hashtags_list"] = df["hashtags"].apply(parse_hashtags)
else:
    df["hashtags_list"] = [[] for _ in range(len(df))]

# =========================================
# 6. 文字欄位整理
# =========================================
def clean_text_basic(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"<[^>]+>", " ", text)   # 移除 HTML tag
    text = re.sub(r"\s+", " ", text).strip()
    return text

def build_text(caption, hashtags_list, merge_caption_hashtags=True):
    caption = clean_text_basic(caption)
    tags = " ".join(hashtags_list) if isinstance(hashtags_list, list) else ""

    if merge_caption_hashtags:
        if caption and tags:
            return f"{caption}\n{tags}"
        elif caption:
            return caption
        else:
            return tags
    else:
        return caption

caption_col = "caption" if "caption" in df.columns else None
df["text"] = df.apply(
    lambda row: build_text(
        row[caption_col] if caption_col else "",
        row["hashtags_list"],
        merge_caption_hashtags=MERGE_CAPTION_HASHTAGS
    ),
    axis=1
)

# 額外文字統計特徵（之後若要用可直接接）
df["char_count"] = df["text"].apply(len)
df["hashtag_count"] = df["hashtags_list"].apply(len)

# =========================================
# 7. 視覺欄位整理（圖片 / 影片）
# =========================================
def infer_visual_type(media_type, local_media_path):
    mt = "" if pd.isna(media_type) else str(media_type).lower()
    path = "" if pd.isna(local_media_path) else str(local_media_path).lower()

    # 先看副檔名
    if path.endswith((".jpg", ".jpeg", ".png", ".webp", ".bmp")):
        return "image"
    if path.endswith((".mp4", ".mov", ".avi", ".mkv", ".webm")):
        return "video"

    # 再看 media_type
    if "video" in mt:
        return "video"
    if "image" in mt or "photo" in mt or "carousel" in mt:
        return "image"

    return "unknown"

def get_visual_path(local_media_path, media_url):
    if pd.notna(local_media_path) and str(local_media_path).strip() != "":
        return str(local_media_path).strip()
    if pd.notna(media_url) and str(media_url).strip() != "":
        return str(media_url).strip()
    return None

media_type_col = "media_type" if "media_type" in df.columns else None
local_media_col = "local_media_path" if "local_media_path" in df.columns else None
media_url_col = "media_url" if "media_url" in df.columns else None

df["visual_type"] = df.apply(
    lambda row: infer_visual_type(
        row[media_type_col] if media_type_col else "",
        row[local_media_col] if local_media_col else ""
    ),
    axis=1
)

df["visual_path"] = df.apply(
    lambda row: get_visual_path(
        row[local_media_col] if local_media_col else None,
        row[media_url_col] if media_url_col else None
    ),
    axis=1
)

df["has_visual"] = df["visual_path"].notna().astype(int)

# 說明欄：先記錄後續怎麼處理
def plan_visual_strategy(vtype):
    if vtype == "image":
        return "use_image_directly"
    elif vtype == "video":
        return "extract_middle_frame"
    else:
        return "use_black_image"

df["visual_strategy"] = df["visual_type"].apply(plan_visual_strategy)

# =========================================
# 8. 保留之後模型真正需要的 post-level 欄位
# =========================================
keep_cols = [
    "username",
    "post_id",
    "user_label",
    "text",
    "char_count",
    "hashtag_count",
    "visual_path",
    "visual_type",
    "visual_strategy",
    "has_visual",
    "taken_at",
    "taken_at_taipei",
    "tau",
    "tau_transformed",
    "relative_days",
    "hour",
    "weekday",
    "is_night",
]

# 若原表中有這些欄位就補進來
optional_cols = [
    "post_url",
    "caption",
    "hashtags",
    "like_count",
    "comment_count",
    "media_type",
    "media_url",
    "local_media_path",
    "p_depression_t1",
    "p_depression_tcal",
    "logit_margin",
    "pseudo_label",
]
for c in optional_cols:
    if c in df.columns:
        keep_cols.append(c)

model_input_df = df[keep_cols].copy()

# =========================================
# 9. 產出 user-level JSON
# =========================================
user_level_data = []
user_summary_rows = []

for username, g in model_input_df.groupby("username"):
    g = g.sort_values("taken_at").copy()

    user_label = int(g["user_label"].iloc[0])

    posts = []
    for _, row in g.iterrows():
        posts.append({
            "post_id": str(row["post_id"]),
            "text": row["text"],
            "visual_path": row["visual_path"] if pd.notna(row["visual_path"]) else None,
            "visual_type": row["visual_type"],
            "visual_strategy": row["visual_strategy"],
            "has_visual": int(row["has_visual"]),
            "taken_at_utc": pd.Timestamp(row["taken_at"]).isoformat(),
            "taken_at_taipei": pd.Timestamp(row["taken_at_taipei"]).isoformat(),
            "tau": int(row["tau"]),
            "tau_transformed": float(row["tau_transformed"]),
            "relative_days": float(row["relative_days"]),
            "hour": int(row["hour"]),
            "weekday": int(row["weekday"]),
            "is_night": int(row["is_night"]),
            "char_count": int(row["char_count"]),
            "hashtag_count": int(row["hashtag_count"]),
            "p_depression_tcal": None if "p_depression_tcal" not in row or pd.isna(row.get("p_depression_tcal", np.nan)) else float(row["p_depression_tcal"]),
            "pseudo_label": None if "pseudo_label" not in row or pd.isna(row.get("pseudo_label", np.nan)) else int(row["pseudo_label"]),
        })

    user_level_data.append({
        "username": username,
        "label": user_label,
        "num_posts": int(len(g)),
        "posts": posts
    })

    user_summary_rows.append({
        "username": username,
        "label": user_label,
        "num_posts": int(len(g)),
        "avg_char_count": float(g["char_count"].mean()),
        "avg_hashtag_count": float(g["hashtag_count"].mean()),
        "visual_ratio": float(g["has_visual"].mean()),
        "video_ratio": float((g["visual_type"] == "video").mean()),
        "image_ratio": float((g["visual_type"] == "image").mean()),
        "avg_p_depression_tcal": float(g["p_depression_tcal"].mean()) if "p_depression_tcal" in g.columns else np.nan,
        "first_post_time": g["taken_at"].min().isoformat(),
        "last_post_time": g["taken_at"].max().isoformat(),
    })

user_summary_df = pd.DataFrame(user_summary_rows)

# =========================================
# 10. 輸出
# =========================================
model_input_df.to_csv(OUTPUT_POST_LEVEL, index=False, encoding="utf-8-sig")

with open(OUTPUT_USER_JSON, "w", encoding="utf-8") as f:
    json.dump(user_level_data, f, ensure_ascii=False, indent=2)

user_summary_df.to_csv(OUTPUT_USER_SUMMARY, index=False, encoding="utf-8-sig")

# =========================================
# 11. 顯示摘要
# =========================================
print("=== 前處理完成 ===")
print(f"post-level 檔案：{OUTPUT_POST_LEVEL}")
print(f"user-level JSON：{OUTPUT_USER_JSON}")
print(f"user-level 摘要：{OUTPUT_USER_SUMMARY}")

print("\n=== model_input_post_level 前幾列 ===")
print(model_input_df.head())

print("\n=== user_level_summary 前幾列 ===")
print(user_summary_df.head())

print("\n=== 第一位使用者範例（前 1 筆） ===")
if len(user_level_data) > 0:
    print(json.dumps(user_level_data[0], ensure_ascii=False, indent=2)[:2500])

=== 前處理完成 ===
post-level 檔案：D:\時間序列\final_preprocessed_data_all\model_input_post_level.csv
user-level JSON：D:\時間序列\final_preprocessed_data_all\user_level_data.json
user-level 摘要：D:\時間序列\final_preprocessed_data_all\user_level_summary.csv

=== model_input_post_level 前幾列 ===
     username                          post_id  user_label  \
0  156.melody  2960932059168283176_10006613875           0   
1  156.melody  2961656747658029011_10006613875           0   
2  156.melody  2962352246316842299_10006613875           0   
3  156.melody  2963094277490097632_10006613875           0   
4  156.melody  2963821833539625073_10006613875           0   

                                                text  char_count  \
0  朋友說這套不像警察 比較像保全 來當一日大樓保全😂 . . . . #萬聖節 #萬聖節快樂 ...         541   
1  買女兒的衣服🤭 . , . . #cani生活有機棉 #ootd #ootdfashion ...         447   
2  好像還很少人知道這家咖啡廳 一個包場的概念 . . . . #ootd #fashion #...         445   
3  女兒感冒第5天了 才知道以前媽媽多辛苦😂 . . . . #taiwan #ootd #oo...         448   
4  很適合超級愛吃蔬菜的我

In [35]:
# =========================================
# 前處理前：檢查所有使用者資料是否齊全
# 適用檔案：model_input_post_level.csv 或 post_with_user_label_B_main.csv
# =========================================

import pandas as pd
import numpy as np
from pathlib import Path

# =========================================
# 1. 路徑設定
# =========================================
INPUT_CSV = "D:/時間序列/final_preprocessed_data_all/model_input_post_level.csv"

OUTPUT_DIR = Path("D:/時間序列/final_preprocessed_data_all/final_data_quality_check")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_USER_REPORT = OUTPUT_DIR / "user_completeness_report.csv"
OUTPUT_PROBLEM_POSTS = OUTPUT_DIR / "problem_posts.csv"
OUTPUT_SUMMARY = OUTPUT_DIR / "data_quality_summary.txt"

# =========================================
# 2. 讀檔
# =========================================
df = pd.read_csv(INPUT_CSV)
df.columns = [c.strip() for c in df.columns]

required_cols = ["username", "post_id", "user_label", "text", "taken_at"]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"缺少必要欄位: {col}")

# 若沒有 visual_path / visual_type，也先補空欄
if "visual_path" not in df.columns:
    df["visual_path"] = np.nan
if "visual_type" not in df.columns:
    df["visual_type"] = np.nan

# =========================================
# 3. 基本標記欄位
# =========================================
df["text"] = df["text"].fillna("").astype(str)
df["visual_path"] = df["visual_path"].fillna("").astype(str)
df["visual_type"] = df["visual_type"].fillna("").astype(str)

# 時間欄位檢查
df["taken_at_parsed"] = pd.to_datetime(df["taken_at"], utc=True, errors="coerce")

# 各貼文層級是否有問題
df["missing_text"] = df["text"].str.strip().eq("")
df["bad_time"] = df["taken_at_parsed"].isna()
df["missing_label"] = df["user_label"].isna()

# 視覺缺失：沒有 path 或 visual_type 不合理
df["missing_visual_path"] = df["visual_path"].str.strip().eq("")
df["unknown_visual_type"] = ~df["visual_type"].str.lower().isin(["image", "video", "unknown", ""])
df["has_any_visual_info"] = ~df["missing_visual_path"]

# 問題貼文彙總
df["has_problem"] = (
    df["missing_text"] |
    df["bad_time"] |
    df["missing_label"] |
    df["unknown_visual_type"]
)

# =========================================
# 4. 使用者層級彙整
# =========================================
user_rows = []

for username, g in df.groupby("username"):
    num_posts = len(g)

    # label 一致性：同一 user 是否出現多個 user_label
    unique_labels = g["user_label"].dropna().unique().tolist()
    label_consistent = len(unique_labels) == 1
    final_label = unique_labels[0] if len(unique_labels) == 1 else np.nan

    # 文字
    n_missing_text = int(g["missing_text"].sum())
    all_posts_missing_text = int(n_missing_text == num_posts)

    # 時間
    n_bad_time = int(g["bad_time"].sum())
    all_posts_bad_time = int(n_bad_time == num_posts)

    # 視覺
    n_missing_visual = int(g["missing_visual_path"].sum())
    n_has_visual = int(g["has_any_visual_info"].sum())
    all_posts_no_visual = int(n_has_visual == 0)

    # 問題貼文數
    n_problem_posts = int(g["has_problem"].sum())

    # 最早 / 最晚時間
    valid_times = g["taken_at_parsed"].dropna()
    first_post_time = valid_times.min().isoformat() if len(valid_times) > 0 else None
    last_post_time = valid_times.max().isoformat() if len(valid_times) > 0 else None

    # 使用者層級是否可用（你可以依需求調整規則）
    # 目前規則：
    # 1. label 必須一致
    # 2. 不能所有貼文都沒文字
    # 3. 不能所有貼文時間都壞掉
    user_usable = int(
        label_consistent and
        (all_posts_missing_text == 0) and
        (all_posts_bad_time == 0)
    )

    user_rows.append({
        "username": username,
        "num_posts": num_posts,
        "final_label": final_label,
        "label_consistent": int(label_consistent),
        "num_unique_labels": len(unique_labels),
        "n_missing_text": n_missing_text,
        "all_posts_missing_text": all_posts_missing_text,
        "n_bad_time": n_bad_time,
        "all_posts_bad_time": all_posts_bad_time,
        "n_missing_visual": n_missing_visual,
        "n_has_visual": n_has_visual,
        "all_posts_no_visual": all_posts_no_visual,
        "n_problem_posts": n_problem_posts,
        "first_post_time": first_post_time,
        "last_post_time": last_post_time,
        "user_usable": user_usable,
    })

user_report = pd.DataFrame(user_rows)

# =========================================
# 5. 輸出檔案
# =========================================
user_report.to_csv(OUTPUT_USER_REPORT, index=False, encoding="utf-8-sig")

problem_posts = df[df["has_problem"]].copy()
problem_posts.to_csv(OUTPUT_PROBLEM_POSTS, index=False, encoding="utf-8-sig")

# =========================================
# 6. 摘要統計
# =========================================
n_users = user_report["username"].nunique()
n_usable_users = int(user_report["user_usable"].sum())
n_unusable_users = int((user_report["user_usable"] == 0).sum())

summary_lines = []
summary_lines.append("=== 資料完整性檢查摘要 ===")
summary_lines.append(f"總使用者數: {n_users}")
summary_lines.append(f"可用使用者數: {n_usable_users}")
summary_lines.append(f"不可用使用者數: {n_unusable_users}")
summary_lines.append("")
summary_lines.append("=== 使用者層級問題統計 ===")
summary_lines.append(f"label 不一致的使用者數: {int((user_report['label_consistent'] == 0).sum())}")
summary_lines.append(f"所有貼文都缺文字的使用者數: {int(user_report['all_posts_missing_text'].sum())}")
summary_lines.append(f"所有貼文時間都壞掉的使用者數: {int(user_report['all_posts_bad_time'].sum())}")
summary_lines.append(f"完全沒有任何視覺資料的使用者數: {int(user_report['all_posts_no_visual'].sum())}")
summary_lines.append("")
summary_lines.append("=== 貼文層級問題統計 ===")
summary_lines.append(f"有問題貼文總數: {len(problem_posts)}")
summary_lines.append(f"缺文字貼文數: {int(df['missing_text'].sum())}")
summary_lines.append(f"時間解析失敗貼文數: {int(df['bad_time'].sum())}")
summary_lines.append(f"user_label 缺失貼文數: {int(df['missing_label'].sum())}")
summary_lines.append(f"未知 visual_type 貼文數: {int(df['unknown_visual_type'].sum())}")

with open(OUTPUT_SUMMARY, "w", encoding="utf-8") as f:
    f.write("\n".join(summary_lines))

# =========================================
# 7. 顯示結果
# =========================================
print("\n".join(summary_lines))

print("\n=== user_report 前幾列 ===")
print(user_report.head())

print("\n=== problem_posts 前幾列 ===")
print(problem_posts[[
    "username", "post_id", "text", "taken_at", "visual_path", "visual_type",
    "missing_text", "bad_time", "missing_label", "unknown_visual_type"
]].head())

print("\n輸出檔案：")
print(OUTPUT_USER_REPORT)
print(OUTPUT_PROBLEM_POSTS)
print(OUTPUT_SUMMARY)

=== 資料完整性檢查摘要 ===
總使用者數: 161
可用使用者數: 161
不可用使用者數: 0

=== 使用者層級問題統計 ===
label 不一致的使用者數: 0
所有貼文都缺文字的使用者數: 0
所有貼文時間都壞掉的使用者數: 0
完全沒有任何視覺資料的使用者數: 0

=== 貼文層級問題統計 ===
有問題貼文總數: 96
缺文字貼文數: 96
時間解析失敗貼文數: 0
user_label 缺失貼文數: 0
未知 visual_type 貼文數: 0

=== user_report 前幾列 ===
       username  num_posts  final_label  label_consistent  num_unique_labels  \
0    156.melody        935            0                 1                  1   
1    914_planet         26            0                 1                  1   
2  _kana.words_         71            1                 1                  1   
3    _lydiawong       1084            0                 1                  1   
4    _midleave_        740            1                 1                  1   

   n_missing_text  all_posts_missing_text  n_bad_time  all_posts_bad_time  \
0               0                       0           0                   0   
1               0                       0           0                   0   
2               0       

In [4]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

# =========================================
# 1. 路徑設定
# =========================================
INPUT_CSV = "D:/時間序列/final_preprocessed_data_all/model_input_post_level.csv"

# 你的媒體根目錄
MEDIA_ROOT = Path("D:/時間序列/depress_dataset/downloaded_media")

OUTPUT_DIR = Path("D:/時間序列/final_preprocessed_data_all/final_data_quality_check")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_POST_CHECK = OUTPUT_DIR / "visual_file_check_post_level.csv"
OUTPUT_USER_SUMMARY = OUTPUT_DIR / "visual_file_check_user_summary.csv"
OUTPUT_MISSING_ONLY = OUTPUT_DIR / "visual_file_missing_only.csv"

# =========================================
# 2. 讀檔
# =========================================
df = pd.read_csv(INPUT_CSV)
df.columns = [c.strip() for c in df.columns]

required_cols = ["username", "post_id", "visual_path", "visual_type"]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"缺少必要欄位: {col}")

df["username"] = df["username"].astype(str)
df["visual_path"] = df["visual_path"].fillna("").astype(str)
df["visual_type"] = df["visual_type"].fillna("unknown").astype(str)

# =========================================
# 3. 工具函式
# =========================================
def resolve_visual_path(username, visual_path, media_root):
    """
    檢查 visual_path 是否存在於 download_media/<username>/ 下
    回傳：
    - resolved_path: 實際找到的路徑，否則 None
    - exists_in_user_folder: 0/1
    - check_mode: 用哪種方式找到
    """
    if not visual_path or str(visual_path).strip() == "":
        return None, 0, "empty_path"

    vp = str(visual_path).strip()
    user_dir = media_root / str(username)

    # 情況 1：visual_path 本身就是存在的完整路徑
    if os.path.exists(vp):
        try:
            abs_vp = Path(vp).resolve()
            abs_user_dir = user_dir.resolve()
            # 檢查是否在 username 子資料夾底下
            if str(abs_vp).startswith(str(abs_user_dir)):
                return str(abs_vp), 1, "full_path_in_user_folder"
            else:
                return str(abs_vp), 0, "full_path_but_not_in_user_folder"
        except Exception:
            return vp, 0, "full_path_resolve_error"

    # 情況 2：把 visual_path 當成檔名，去 download_media/<username>/ 找
    candidate_by_name = user_dir / Path(vp).name
    if candidate_by_name.exists():
        return str(candidate_by_name.resolve()), 1, "filename_matched_in_user_folder"

    # 情況 3：若 visual_path 是相對路徑，也試著接在 user_dir 後面
    candidate_relative = user_dir / vp
    if candidate_relative.exists():
        return str(candidate_relative.resolve()), 1, "relative_path_matched_in_user_folder"

    # 找不到
    return None, 0, "not_found"

# =========================================
# 4. 逐筆檢查
# =========================================
resolved_paths = []
exists_flags = []
check_modes = []
user_folder_paths = []

for _, row in df.iterrows():
    username = row["username"]
    visual_path = row["visual_path"]

    user_dir = MEDIA_ROOT / username
    resolved_path, exists_flag, check_mode = resolve_visual_path(username, visual_path, MEDIA_ROOT)

    resolved_paths.append(resolved_path)
    exists_flags.append(exists_flag)
    check_modes.append(check_mode)
    user_folder_paths.append(str(user_dir))

df["user_media_folder"] = user_folder_paths
df["resolved_visual_path"] = resolved_paths
df["exists_in_user_folder"] = exists_flags
df["check_mode"] = check_modes

# 是否為需要視覺檔案的貼文
df["needs_visual_file"] = df["visual_type"].str.lower().isin(["image", "video"]).astype(int)

# 真正缺失：需要視覺資料，但找不到檔案
df["missing_required_visual"] = (
    (df["needs_visual_file"] == 1) &
    (df["exists_in_user_folder"] == 0)
).astype(int)

# =========================================
# 5. user-level 摘要
# =========================================
user_rows = []
for username, g in df.groupby("username"):
    n_posts = len(g)
    n_visual_posts = int(g["needs_visual_file"].sum())
    n_found = int(((g["needs_visual_file"] == 1) & (g["exists_in_user_folder"] == 1)).sum())
    n_missing = int(g["missing_required_visual"].sum())

    user_rows.append({
        "username": username,
        "num_posts": n_posts,
        "num_visual_posts": n_visual_posts,
        "num_visual_found": n_found,
        "num_visual_missing": n_missing,
        "visual_file_completeness": (n_found / n_visual_posts) if n_visual_posts > 0 else np.nan,
        "all_visual_found": int((n_visual_posts == 0) or (n_missing == 0)),
        "user_media_folder": str(MEDIA_ROOT / username)
    })

user_summary = pd.DataFrame(user_rows)

# =========================================
# 6. 輸出
# =========================================
df.to_csv(OUTPUT_POST_CHECK, index=False, encoding="utf-8-sig")
user_summary.to_csv(OUTPUT_USER_SUMMARY, index=False, encoding="utf-8-sig")

missing_only = df[df["missing_required_visual"] == 1].copy()
missing_only.to_csv(OUTPUT_MISSING_ONLY, index=False, encoding="utf-8-sig")

# =========================================
# 7. 顯示結果
# =========================================
print("=== 視覺檔案檢查完成 ===")
print(f"貼文層級檢查：{OUTPUT_POST_CHECK}")
print(f"使用者摘要：{OUTPUT_USER_SUMMARY}")
print(f"缺失檔案清單：{OUTPUT_MISSING_ONLY}")

print("\n=== 整體統計 ===")
print(f"總貼文數: {len(df)}")
print(f"需要視覺檔案的貼文數: {int(df['needs_visual_file'].sum())}")
print(f"成功找到檔案數: {int(((df['needs_visual_file'] == 1) & (df['exists_in_user_folder'] == 1)).sum())}")
print(f"缺失視覺檔案數: {int(df['missing_required_visual'].sum())}")

print("\n=== check_mode 統計 ===")
print(df["check_mode"].value_counts(dropna=False))

print("\n=== 使用者摘要前幾列 ===")
print(user_summary.head())

print("\n=== 缺失檔案前幾列 ===")
print(missing_only[[
    "username", "post_id", "visual_type", "visual_path",
    "user_media_folder", "resolved_visual_path", "check_mode"
]].head())

=== 視覺檔案檢查完成 ===
貼文層級檢查：D:\時間序列\final_preprocessed_data_all\final_data_quality_check\visual_file_check_post_level.csv
使用者摘要：D:\時間序列\final_preprocessed_data_all\final_data_quality_check\visual_file_check_user_summary.csv
缺失檔案清單：D:\時間序列\final_preprocessed_data_all\final_data_quality_check\visual_file_missing_only.csv

=== 整體統計 ===
總貼文數: 76042
需要視覺檔案的貼文數: 43830
成功找到檔案數: 43827
缺失視覺檔案數: 3

=== check_mode 統計 ===
filename_matched_in_user_folder    43844
not_found                          32195
empty_path                             3
Name: check_mode, dtype: int64

=== 使用者摘要前幾列 ===
       username  num_posts  num_visual_posts  num_visual_found  \
0    156.melody        935               246               246   
1    914_planet         26                26                26   
2  _kana.words_         71                71                71   
3    _lydiawong       1084               199               199   
4    _midleave_        740               740               740   

   num_visual_missing

### 前處理


In [5]:
!pip -q install transformers pillow opencv-python pyarrow

In [33]:
# =========================================
# (1) 文字 tokenizer + 時間前處理
# 適用檔案：model_input_post_level.csv
# =========================================
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
from transformers import AutoTokenizer

# =========================================
# 1. 路徑設定
# =========================================
INPUT_CSV = "D:/時間序列/final_preprocessed_data_all/model_input_post_level.csv"

OUTPUT_DIR = Path("D:/時間序列/final_model_inputs_text_time_all")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PARQUET = OUTPUT_DIR / "text_time_features.parquet"
OUTPUT_CSV = OUTPUT_DIR / "text_time_features.csv"

# =========================================
# 2. 參數設定
# =========================================
BERT_NAME = "bert-base-chinese"
MAX_LENGTH = 128

# =========================================
# 3. 讀取資料
# =========================================
df = pd.read_csv(INPUT_CSV)
df.columns = [c.strip() for c in df.columns]

required_cols = ["username", "post_id", "user_label", "text", "taken_at"]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"缺少必要欄位: {col}")

# =========================================
# 4. 基本清理
# =========================================
df["text"] = df["text"].fillna("").astype(str)

# 時間轉 datetime（你的原始格式可為 2025-06-19 17:14:25+00:00）
df["taken_at"] = pd.to_datetime(df["taken_at"], utc=True, errors="coerce")
df = df.dropna(subset=["taken_at"]).copy()

# 排序
df = df.sort_values(["username", "taken_at", "post_id"]).reset_index(drop=True)

# =========================================
# 5. 時間前處理
# =========================================
# UTC Unix timestamp（秒）
df["tau"] = df["taken_at"].astype("int64") // 10**9

# 依照論文方式：g(tau) = 1 / (tau + 1)
df["tau_transformed"] = 1.0 / (df["tau"] + 1.0)

# 轉台灣時間（額外行為特徵）
df["taken_at_taipei"] = df["taken_at"].dt.tz_convert("Asia/Taipei")
df["hour"] = df["taken_at_taipei"].dt.hour
df["weekday"] = df["taken_at_taipei"].dt.weekday
df["is_night"] = df["hour"].apply(lambda x: 1 if x < 6 else 0)

# 相對時間：以每位使用者第一篇貼文為基準（單位：天）
df["user_first_time"] = df.groupby("username")["taken_at"].transform("min")
df["relative_days"] = (df["taken_at"] - df["user_first_time"]).dt.total_seconds() / 86400.0

# =========================================
# 6. 文字 tokenizer
# =========================================
tokenizer = AutoTokenizer.from_pretrained(BERT_NAME)

input_ids_list = []
attention_mask_list = []
token_type_ids_list = []
token_count_list = []

print("開始進行文字 tokenizer ...")
for text in tqdm(df["text"].tolist(), total=len(df)):
    enc = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
        return_attention_mask=True,
        return_token_type_ids=True
    )
    input_ids_list.append(enc["input_ids"])
    attention_mask_list.append(enc["attention_mask"])
    token_type_ids_list.append(enc["token_type_ids"])
    token_count_list.append(int(sum(enc["attention_mask"])))

df["input_ids"] = input_ids_list
df["attention_mask"] = attention_mask_list
df["token_type_ids"] = token_type_ids_list
df["token_count"] = token_count_list

# =========================================
# 7. 輸出欄位
# =========================================
keep_cols = [
    "username",
    "post_id",
    "user_label",
    "text",
    "taken_at",
    "taken_at_taipei",
    "tau",
    "tau_transformed",
    "relative_days",
    "hour",
    "weekday",
    "is_night",
    "token_count",
    "input_ids",
    "attention_mask",
    "token_type_ids",
]

# 若原本有這些欄位就保留
optional_cols = [
    "caption",
    "hashtags",
    "char_count",
    "hashtag_count",
    "p_depression_t1",
    "p_depression_tcal",
    "logit_margin",
    "pseudo_label",
    "visual_path",
    "visual_type",
]
for c in optional_cols:
    if c in df.columns and c not in keep_cols:
        keep_cols.append(c)

out_df = df[keep_cols].copy()

# parquet 可保留 list 欄位
out_df.to_parquet(OUTPUT_PARQUET, index=False)

# 另外輸出 csv（list 欄位會變字串，但方便檢查）
out_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

# =========================================
# 8. 顯示結果
# =========================================
print("=== 文字 tokenizer + 時間前處理完成 ===")
print(f"Parquet：{OUTPUT_PARQUET}")
print(f"CSV：{OUTPUT_CSV}")

print("\n=== 前幾列 ===")
print(out_df.head())

print("\n=== token_count 描述統計 ===")
print(out_df["token_count"].describe())


開始進行文字 tokenizer ...


c:\Users\Angle\Anaconda3\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


  0%|          | 0/76042 [00:00<?, ?it/s]

=== 文字 tokenizer + 時間前處理完成 ===
Parquet：D:\時間序列\final_model_inputs_text_time_all\text_time_features.parquet
CSV：D:\時間序列\final_model_inputs_text_time_all\text_time_features.csv

=== 前幾列 ===
     username                          post_id  user_label  \
0  156.melody  2960932059168283176_10006613875           0   
1  156.melody  2961656747658029011_10006613875           0   
2  156.melody  2962352246316842299_10006613875           0   
3  156.melody  2963094277490097632_10006613875           0   
4  156.melody  2963821833539625073_10006613875           0   

                                                text  \
0  朋友說這套不像警察 比較像保全 來當一日大樓保全😂 . . . . #萬聖節 #萬聖節快樂 ...   
1  買女兒的衣服🤭 . , . . #cani生活有機棉 #ootd #ootdfashion ...   
2  好像還很少人知道這家咖啡廳 一個包場的概念 . . . . #ootd #fashion #...   
3  女兒感冒第5天了 才知道以前媽媽多辛苦😂 . . . . #taiwan #ootd #oo...   
4  很適合超級愛吃蔬菜的我 多種蔬菜 原型食物 優質澱粉和蛋白質 最後搭配一個醬料 很有飽足感又...   

                   taken_at           taken_at_taipei         tau  \
0 2022-10-31 04:30:28+00:00 2022-

In [6]:
# =========================================
# (2) 圖片 / 影片代表圖前處理
# 適用檔案：model_input_post_level.csv
# =========================================

# !pip -q install pillow opencv-python

import os
import cv2
import torch
import numpy as np
import pandas as pd

from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm

# =========================================
# 1. 路徑設定
# =========================================
INPUT_CSV = r"D:/時間序列/final_preprocessed_data_all/model_input_post_level.csv"

# 你的媒體根目錄
MEDIA_ROOT = Path(r"D:/時間序列/depress_dataset/downloaded_media")

OUTPUT_DIR = Path("D:/時間序列/final_model_inputs_vision_all")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_TENSOR = OUTPUT_DIR / "vision_tensors.pt"
OUTPUT_META = OUTPUT_DIR / "vision_metadata.csv"


# =========================================
# 2. 參數設定
# =========================================
IMAGE_SIZE = 224

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

FALLBACK_TO_FIRST_FRAME = True

# =========================================
# 3. 讀取資料
# =========================================
df = pd.read_csv(INPUT_CSV)
df.columns = [c.strip() for c in df.columns]

required_cols = ["username", "post_id", "visual_path", "visual_type"]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"缺少必要欄位: {col}")

df["username"] = df["username"].astype(str)
df["visual_path"] = df["visual_path"].fillna("").astype(str)
df["visual_type"] = df["visual_type"].fillna("unknown").astype(str)

# =========================================
# 4. 路徑解析函式
# =========================================
def resolve_visual_path(username, visual_path, media_root):
    """
    依序嘗試：
    1. visual_path 本身就是可用完整路徑
    2. media_root / username / 檔名
    3. media_root / username / visual_path
    """
    if not visual_path or str(visual_path).strip() == "":
        return None, "empty_path"

    vp = str(visual_path).strip()

    # 1. visual_path 本身存在
    if os.path.exists(vp):
        return str(Path(vp)), "direct_path"

    user_dir = media_root / str(username)

    # 2. 用檔名拼接到 username 資料夾
    candidate_by_name = user_dir / Path(vp).name
    if candidate_by_name.exists():
        return str(candidate_by_name), "filename_in_user_folder"

    # 3. 若 visual_path 是相對路徑
    candidate_relative = user_dir / vp
    if candidate_relative.exists():
        return str(candidate_relative), "relative_in_user_folder"

    return None, "not_found"

# =========================================
# 5. 工具函式
# =========================================
def make_black_image(size=224):
    arr = np.zeros((size, size, 3), dtype=np.uint8)
    return Image.fromarray(arr)

def load_image_from_path(path):
    if not path or not os.path.exists(path):
        return None
    try:
        img = Image.open(path).convert("RGB")
        return img
    except Exception:
        return None

def extract_middle_frame(video_path, fallback_to_first=True):
    if not video_path or not os.path.exists(video_path):
        return None

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None

    try:
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if frame_count <= 0:
            target_idx = 0 if fallback_to_first else None
        else:
            target_idx = frame_count // 2

        if target_idx is None:
            cap.release()
            return None

        cap.set(cv2.CAP_PROP_POS_FRAMES, target_idx)
        success, frame = cap.read()

        if (not success or frame is None) and fallback_to_first:
            cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
            success, frame = cap.read()

        if not success or frame is None:
            cap.release()
            return None

        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        img = Image.fromarray(frame)
        cap.release()
        return img

    except Exception:
        cap.release()
        return None

def resize_and_normalize(img_pil, image_size=224):
    img = img_pil.resize((image_size, image_size))
    arr = np.asarray(img).astype(np.float32) / 255.0
    arr = (arr - IMAGENET_MEAN) / IMAGENET_STD
    arr = np.transpose(arr, (2, 0, 1))  # HWC -> CHW
    return arr

# =========================================
# 6. 前處理主流程
# =========================================
vision_tensors = []
meta_rows = []

print("開始進行圖片 / 影片代表圖前處理 ...")
for idx, row in tqdm(df.iterrows(), total=len(df)):
    username = row["username"]
    visual_path = row["visual_path"].strip()
    visual_type = row["visual_type"].lower().strip()

    resolved_path, path_mode = resolve_visual_path(username, visual_path, MEDIA_ROOT)

    img_pil = None
    source_type_used = "black_image"
    load_success = 0

    if visual_type == "image":
        img_pil = load_image_from_path(resolved_path)
        if img_pil is not None:
            source_type_used = "image"
            load_success = 1

    elif visual_type == "video":
        img_pil = extract_middle_frame(resolved_path, fallback_to_first=FALLBACK_TO_FIRST_FRAME)
        if img_pil is not None:
            source_type_used = "video_middle_frame"
            load_success = 1

    if img_pil is None:
        img_pil = make_black_image(IMAGE_SIZE)
        source_type_used = "black_image"
        load_success = 0

    arr = resize_and_normalize(img_pil, image_size=IMAGE_SIZE)
    vision_tensors.append(arr)

    meta_rows.append({
        "row_idx": idx,
        "username": username,
        "post_id": row["post_id"],
        "visual_path_original": visual_path,
        "resolved_visual_path": resolved_path,
        "path_mode": path_mode,
        "visual_type": visual_type,
        "source_type_used": source_type_used,
        "load_success": load_success,
    })

# 轉 tensor
vision_tensors = np.stack(vision_tensors, axis=0)
vision_tensors = torch.tensor(vision_tensors, dtype=torch.float32)

meta_df = pd.DataFrame(meta_rows)

# =========================================
# 7. 輸出
# =========================================
torch.save(vision_tensors, OUTPUT_TENSOR)
meta_df.to_csv(OUTPUT_META, index=False, encoding="utf-8-sig")

# =========================================
# 8. 顯示結果
# =========================================
print("=== 圖片 / 影片代表圖前處理完成 ===")
print(f"Tensor：{OUTPUT_TENSOR}")
print(f"Metadata：{OUTPUT_META}")

print("\n=== vision_tensors shape ===")
print(vision_tensors.shape)

print("\n=== path_mode 統計 ===")
print(meta_df["path_mode"].value_counts(dropna=False))

print("\n=== source_type_used 統計 ===")
print(meta_df["source_type_used"].value_counts(dropna=False))

print("\n=== 前幾列 metadata ===")
print(meta_df.head())

開始進行圖片 / 影片代表圖前處理 ...


  0%|          | 0/76042 [00:00<?, ?it/s]

c:\Users\Angle\Anaconda3\lib\site-packages\PIL\Image.py:2966: UserWarning: image file could not be identified because WEBP support not installed
  warnings.warn(message)


=== 圖片 / 影片代表圖前處理完成 ===
Tensor：D:\時間序列\final_model_inputs_vision_all\vision_tensors.pt
Metadata：D:\時間序列\final_model_inputs_vision_all\vision_metadata.csv

=== vision_tensors shape ===
torch.Size([76042, 3, 224, 224])

=== path_mode 統計 ===
filename_in_user_folder    43844
not_found                  32195
empty_path                     3
Name: path_mode, dtype: int64

=== source_type_used 統計 ===
black_image           34379
image                 33578
video_middle_frame     8085
Name: source_type_used, dtype: int64

=== 前幾列 metadata ===
   row_idx    username                          post_id  \
0        0  156.melody  2960932059168283176_10006613875   
1        1  156.melody  2961656747658029011_10006613875   
2        2  156.melody  2962352246316842299_10006613875   
3        3  156.melody  2963094277490097632_10006613875   
4        4  156.melody  2963821833539625073_10006613875   

                                visual_path_original resolved_visual_path  \
0  downloaded_media\156.melo

In [7]:
import numpy as np
from pathlib import Path

OUTPUT_NPY = Path(r"D:/時間序列/final_model_inputs_vision_all/vision_tensors.npy")

np.save(OUTPUT_NPY, vision_tensors.numpy())
print("已存成 npy:", OUTPUT_NPY)

已存成 npy: D:\時間序列\final_model_inputs_vision_all\vision_tensors.npy


In [48]:
import numpy as np
import torch

arr = np.load(r"D:/時間序列/final_model_inputs_vision_all/vision_tensors.npy", mmap_mode="r")
print(arr.shape)

# 若要轉 torch tensor
x = torch.from_numpy(arr)

(76042, 3, 224, 224)


<ipython-input-48-9728ba06a78b>:8: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\torch\csrc\utils\tensor_numpy.cpp:212.)
  x = torch.from_numpy(arr)


In [110]:
import ast
import random
import numpy as np
import pandas as pd
import torch
import ast

from pathlib import Path
from torch.utils.data import Dataset, DataLoader

# =========================================================
# 1. 路徑設定
# =========================================================
TEXT_TIME_PATH = Path(r"D:/時間序列/final_model_inputs_text_time_all/text_time_features.parquet")
VISION_NPY_PATH  = Path(r"D:/時間序列/final_model_inputs_vision_all/vision_tensors.npy")
VISION_META_PATH = Path(r"D:/時間序列/final_model_inputs_vision_all/vision_metadata.csv")

# =========================================================
# 2. 可調參數
# =========================================================
K = 32
BATCH_SIZE = 4
NUM_WORKERS = 0    # Windows / Jupyter 建議先用 0
SHUFFLE_TRAIN = True
SEED = 42

# 取樣策略：
# "recent" / "first" / "random"
SEQUENCE_MODE = "recent"

# =========================================================
# 3. 工具函式
# =========================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(SEED)



def ensure_list(x):
    if isinstance(x, list):
        return x
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, tuple):
        return list(x)
    if isinstance(x, str):
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, np.ndarray):
                return parsed.tolist()
            if isinstance(parsed, tuple):
                return list(parsed)
            if isinstance(parsed, list):
                return parsed
            return []
        except Exception:
            return []
    return []

def build_user_splits(usernames, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, seed=42):
    usernames = list(usernames)
    rng = random.Random(seed)
    rng.shuffle(usernames)

    n = len(usernames)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)

    train_users = usernames[:n_train]
    val_users = usernames[n_train:n_train + n_val]
    test_users = usernames[n_train + n_val:]

    return train_users, val_users, test_users

# =========================================================
# 4. 載入前處理資料
# =========================================================
text_df = pd.read_parquet(TEXT_TIME_PATH)
vision_array = np.load(VISION_NPY_PATH, mmap_mode="r")   # 這裡改成 npy/memmap
vision_meta = pd.read_csv(VISION_META_PATH)

# 基本檢查
assert len(text_df) == len(vision_array), "text_time_features 與 vision_tensors.npy 筆數不一致"
assert len(text_df) == len(vision_meta), "text_time_features 與 vision_metadata 筆數不一致"

# 重新整理索引，確保逐列對齊
text_df = text_df.reset_index(drop=True)
vision_meta = vision_meta.reset_index(drop=True)

# 檢查 username/post_id 是否一致（若 metadata 有這些欄位）
if "post_id" in vision_meta.columns and "username" in vision_meta.columns:
    mismatch_mask = (
        (text_df["post_id"].astype(str) != vision_meta["post_id"].astype(str)) |
        (text_df["username"].astype(str) != vision_meta["username"].astype(str))
    )
    mismatch_count = int(mismatch_mask.sum())
    print(f"[對齊檢查] text_df vs vision_meta 不一致筆數: {mismatch_count}")
    if mismatch_count > 0:
        raise ValueError("text_df 與 vision_meta 的 username/post_id 對不齊，請先檢查前處理輸出。")

# list 欄位轉回真正 list
for col in ["input_ids", "attention_mask", "token_type_ids"]:
    if col in text_df.columns:
        text_df[col] = text_df[col].apply(ensure_list)

# 時間排序欄位
if "taken_at" in text_df.columns:
    text_df["taken_at"] = pd.to_datetime(text_df["taken_at"], utc=True, errors="coerce")

# row_idx：對應 npy 第一維索引
text_df["row_idx"] = np.arange(len(text_df))

# =========================================================
# 5. 建立 user-level sample index
# =========================================================
def build_user_index(df):
    user_to_rows = {}
    for username, g in df.groupby("username"):
        if "taken_at" in g.columns:
            g = g.sort_values(["taken_at", "post_id"]).copy()
        else:
            g = g.sort_values(["post_id"]).copy()
        user_to_rows[username] = g["row_idx"].tolist()
    return user_to_rows

user_to_rows = build_user_index(text_df)
all_users = sorted(user_to_rows.keys())

print(f"總使用者數: {len(all_users)}")
print(f"總貼文數: {len(text_df)}")
print(f"vision_array shape: {vision_array.shape}")

# =========================================================
# 6. User-level Dataset（改讀 npy/memmap）
# =========================================================
class UserLevelDatasetNPY(Dataset):
    def __init__(
        self,
        text_df,
        vision_npy_path,
        user_to_rows,
        usernames,
        k=32,
        sequence_mode="recent"
    ):
        self.text_df = text_df
        self.vision_npy_path = vision_npy_path
        self.vision_array = np.load(self.vision_npy_path, mmap_mode="r")
        self.user_to_rows = user_to_rows
        self.usernames = list(usernames)
        self.k = k
        self.sequence_mode = sequence_mode

        example_row = self.text_df.iloc[0]
        self.text_len = len(example_row["input_ids"])
        self.vision_shape = tuple(self.vision_array[0].shape)   # (3, 224, 224)

    def __len__(self):
        return len(self.usernames)

    def _select_rows(self, rows):
        rows = list(rows)

        if len(rows) <= self.k:
            return rows

        if self.sequence_mode == "recent":
            return rows[-self.k:]
        elif self.sequence_mode == "first":
            return rows[:self.k]
        elif self.sequence_mode == "random":
            return sorted(random.sample(rows, self.k))
        else:
            raise ValueError(f"未知 sequence_mode: {self.sequence_mode}")

    def __getitem__(self, idx):
        username = self.usernames[idx]
        rows = self.user_to_rows[username]
        selected_rows = self._select_rows(rows)

        user_label = int(self.text_df.loc[selected_rows[0], "user_label"])
        n_real = len(selected_rows)

        input_ids = []
        attention_mask = []
        token_type_ids = []
        vision_list = []

        tau_transformed = []
        relative_days = []
        hour = []
        weekday = []
        is_night = []

        post_ids = []

        for ridx in selected_rows:
            row = self.text_df.iloc[ridx]

            input_ids.append(torch.tensor(row["input_ids"], dtype=torch.long))
            attention_mask.append(torch.tensor(row["attention_mask"], dtype=torch.long))
            token_type_ids.append(torch.tensor(row["token_type_ids"], dtype=torch.long))

            # 這裡改成從 npy/memmap 讀單筆
            vision_np = np.array(self.vision_array[ridx], dtype=np.float32)
            vision_list.append(torch.from_numpy(vision_np))

            tau_transformed.append(float(row["tau_transformed"]) if pd.notna(row["tau_transformed"]) else 0.0)
            relative_days.append(float(row["relative_days"]) if pd.notna(row["relative_days"]) else 0.0)
            hour.append(float(row["hour"]) if "hour" in row and pd.notna(row["hour"]) else 0.0)
            weekday.append(float(row["weekday"]) if "weekday" in row and pd.notna(row["weekday"]) else 0.0)
            is_night.append(float(row["is_night"]) if "is_night" in row and pd.notna(row["is_night"]) else 0.0)

            post_ids.append(str(row["post_id"]))

        pad_n = self.k - n_real

        if pad_n > 0:
            pad_input_ids = torch.zeros(self.text_len, dtype=torch.long)
            pad_attention_mask = torch.zeros(self.text_len, dtype=torch.long)
            pad_token_type_ids = torch.zeros(self.text_len, dtype=torch.long)
            pad_vision = torch.zeros(self.vision_shape, dtype=torch.float32)

            for _ in range(pad_n):
                input_ids.append(pad_input_ids.clone())
                attention_mask.append(pad_attention_mask.clone())
                token_type_ids.append(pad_token_type_ids.clone())
                vision_list.append(pad_vision.clone())

                tau_transformed.append(0.0)
                relative_days.append(0.0)
                hour.append(0.0)
                weekday.append(0.0)
                is_night.append(0.0)
                post_ids.append("PAD")

        input_ids = torch.stack(input_ids, dim=0)               # [K, L]
        attention_mask = torch.stack(attention_mask, dim=0)     # [K, L]
        token_type_ids = torch.stack(token_type_ids, dim=0)     # [K, L]
        vision_tensor = torch.stack(vision_list, dim=0)         # [K, 3, 224, 224]

        tau_transformed = torch.tensor(tau_transformed, dtype=torch.float32)
        relative_days = torch.tensor(relative_days, dtype=torch.float32)
        hour = torch.tensor(hour, dtype=torch.float32)
        weekday = torch.tensor(weekday, dtype=torch.float32)
        is_night = torch.tensor(is_night, dtype=torch.float32)

        post_mask = torch.tensor([1] * n_real + [0] * pad_n, dtype=torch.float32)

        return {
            "username": username,
            "label": torch.tensor(user_label, dtype=torch.long),
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "token_type_ids": token_type_ids,
            "vision_tensor": vision_tensor,
            "tau_transformed": tau_transformed,
            "relative_days": relative_days,
            "hour": hour,
            "weekday": weekday,
            "is_night": is_night,
            "post_mask": post_mask,
            "num_real_posts": torch.tensor(n_real, dtype=torch.long),
            "post_ids": post_ids,
        }

# =========================================================
# 7. train / val / test split
# =========================================================
train_users, val_users, test_users = build_user_splits(
    all_users,
    train_ratio=0.7,
    val_ratio=0.15,
    test_ratio=0.15,
    seed=SEED
)

print(f"Train users: {len(train_users)}")
print(f"Val users:   {len(val_users)}")
print(f"Test users:  {len(test_users)}")

# =========================================================
# 8. 建立 Dataset
# =========================================================
train_dataset = UserLevelDatasetNPY(
    text_df=text_df,
    vision_npy_path=VISION_NPY_PATH,
    user_to_rows=user_to_rows,
    usernames=train_users,
    k=K,
    sequence_mode=SEQUENCE_MODE
)

val_dataset = UserLevelDatasetNPY(
    text_df=text_df,
    vision_npy_path=VISION_NPY_PATH,
    user_to_rows=user_to_rows,
    usernames=val_users,
    k=K,
    sequence_mode=SEQUENCE_MODE
)

test_dataset = UserLevelDatasetNPY(
    text_df=text_df,
    vision_npy_path=VISION_NPY_PATH,
    user_to_rows=user_to_rows,
    usernames=test_users,
    k=K,
    sequence_mode=SEQUENCE_MODE
)

# =========================================================
# 9. DataLoader
# =========================================================
def user_level_collate_fn(batch):
    return {
        "username": [x["username"] for x in batch],
        "label": torch.stack([x["label"] for x in batch], dim=0),                       # [B]
        "input_ids": torch.stack([x["input_ids"] for x in batch], dim=0),               # [B, K, L]
        "attention_mask": torch.stack([x["attention_mask"] for x in batch], dim=0),     # [B, K, L]
        "token_type_ids": torch.stack([x["token_type_ids"] for x in batch], dim=0),     # [B, K, L]
        "vision_tensor": torch.stack([x["vision_tensor"] for x in batch], dim=0),       # [B, K, 3, 224, 224]
        "tau_transformed": torch.stack([x["tau_transformed"] for x in batch], dim=0),   # [B, K]
        "relative_days": torch.stack([x["relative_days"] for x in batch], dim=0),       # [B, K]
        "hour": torch.stack([x["hour"] for x in batch], dim=0),                         # [B, K]
        "weekday": torch.stack([x["weekday"] for x in batch], dim=0),                   # [B, K]
        "is_night": torch.stack([x["is_night"] for x in batch], dim=0),                 # [B, K]
        "post_mask": torch.stack([x["post_mask"] for x in batch], dim=0),               # [B, K]
        "num_real_posts": torch.stack([x["num_real_posts"] for x in batch], dim=0),     # [B]
        "post_ids": [x["post_ids"] for x in batch],
    }

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,#SHUFFLE_TRAIN,
    num_workers=NUM_WORKERS,
    collate_fn=user_level_collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=user_level_collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=user_level_collate_fn
)

# =========================================================
# 10. 測試一個 batch
# =========================================================
batch = next(iter(train_loader))

print("=== Batch keys ===")
print(batch.keys())

print("\n=== Shapes ===")
print("label:", batch["label"].shape)
print("input_ids:", batch["input_ids"].shape)
print("attention_mask:", batch["attention_mask"].shape)
print("token_type_ids:", batch["token_type_ids"].shape)
print("vision_tensor:", batch["vision_tensor"].shape)
print("tau_transformed:", batch["tau_transformed"].shape)
print("relative_days:", batch["relative_days"].shape)
print("hour:", batch["hour"].shape)
print("weekday:", batch["weekday"].shape)
print("is_night:", batch["is_night"].shape)
print("post_mask:", batch["post_mask"].shape)
print("num_real_posts:", batch["num_real_posts"].shape)

print("\n=== Example usernames ===")
print(batch["username"][:2])

print("\n=== Example labels ===")
print(batch["label"][:2])

[對齊檢查] text_df vs vision_meta 不一致筆數: 0
總使用者數: 161
總貼文數: 76042
vision_array shape: (76042, 3, 224, 224)
Train users: 112
Val users:   24
Test users:  25
=== Batch keys ===
dict_keys(['username', 'label', 'input_ids', 'attention_mask', 'token_type_ids', 'vision_tensor', 'tau_transformed', 'relative_days', 'hour', 'weekday', 'is_night', 'post_mask', 'num_real_posts', 'post_ids'])

=== Shapes ===
label: torch.Size([4])
input_ids: torch.Size([4, 32, 128])
attention_mask: torch.Size([4, 32, 128])
token_type_ids: torch.Size([4, 32, 128])
vision_tensor: torch.Size([4, 32, 3, 224, 224])
tau_transformed: torch.Size([4, 32])
relative_days: torch.Size([4, 32])
hour: torch.Size([4, 32])
weekday: torch.Size([4, 32])
is_night: torch.Size([4, 32])
post_mask: torch.Size([4, 32])
num_real_posts: torch.Size([4])

=== Example usernames ===
['help_.me54', 'lueg0810']

=== Example labels ===
tensor([1, 0])


# 4/8 


### 第一版 baseline 模型

In [52]:
!pip -q install transformers

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, CLIPVisionModel

In [142]:
class Time2Vec(nn.Module):
    """
    依照論文概念實作的 time2vec
    輸入: [B, K] 或 [N]
    輸出: [B, K, out_dim]
    """
    def __init__(self, out_dim):
        super().__init__()
        assert out_dim >= 1, "out_dim 至少要 >= 1"
        self.out_dim = out_dim

        # 第一維是線性項，其餘是 periodic 項
        self.w0 = nn.Parameter(torch.randn(1))
        self.b0 = nn.Parameter(torch.randn(1))

        self.w = nn.Parameter(torch.randn(out_dim - 1))
        self.b = nn.Parameter(torch.randn(out_dim - 1))

    def forward(self, tau):
        """
        tau: [B, K] 或 [N]
        """
        if tau.dim() == 1:
            tau = tau.unsqueeze(-1)   # [N, 1]
        else:
            tau = tau.unsqueeze(-1)   # [B, K, 1]

        v0 = self.w0 * tau + self.b0
        if self.out_dim == 1:
            return v0

        vp = torch.sin(tau * self.w + self.b)   # broadcast
        return torch.cat([v0, vp], dim=-1)

In [143]:
class BaselineMultimodalModel(nn.Module):
    def __init__(
        self,
        text_model_name="bert-base-chinese",
        vision_model_name="openai/clip-vit-base-patch32",
        text_hidden_dim=768,
        vision_hidden_dim=768,
        proj_dim=256,
        time_dim=16,
        lstm_hidden_dim=256,
        num_lstm_layers=1,
        dropout=0.2,
        freeze_text=True, #False
        freeze_vision=True,
        use_token_type_ids=True,
    ):
        super().__init__()

        # ========= 文字 encoder =========
        self.text_encoder = AutoModel.from_pretrained(text_model_name)
        self.use_token_type_ids = use_token_type_ids

        if freeze_text:
            for p in self.text_encoder.parameters():
                p.requires_grad = False

        # ========= 圖片 encoder（CLIP vision encoder） =========
        self.vision_encoder = CLIPVisionModel.from_pretrained(vision_model_name)

        if freeze_vision:
            for p in self.vision_encoder.parameters():
                p.requires_grad = False

        # ========= 時間 encoder（time2vec） =========
        self.time2vec = Time2Vec(out_dim=time_dim)

        # ========= projection =========
        self.text_proj = nn.Linear(text_hidden_dim, proj_dim)
        self.vision_proj = nn.Linear(vision_hidden_dim, proj_dim)
        self.time_proj = nn.Linear(time_dim, proj_dim)

        # 額外時間行為特徵：hour, weekday, is_night
        self.behavior_proj = nn.Linear(3, proj_dim)

        # ========= post-level fusion =========
        fusion_in_dim = proj_dim * 4   # text + vision + time + behavior
        self.post_fusion = nn.Sequential(
            nn.Linear(fusion_in_dim, proj_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        # ========= user-level sequence model =========
        self.user_encoder = nn.LSTM(
            input_size=proj_dim,
            hidden_size=lstm_hidden_dim,
            num_layers=num_lstm_layers,
            batch_first=True,
            dropout=dropout if num_lstm_layers > 1 else 0.0,
            bidirectional=True,
        )

        self.classifier = nn.Sequential(
            nn.Linear(lstm_hidden_dim * 2, lstm_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden_dim, 1)
        )

    def masked_mean_pool(self, x, mask):
        """
        x: [B, K, D]
        mask: [B, K]
        """
        mask = mask.unsqueeze(-1)  # [B, K, 1]
        x = x * mask
        denom = mask.sum(dim=1).clamp(min=1e-6)
        return x.sum(dim=1) / denom

    def forward(
        self,
        input_ids,
        attention_mask,
        token_type_ids,
        vision_tensor,
        tau_transformed,
        hour,
        weekday,
        is_night,
        post_mask,
    ):
        """
        input_ids:       [B, K, L]
        attention_mask:  [B, K, L]
        token_type_ids:  [B, K, L]
        vision_tensor:   [B, K, 3, 224, 224]
        tau_transformed: [B, K]
        hour:            [B, K]
        weekday:         [B, K]
        is_night:        [B, K]
        post_mask:       [B, K]
        """

        B, K, L = input_ids.shape

        # =====================================================
        # 1. 文字編碼
        # =====================================================
        flat_input_ids = input_ids.view(B * K, L)
        flat_attention_mask = attention_mask.view(B * K, L)
        flat_token_type_ids = token_type_ids.view(B * K, L)

        if self.use_token_type_ids:
            text_outputs = self.text_encoder(
                input_ids=flat_input_ids,
                attention_mask=flat_attention_mask,
                token_type_ids=flat_token_type_ids
            )
        else:
            text_outputs = self.text_encoder(
                input_ids=flat_input_ids,
                attention_mask=flat_attention_mask
            )

        # BERT [CLS]
        text_feat = text_outputs.last_hidden_state[:, 0, :]   # [B*K, 768]
        text_feat = self.text_proj(text_feat)                 # [B*K, proj_dim]

        # =====================================================
        # 2. 圖片編碼（CLIP vision）
        # =====================================================
        flat_vision = vision_tensor.view(B * K, *vision_tensor.shape[2:])  # [B*K, 3, 224, 224]

        vision_outputs = self.vision_encoder(pixel_values=flat_vision)
        vision_feat = vision_outputs.pooler_output   # [B*K, 768]
        vision_feat = self.vision_proj(vision_feat)  # [B*K, proj_dim]

        # =====================================================
        # 3. 時間編碼（time2vec）
        # =====================================================
        time_feat = self.time2vec(tau_transformed)   # [B, K, time_dim]
        time_feat = self.time_proj(time_feat)        # [B, K, proj_dim]

        # =====================================================
        # 4. 行為時間特徵
        # =====================================================
        behavior_feat = torch.stack([hour, weekday, is_night], dim=-1)  # [B, K, 3]
        behavior_feat = self.behavior_proj(behavior_feat)               # [B, K, proj_dim]

        # =====================================================
        # 5. post-level fusion
        # =====================================================
        text_feat = text_feat.view(B, K, -1)
        vision_feat = vision_feat.view(B, K, -1)

        post_feat = torch.cat([text_feat, vision_feat, time_feat, behavior_feat], dim=-1)
        post_feat = self.post_fusion(post_feat)   # [B, K, proj_dim]

        # padding 位置清零
        post_feat = post_feat * post_mask.unsqueeze(-1)

        # =====================================================
        # 6. user-level sequence encoding
        # =====================================================
        seq_out, _ = self.user_encoder(post_feat)   # [B, K, 2*lstm_hidden_dim]

        # masked mean pooling
        user_feat = self.masked_mean_pool(seq_out, post_mask)   # [B, 2*lstm_hidden_dim]

        # =====================================================
        # 7. binary classification
        # =====================================================
        logits = self.classifier(user_feat).squeeze(-1)   # [B]

        return logits

In [144]:
# =========================================================
# 4. 訓練與驗證函式
# =========================================================

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
import numpy as np
import torch

def train_one_epoch(model, loader, optimizer, criterion, device, epoch_idx=1, log_every=10):
    model.train()
    total_loss = 0.0

    all_labels = []
    all_probs = []
    all_preds = []

    pbar = tqdm(loader, desc=f"Train Epoch {epoch_idx}", leave=True)

    for step, batch in enumerate(pbar, start=1):
        labels = batch["label"].float().to(device)

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)
        vision_tensor = batch["vision_tensor"].to(device)

        tau_transformed = batch["tau_transformed"].to(device)
        hour = batch["hour"].to(device)
        weekday = batch["weekday"].to(device)
        is_night = batch["is_night"].to(device)
        post_mask = batch["post_mask"].to(device)

        optimizer.zero_grad()

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            vision_tensor=vision_tensor,
            tau_transformed=tau_transformed,
            hour=hour,
            weekday=weekday,
            is_night=is_night,
            post_mask=post_mask,
        )

        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        probs = torch.sigmoid(logits).detach().cpu().numpy()
        preds = (probs >= 0.5).astype(int)

        all_labels.extend(labels.cpu().numpy().astype(int).tolist())
        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())

        avg_loss_so_far = total_loss / step
        pbar.set_postfix(loss=f"{avg_loss_so_far:.4f}")

        if step % log_every == 0:
            print(f"[Train][Epoch {epoch_idx}] step {step}/{len(loader)} | avg_loss={avg_loss_so_far:.4f}")

    avg_loss = total_loss / len(loader)

    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="binary", zero_division=0
    )

    try:
        auc = roc_auc_score(all_labels, all_probs)
    except:
        auc = np.nan

    return {
        "loss": avg_loss,
        "acc": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc
    }


@torch.no_grad()
def evaluate(model, loader, criterion, device, epoch_idx=1, split_name="Val"):
    model.eval()
    total_loss = 0.0

    all_labels = []
    all_probs = []
    all_preds = []

    pbar = tqdm(loader, desc=f"{split_name} Epoch {epoch_idx}", leave=True)

    for step, batch in enumerate(pbar, start=1):
        labels = batch["label"].float().to(device)

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)
        vision_tensor = batch["vision_tensor"].to(device)

        tau_transformed = batch["tau_transformed"].to(device)
        hour = batch["hour"].to(device)
        weekday = batch["weekday"].to(device)
        is_night = batch["is_night"].to(device)
        post_mask = batch["post_mask"].to(device)

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            vision_tensor=vision_tensor,
            tau_transformed=tau_transformed,
            hour=hour,
            weekday=weekday,
            is_night=is_night,
            post_mask=post_mask,
        )

        loss = criterion(logits, labels)
        total_loss += loss.item()

        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs >= 0.5).astype(int)

        all_labels.extend(labels.cpu().numpy().astype(int).tolist())
        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())

        avg_loss_so_far = total_loss / step
        pbar.set_postfix(loss=f"{avg_loss_so_far:.4f}")

    avg_loss = total_loss / len(loader)

    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="binary", zero_division=0
    )

    try:
        auc = roc_auc_score(all_labels, all_probs)
    except:
        auc = np.nan

    return {
        "loss": avg_loss,
        "acc": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc
    }

In [145]:
# =========================================================
# 5. 建立模型與測試一個 batch
# =========================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device =", device)

model = BaselineMultimodalModel(
    text_model_name="bert-base-chinese",
    vision_model_name="openai/clip-vit-base-patch32",
    text_hidden_dim=768,
    vision_hidden_dim=768,
    proj_dim=256,
    time_dim=16,
    lstm_hidden_dim=256,
    num_lstm_layers=1,
    dropout=0.2,
    freeze_text=True,     # 若顯存不夠可改 True
    freeze_vision=True,    # 第一版 baseline 建議先凍結 vision
    use_token_type_ids=True,
).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=2e-5)

# 先測試一個 batch forward
batch = next(iter(train_loader))

with torch.no_grad():
    logits = model(
        input_ids=batch["input_ids"].to(device),
        attention_mask=batch["attention_mask"].to(device),
        token_type_ids=batch["token_type_ids"].to(device),
        vision_tensor=batch["vision_tensor"].to(device),
        tau_transformed=batch["tau_transformed"].to(device),
        hour=batch["hour"].to(device),
        weekday=batch["weekday"].to(device),
        is_night=batch["is_night"].to(device),
        post_mask=batch["post_mask"].to(device),
    )

print("logits shape:", logits.shape)
print("logits[:5]:", logits[:5])

device = cuda
logits shape: torch.Size([4])
logits[:5]: tensor([-0.0433,  0.0197,     nan,     nan], device='cuda:0')


In [146]:
model.train()

for i, batch in enumerate(train_loader):
    print(f"Running batch {i+1}")
    
    labels = batch["label"].float().to(device)

    logits = model(
        input_ids=batch["input_ids"].to(device),
        attention_mask=batch["attention_mask"].to(device),
        token_type_ids=batch["token_type_ids"].to(device),
        vision_tensor=batch["vision_tensor"].to(device),
        tau_transformed=batch["tau_transformed"].to(device),
        hour=batch["hour"].to(device),
        weekday=batch["weekday"].to(device),
        is_night=batch["is_night"].to(device),
        post_mask=batch["post_mask"].to(device),
    )

    loss = criterion(logits, labels)
    print("loss =", loss.item())

    if i == 1:
        break

Running batch 1
loss = nan
Running batch 2
loss = 0.6665678024291992


In [73]:
EPOCHS = 1 # 5

best_val_f1 = -1
best_state = None

for epoch in range(1, EPOCHS + 1):
    print(f"\n========== Epoch {epoch}/{EPOCHS} ==========")

    train_metrics = train_one_epoch(
        model, train_loader, optimizer, criterion, device,
        epoch_idx=epoch,
        log_every=5
    )

    val_metrics = evaluate(
        model, val_loader, criterion, device,
        epoch_idx=epoch,
        split_name="Val"
    )

    print(f"[Train][Epoch {epoch}] {train_metrics}")
    print(f"[Val][Epoch {epoch}]   {val_metrics}")

    if val_metrics["f1"] > best_val_f1:
        best_val_f1 = val_metrics["f1"]
        best_state = {k: v.cpu() for k, v in model.state_dict().items()}
        print(f"*** Best model updated at epoch {epoch}, val_f1={best_val_f1:.4f}")

if best_state is not None:
    model.load_state_dict(best_state)

test_metrics = evaluate(model, test_loader, criterion, device, epoch_idx=0, split_name="Test")
print("\n[Test]", test_metrics)


========== Epoch 1/1 ==========


Train Epoch 1:   0%|          | 0/28 [00:00<?, ?it/s]

[Train][Epoch 1] step 5/28 | avg_loss=nan
[Train][Epoch 1] step 10/28 | avg_loss=nan
[Train][Epoch 1] step 15/28 | avg_loss=nan
[Train][Epoch 1] step 20/28 | avg_loss=nan
[Train][Epoch 1] step 25/28 | avg_loss=nan


Val Epoch 1:   0%|          | 0/6 [00:00<?, ?it/s]

[Train][Epoch 1] {'loss': nan, 'acc': 0.6517857142857143, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'auc': nan}
[Val][Epoch 1]   {'loss': nan, 'acc': 0.7916666666666666, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'auc': nan}
*** Best model updated at epoch 1, val_f1=0.0000


Test Epoch 0:   0%|          | 0/7 [00:00<?, ?it/s]


[Test] {'loss': nan, 'acc': 0.8, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'auc': nan}


In [67]:
EPOCHS = 5

best_val_f1 = -1
best_state = None

for epoch in range(1, EPOCHS + 1):
    train_metrics = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_metrics = evaluate(model, val_loader, criterion, device)

    print(f"\nEpoch {epoch}")
    print("[Train]", train_metrics)
    print("[Val]  ", val_metrics)

    if val_metrics["f1"] > best_val_f1:
        best_val_f1 = val_metrics["f1"]
        best_state = {k: v.cpu() for k, v in model.state_dict().items()}

# 載入最佳權重
if best_state is not None:
    model.load_state_dict(best_state)

test_metrics = evaluate(model, test_loader, criterion, device)
print("\n[Test]", test_metrics)

KeyboardInterrupt: 

### 文字 + 時間 baseline

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel

class Time2Vec(nn.Module):
    def __init__(self, out_dim):
        super().__init__()
        assert out_dim >= 1
        self.out_dim = out_dim

        self.w0 = nn.Parameter(torch.randn(1))
        self.b0 = nn.Parameter(torch.randn(1))

        self.w = nn.Parameter(torch.randn(out_dim - 1))
        self.b = nn.Parameter(torch.randn(out_dim - 1))

    def forward(self, tau):
        # tau: [B, K]
        tau = tau.unsqueeze(-1)   # [B, K, 1]

        v0 = self.w0 * tau + self.b0
        if self.out_dim == 1:
            return v0

        vp = torch.sin(tau * self.w + self.b)
        return torch.cat([v0, vp], dim=-1)


class TextTimeBaselineModel(nn.Module):
    def __init__(
        self,
        text_model_name="bert-base-chinese",
        text_hidden_dim=768,
        proj_dim=256,
        time_dim=16,
        behavior_dim=32,
        lstm_hidden_dim=256,
        num_lstm_layers=1,
        dropout=0.2,
        freeze_text=True,
        use_token_type_ids=True,
    ):
        super().__init__()

        # ========= 文字 encoder =========
        self.text_encoder = AutoModel.from_pretrained(text_model_name)
        self.use_token_type_ids = use_token_type_ids

        if freeze_text:
            for p in self.text_encoder.parameters():
                p.requires_grad = False

        # ========= 時間 encoder =========
        self.time2vec = Time2Vec(out_dim=time_dim)

        # ========= projection =========
        self.text_proj = nn.Linear(text_hidden_dim, proj_dim)
        self.time_proj = nn.Linear(time_dim, proj_dim)

        # 行為時間特徵：hour, weekday, is_night
        self.behavior_proj = nn.Sequential(
            nn.Linear(3, behavior_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(behavior_dim, proj_dim)
        )

        # ========= post-level fusion =========
        fusion_in_dim = proj_dim * 3   # text + time + behavior
        self.post_fusion = nn.Sequential(
            nn.Linear(fusion_in_dim, proj_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        # ========= user-level sequence model =========
        self.user_encoder = nn.LSTM(
            input_size=proj_dim,
            hidden_size=lstm_hidden_dim,
            num_layers=num_lstm_layers,
            batch_first=True,
            dropout=dropout if num_lstm_layers > 1 else 0.0,
            bidirectional=True,
        )

        self.classifier = nn.Sequential(
            nn.Linear(lstm_hidden_dim * 2, lstm_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden_dim, 1)
        )

    def masked_mean_pool(self, x, mask):
        # x: [B, K, D], mask: [B, K]
        mask = mask.unsqueeze(-1)  # [B, K, 1]
        x = x * mask
        denom = mask.sum(dim=1).clamp(min=1e-6)
        return x.sum(dim=1) / denom

    def forward(
        self,
        input_ids,
        attention_mask,
        token_type_ids,
        tau_transformed,
        hour,
        weekday,
        is_night,
        post_mask,
    ):
        """
        input_ids:       [B, K, L]
        attention_mask:  [B, K, L]
        token_type_ids:  [B, K, L]
        tau_transformed: [B, K]
        hour:            [B, K]
        weekday:         [B, K]
        is_night:        [B, K]
        post_mask:       [B, K]
        """

        B, K, L = input_ids.shape

        # =====================================================
        # 1. 文字編碼
        # =====================================================
        flat_input_ids = input_ids.view(B * K, L)
        flat_attention_mask = attention_mask.view(B * K, L)
        flat_token_type_ids = token_type_ids.view(B * K, L)

        if self.use_token_type_ids:
            text_outputs = self.text_encoder(
                input_ids=flat_input_ids,
                attention_mask=flat_attention_mask,
                token_type_ids=flat_token_type_ids
            )
        else:
            text_outputs = self.text_encoder(
                input_ids=flat_input_ids,
                attention_mask=flat_attention_mask
            )

        text_feat = text_outputs.last_hidden_state[:, 0, :]   # [B*K, 768]
        text_feat = self.text_proj(text_feat)                 # [B*K, proj_dim]
        text_feat = text_feat.view(B, K, -1)                 # [B, K, proj_dim]

        # =====================================================
        # 2. time2vec
        # =====================================================
        # time_feat = self.time2vec(tau_transformed)   # [B, K, time_dim]
        time_feat = self.time2vec(relative_days)
        time_feat = self.time_proj(time_feat)        # [B, K, proj_dim]

        # =====================================================
        # 3. 行為時間特徵
        # =====================================================
        behavior_feat = torch.stack([hour, weekday, is_night], dim=-1)  # [B, K, 3]
        behavior_feat = self.behavior_proj(behavior_feat)               # [B, K, proj_dim]

        # =====================================================
        # 4. post-level fusion
        # =====================================================
        post_feat = torch.cat([text_feat, time_feat, behavior_feat], dim=-1)
        post_feat = self.post_fusion(post_feat)   # [B, K, proj_dim]

        # padding 位置清零
        post_feat = post_feat * post_mask.unsqueeze(-1)

        # =====================================================
        # 5. user-level sequence encoding
        # =====================================================
        seq_out, _ = self.user_encoder(post_feat)   # [B, K, 2*lstm_hidden_dim]
        user_feat = self.masked_mean_pool(seq_out, post_mask)

        # =====================================================
        # 6. binary classification
        # =====================================================
        logits = self.classifier(user_feat).squeeze(-1)   # [B]
        return logits

In [90]:
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
import numpy as np
import torch

def train_one_epoch_text_time(model, loader, optimizer, criterion, device, epoch_idx=1, log_every=10):
    model.train()
    total_loss = 0.0

    all_labels = []
    all_probs = []
    all_preds = []

    pbar = tqdm(loader, desc=f"Train Epoch {epoch_idx}", leave=True)

    for step, batch in enumerate(pbar, start=1):
        labels = batch["label"].float().to(device)

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)

        tau_transformed = batch["tau_transformed"].to(device)
        hour = batch["hour"].to(device)
        weekday = batch["weekday"].to(device)
        is_night = batch["is_night"].to(device)
        post_mask = batch["post_mask"].to(device)

        optimizer.zero_grad()

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            tau_transformed=tau_transformed,
            hour=hour,
            weekday=weekday,
            is_night=is_night,
            post_mask=post_mask,
        )

        loss = criterion(logits, labels)
        loss.backward()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()

        probs = torch.sigmoid(logits).detach().cpu().numpy()
        preds = (probs >= 0.5).astype(int)

        all_labels.extend(labels.cpu().numpy().astype(int).tolist())
        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())

        avg_loss_so_far = total_loss / step
        pbar.set_postfix(loss=f"{avg_loss_so_far:.4f}")

        if step % log_every == 0:
            print(f"[Train][Epoch {epoch_idx}] step {step}/{len(loader)} | avg_loss={avg_loss_so_far:.4f}")

    avg_loss = total_loss / len(loader)

    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="binary", zero_division=0
    )

    try:
        auc = roc_auc_score(all_labels, all_probs)
    except:
        auc = np.nan

    return {
        "loss": avg_loss,
        "acc": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc
    }


@torch.no_grad()
def evaluate_text_time(model, loader, criterion, device, epoch_idx=1, split_name="Val"):
    model.eval()
    total_loss = 0.0

    all_labels = []
    all_probs = []
    all_preds = []

    pbar = tqdm(loader, desc=f"{split_name} Epoch {epoch_idx}", leave=True)

    for step, batch in enumerate(pbar, start=1):
        labels = batch["label"].float().to(device)

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)

        tau_transformed = batch["tau_transformed"].to(device)
        hour = batch["hour"].to(device)
        weekday = batch["weekday"].to(device)
        is_night = batch["is_night"].to(device)
        post_mask = batch["post_mask"].to(device)

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            tau_transformed=tau_transformed,
            hour=hour,
            weekday=weekday,
            is_night=is_night,
            post_mask=post_mask,
        )

        loss = criterion(logits, labels)
        total_loss += loss.item()

        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs >= 0.5).astype(int)

        all_labels.extend(labels.cpu().numpy().astype(int).tolist())
        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())

        avg_loss_so_far = total_loss / step
        pbar.set_postfix(loss=f"{avg_loss_so_far:.4f}")

    avg_loss = total_loss / len(loader)

    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="binary", zero_division=0
    )

    try:
        auc = roc_auc_score(all_labels, all_probs)
    except:
        auc = np.nan

    return {
        "loss": avg_loss,
        "acc": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc
    }

In [91]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device =", device)

model = TextTimeBaselineModel(
    text_model_name="bert-base-chinese",
    text_hidden_dim=768,
    proj_dim=256,
    time_dim=16,
    behavior_dim=32,
    lstm_hidden_dim=256,
    num_lstm_layers=1,
    dropout=0.2,
    freeze_text=True,      # 若太慢可改 True
    use_token_type_ids=True,
).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-5
)

# 測試一個 batch
batch = next(iter(train_loader))

with torch.no_grad():
    logits = model(
        input_ids=batch["input_ids"].to(device),
        attention_mask=batch["attention_mask"].to(device),
        token_type_ids=batch["token_type_ids"].to(device),
        tau_transformed=batch["tau_transformed"].to(device),
        hour=batch["hour"].to(device),
        weekday=batch["weekday"].to(device),
        is_night=batch["is_night"].to(device),
        post_mask=batch["post_mask"].to(device),
    )

print("logits shape:", logits.shape)
print("logits[:5]:", logits[:5])

device = cuda
logits shape: torch.Size([4])
logits[:5]: tensor([-0.0053,  0.0206,  0.0004, -0.0075], device='cuda:0')


In [95]:
EPOCHS = 1

best_val_f1 = -1
best_state = None

for epoch in range(1, EPOCHS + 1):
    print(f"\n========== Epoch {epoch}/{EPOCHS} ==========")

    train_metrics = train_one_epoch_text_time(
        model, train_loader, optimizer, criterion, device,
        epoch_idx=epoch,
        log_every=2
    )

    val_metrics = evaluate_text_time(
        model, val_loader, criterion, device,
        epoch_idx=epoch,
        split_name="Val"
    )

    print(f"[Train][Epoch {epoch}] {train_metrics}")
    print(f"[Val][Epoch {epoch}]   {val_metrics}")

    if val_metrics["f1"] > best_val_f1:
        best_val_f1 = val_metrics["f1"]
        best_state = {k: v.cpu() for k, v in model.state_dict().items()}
        print(f"*** Best model updated at epoch {epoch}, val_f1={best_val_f1:.4f}")

if best_state is not None:
    model.load_state_dict(best_state)

test_metrics = evaluate_text_time(model, test_loader, criterion, device, epoch_idx=0, split_name="Test")
print("\n[Test]", test_metrics)


========== Epoch 1/1 ==========


Train Epoch 1:   0%|          | 0/28 [00:00<?, ?it/s]

RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.

In [81]:
batch = next(iter(train_loader))

print("input_ids nan:", torch.isnan(batch["input_ids"].float()).any().item())
print("attention_mask nan:", torch.isnan(batch["attention_mask"].float()).any().item())
print("token_type_ids nan:", torch.isnan(batch["token_type_ids"].float()).any().item())
print("vision_tensor nan:", torch.isnan(batch["vision_tensor"]).any().item())
print("tau_transformed nan:", torch.isnan(batch["tau_transformed"]).any().item())
print("relative_days nan:", torch.isnan(batch["relative_days"]).any().item())
print("hour nan:", torch.isnan(batch["hour"]).any().item())
print("weekday nan:", torch.isnan(batch["weekday"]).any().item())
print("is_night nan:", torch.isnan(batch["is_night"]).any().item())
print("post_mask nan:", torch.isnan(batch["post_mask"]).any().item())
print("label nan:", torch.isnan(batch["label"].float()).any().item())

input_ids nan: False
attention_mask nan: False
token_type_ids nan: False
vision_tensor nan: False
tau_transformed nan: False
relative_days nan: False
hour nan: False
weekday nan: False
is_night nan: False
post_mask nan: False
label nan: False


In [84]:
criterion = torch.nn.BCEWithLogitsLoss()
labels = batch["label"].float().to(device)

loss = criterion(logits, labels)
print("loss =", loss.item())
print("loss is nan =", torch.isnan(loss).item())

loss = nan
loss is nan = True


In [85]:
def check_loader_label_distribution(loader, name="loader"):
    all_labels = []
    for batch in loader:
        all_labels.extend(batch["label"].numpy().tolist())
    all_labels = np.array(all_labels)
    print(f"{name}: total={len(all_labels)}, pos={all_labels.sum()}, neg={(all_labels==0).sum()}, pos_rate={all_labels.mean():.4f}")

check_loader_label_distribution(train_loader, "train")
check_loader_label_distribution(val_loader, "val")
check_loader_label_distribution(test_loader, "test")

train: total=112, pos=39, neg=73, pos_rate=0.3482
val: total=24, pos=5, neg=19, pos_rate=0.2083
test: total=25, pos=5, neg=20, pos_rate=0.2000


In [83]:
model.eval()

with torch.no_grad():
    logits = model(
        input_ids=batch["input_ids"].to(device),
        attention_mask=batch["attention_mask"].to(device),
        token_type_ids=batch["token_type_ids"].to(device),
        # vision_tensor=batch["vision_tensor"].to(device),
        tau_transformed=batch["tau_transformed"].to(device),
        hour=batch["hour"].to(device),
        weekday=batch["weekday"].to(device),
        is_night=batch["is_night"].to(device),
        post_mask=batch["post_mask"].to(device),
    )

print("logits has nan:", torch.isnan(logits).any().item())
print("logits[:10] =", logits[:10])

logits has nan: True
logits[:10] = tensor([nan, nan, nan, nan], device='cuda:0')


### 初始化

In [86]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

fresh_model = TextTimeBaselineModel(
    text_model_name="bert-base-chinese",
    text_hidden_dim=768,
    proj_dim=256,
    time_dim=16,
    behavior_dim=32,
    lstm_hidden_dim=256,
    num_lstm_layers=1,
    dropout=0.2,
    freeze_text=False,
    use_token_type_ids=True,
).to(device)

fresh_model.eval()
batch = next(iter(train_loader))

with torch.no_grad():
    logits = fresh_model(
        input_ids=batch["input_ids"].to(device),
        attention_mask=batch["attention_mask"].to(device),
        token_type_ids=batch["token_type_ids"].to(device),
        tau_transformed=batch["tau_transformed"].to(device),
        hour=batch["hour"].to(device),
        weekday=batch["weekday"].to(device),
        is_night=batch["is_night"].to(device),
        post_mask=batch["post_mask"].to(device),
    )

print("fresh logits has nan:", torch.isnan(logits).any().item())
print("fresh logits[:10]:", logits[:10])

fresh logits has nan: False
fresh logits[:10]: tensor([0.0147, 0.0199, 0.0178, 0.0145], device='cuda:0')


In [100]:
import torch
import torch.nn as nn
from transformers import AutoModel

class Time2Vec(nn.Module):
    def __init__(self, out_dim):
        super().__init__()
        assert out_dim >= 1
        self.out_dim = out_dim

        self.w0 = nn.Parameter(torch.randn(1) * 0.01)
        self.b0 = nn.Parameter(torch.zeros(1))

        if out_dim > 1:
            self.w = nn.Parameter(torch.randn(out_dim - 1) * 0.01)
            self.b = nn.Parameter(torch.zeros(out_dim - 1))

    def forward(self, t):
        # t: [B, K]
        t = t.unsqueeze(-1)  # [B, K, 1]

        v0 = self.w0 * t + self.b0
        if self.out_dim == 1:
            return v0

        vp = torch.sin(t * self.w + self.b)
        return torch.cat([v0, vp], dim=-1)


class TextTimeBaselineModelStable(nn.Module):
    def __init__(
        self,
        text_model_name="bert-base-chinese",
        text_hidden_dim=768,
        proj_dim=64,
        time_dim=8,
        behavior_hidden_dim=16,
        lstm_hidden_dim=64,
        num_lstm_layers=1,
        dropout=0.1,
        freeze_text=True,
        use_token_type_ids=True,
    ):
        super().__init__()

        self.text_encoder = AutoModel.from_pretrained(text_model_name)
        self.use_token_type_ids = use_token_type_ids

        if freeze_text:
            for p in self.text_encoder.parameters():
                p.requires_grad = False

        self.time2vec = Time2Vec(out_dim=time_dim)

        self.text_proj = nn.Linear(text_hidden_dim, proj_dim)
        self.time_proj = nn.Linear(time_dim, proj_dim)

        self.behavior_proj = nn.Sequential(
            nn.Linear(3, behavior_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(behavior_hidden_dim, proj_dim),
        )

        fusion_in_dim = proj_dim * 3
        self.post_fusion = nn.Sequential(
            nn.Linear(fusion_in_dim, proj_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        self.user_encoder = nn.LSTM(
            input_size=proj_dim,
            hidden_size=lstm_hidden_dim,
            num_layers=num_lstm_layers,
            batch_first=True,
            dropout=0.0,
            bidirectional=True,
        )

        self.classifier = nn.Sequential(
            nn.Linear(lstm_hidden_dim * 2, lstm_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden_dim, 1)
        )

    def masked_mean_pool(self, x, mask):
        mask = mask.unsqueeze(-1)
        x = x * mask
        denom = mask.sum(dim=1).clamp(min=1e-6)
        return x.sum(dim=1) / denom

    def forward(
        self,
        input_ids,
        attention_mask,
        token_type_ids,
        relative_days,
        hour,
        weekday,
        is_night,
        post_mask,
    ):
        B, K, L = input_ids.shape

        # ========= 文字 =========
        flat_input_ids = input_ids.view(B * K, L)
        flat_attention_mask = attention_mask.view(B * K, L)
        flat_token_type_ids = token_type_ids.view(B * K, L)

        if self.use_token_type_ids:
            text_outputs = self.text_encoder(
                input_ids=flat_input_ids,
                attention_mask=flat_attention_mask,
                token_type_ids=flat_token_type_ids
            )
        else:
            text_outputs = self.text_encoder(
                input_ids=flat_input_ids,
                attention_mask=flat_attention_mask
            )

        text_feat = text_outputs.last_hidden_state[:, 0, :]
        text_feat = self.text_proj(text_feat)
        text_feat = text_feat.view(B, K, -1)

        # ========= 時間 =========
        # 關鍵修正：先做 log 壓縮
        rel_t = torch.log1p(relative_days.clamp(min=0.0))
        time_feat = self.time2vec(rel_t)
        time_feat = self.time_proj(time_feat)

        # ========= 行為時間特徵 =========
        hour_norm = hour / 23.0
        weekday_norm = weekday / 6.0
        is_night_norm = is_night

        behavior_feat = torch.stack([hour_norm, weekday_norm, is_night_norm], dim=-1)
        behavior_feat = self.behavior_proj(behavior_feat)

        # ========= 融合 =========
        post_feat = torch.cat([text_feat, time_feat, behavior_feat], dim=-1)
        post_feat = self.post_fusion(post_feat)

        post_feat = post_feat * post_mask.unsqueeze(-1)

        # ========= user-level =========
        seq_out, _ = self.user_encoder(post_feat)
        user_feat = self.masked_mean_pool(seq_out, post_mask)

        logits = self.classifier(user_feat).squeeze(-1)
        return logits

In [101]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device =", device)

model = TextTimeBaselineModelStable(
    text_model_name="bert-base-chinese",
    text_hidden_dim=768,
    proj_dim=64,
    time_dim=8,
    behavior_hidden_dim=16,
    lstm_hidden_dim=64,
    num_lstm_layers=1,
    dropout=0.1,
    freeze_text=True,
    use_token_type_ids=True,
).to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=5e-6
)

batch = next(iter(train_loader))

with torch.no_grad():
    logits = model(
        input_ids=batch["input_ids"].to(device),
        attention_mask=batch["attention_mask"].to(device),
        token_type_ids=batch["token_type_ids"].to(device),
        relative_days=batch["relative_days"].to(device),
        hour=batch["hour"].to(device),
        weekday=batch["weekday"].to(device),
        is_night=batch["is_night"].to(device),
        post_mask=batch["post_mask"].to(device),
    )

print("logits shape:", logits.shape)
print("logits has nan:", torch.isnan(logits).any().item())
print("logits[:5]:", logits[:5])

device = cuda
logits shape: torch.Size([4])
logits has nan: False
logits[:5]: tensor([-0.0756, -0.0687, -0.0772, -0.0825], device='cuda:0')


In [102]:
EPOCHS = 1

best_val_f1 = -1
best_state = None

for epoch in range(1, EPOCHS + 1):
    print(f"\n========== Epoch {epoch}/{EPOCHS} ==========")

    train_metrics = train_one_epoch_text_time(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        epoch_idx=epoch,
        log_every=3,
        grad_clip=1.0
    )

    val_metrics = evaluate_text_time(
        model=model,
        loader=val_loader,
        criterion=criterion,
        device=device,
        epoch_idx=epoch,
        split_name="Val"
    )

    print(f"[Train][Epoch {epoch}] {train_metrics}")
    print(f"[Val][Epoch {epoch}]   {val_metrics}")

    if val_metrics["f1"] > best_val_f1:
        best_val_f1 = val_metrics["f1"]
        best_state = {k: v.cpu() for k, v in model.state_dict().items()}
        print(f"*** Best model updated at epoch {epoch}, val_f1={best_val_f1:.4f}")

if best_state is not None:
    model.load_state_dict(best_state)

test_metrics = evaluate_text_time(
    model=model,
    loader=test_loader,
    criterion=criterion,
    device=device,
    epoch_idx=0,
    split_name="Test"
)
print("\n[Test]", test_metrics)


========== Epoch 1/1 ==========


Train Epoch 1:   0%|          | 0/28 [00:00<?, ?it/s]

[Train][Epoch 1] step 3/28 | avg_loss=0.6792
[Train][Epoch 1] step 4: logits 出現 NaN，停止訓練


ValueError: logits contains NaN

### 只用文字（不加時間）baseline

In [115]:
import torch
import torch.nn as nn
from transformers import AutoModel

class TextOnlyBaselineModel(nn.Module):
    def __init__(
        self,
        text_model_name="bert-base-chinese",
        text_hidden_dim=768,
        proj_dim=64,
        lstm_hidden_dim=64,
        num_lstm_layers=1,
        dropout=0.1,
        freeze_text=True,
        use_token_type_ids=True,
    ):
        super().__init__()

        self.text_encoder = AutoModel.from_pretrained(text_model_name)
        self.use_token_type_ids = use_token_type_ids

        if freeze_text:
            for p in self.text_encoder.parameters():
                p.requires_grad = False

        self.text_proj = nn.Sequential(
            nn.Linear(text_hidden_dim, proj_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        self.user_encoder = nn.LSTM(
            input_size=proj_dim,
            hidden_size=lstm_hidden_dim,
            num_layers=num_lstm_layers,
            batch_first=True,
            dropout=0.0 if num_lstm_layers == 1 else dropout,
            bidirectional=True,
        )

        self.classifier = nn.Sequential(
            nn.Linear(lstm_hidden_dim * 2, lstm_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden_dim, 1)
        )

    def masked_mean_pool(self, x, mask):
        mask = mask.unsqueeze(-1)   # [B, K, 1]
        x = x * mask
        denom = mask.sum(dim=1).clamp(min=1e-6)
        return x.sum(dim=1) / denom

    def forward(
        self,
        input_ids,
        attention_mask,
        token_type_ids,
        post_mask,
    ):
        """
        input_ids:      [B, K, L]
        attention_mask: [B, K, L]
        token_type_ids: [B, K, L]
        post_mask:      [B, K]
        """
        B, K, L = input_ids.shape

        flat_input_ids = input_ids.view(B * K, L)
        flat_attention_mask = attention_mask.view(B * K, L)
        flat_token_type_ids = token_type_ids.view(B * K, L)
        flat_post_mask = post_mask.view(B * K)   # [B*K]

        # 只挑真實貼文進 BERT
        valid_idx = flat_post_mask > 0

        # 先建立全 0 的文字特徵容器
        device = input_ids.device
        text_feat_all = torch.zeros(B * K, 768, device=device)

        if valid_idx.sum() > 0:
            valid_input_ids = flat_input_ids[valid_idx]
            valid_attention_mask = flat_attention_mask[valid_idx]
            valid_token_type_ids = flat_token_type_ids[valid_idx]

            if self.use_token_type_ids:
                text_outputs = self.text_encoder(
                    input_ids=valid_input_ids,
                    attention_mask=valid_attention_mask,
                    token_type_ids=valid_token_type_ids
                )
            else:
                text_outputs = self.text_encoder(
                    input_ids=valid_input_ids,
                    attention_mask=valid_attention_mask
                )

            valid_text_feat = text_outputs.last_hidden_state[:, 0, :]   # [N_valid, 768]
            text_feat_all[valid_idx] = valid_text_feat

        # projection
        text_feat = self.text_proj(text_feat_all)   # [B*K, proj_dim]
        text_feat = text_feat.view(B, K, -1)        # [B, K, proj_dim]

        # 再保險一次：padding 位置清零
        text_feat = text_feat * post_mask.unsqueeze(-1)

        seq_out, _ = self.user_encoder(text_feat)   # [B, K, 2*lstm_hidden_dim]
        user_feat = self.masked_mean_pool(seq_out, post_mask)

        logits = self.classifier(user_feat).squeeze(-1)   # [B]
        return logits

In [116]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device =", device)

text_only_model = TextOnlyBaselineModel(
    text_model_name="bert-base-chinese",
    text_hidden_dim=768,
    proj_dim=64,
    lstm_hidden_dim=64,
    num_lstm_layers=1,
    dropout=0.1,
    freeze_text=True,
    use_token_type_ids=True,
).to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, text_only_model.parameters()),
    lr=1e-6
)

device = cuda


In [118]:
batch = next(iter(train_loader))

with torch.no_grad():
    logits = text_only_model(
        input_ids=batch["input_ids"].to(device),
        attention_mask=batch["attention_mask"].to(device),
        token_type_ids=batch["token_type_ids"].to(device),
        post_mask=batch["post_mask"].to(device),
    )

print("logits shape:", logits.shape)
print("logits has nan:", torch.isnan(logits).any().item())
print("logits[:5]:", logits[:5])

logits shape: torch.Size([4])
logits has nan: False
logits[:5]: tensor([0.0287, 0.0399, 0.0082, 0.0296], device='cuda:0')


In [119]:
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
import numpy as np
import torch

def train_one_epoch_text_only(
    model,
    loader,
    optimizer,
    criterion,
    device,
    epoch_idx=1,
    log_every=5,
    grad_clip=1.0
):
    model.train()
    total_loss = 0.0

    all_labels = []
    all_probs = []
    all_preds = []

    pbar = tqdm(loader, desc=f"Train Epoch {epoch_idx}", leave=True)

    for step, batch in enumerate(pbar, start=1):
        labels = batch["label"].float().to(device)

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)
        post_mask = batch["post_mask"].to(device)

        optimizer.zero_grad()

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            post_mask=post_mask,
        )

        if torch.isnan(logits).any():
            print(f"[Train][Epoch {epoch_idx}] step {step}: logits 出現 NaN，停止訓練")
            raise ValueError("logits contains NaN")

        loss = criterion(logits, labels)

        if torch.isnan(loss):
            print(f"[Train][Epoch {epoch_idx}] step {step}: loss 出現 NaN，停止訓練")
            raise ValueError("loss is NaN")

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
        optimizer.step()

        total_loss += loss.item()

        probs = torch.sigmoid(logits).detach().cpu().numpy()
        preds = (probs >= 0.5).astype(int)

        all_labels.extend(labels.cpu().numpy().astype(int).tolist())
        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())

        avg_loss_so_far = total_loss / step
        pbar.set_postfix(loss=f"{avg_loss_so_far:.4f}")

        if step % log_every == 0:
            print(f"[Train][Epoch {epoch_idx}] step {step}/{len(loader)} | avg_loss={avg_loss_so_far:.4f}")

    avg_loss = total_loss / len(loader)

    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="binary", zero_division=0
    )

    try:
        auc = roc_auc_score(all_labels, all_probs)
    except:
        auc = np.nan

    return {
        "loss": avg_loss,
        "acc": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc
    }


@torch.no_grad()
def evaluate_text_only(model, loader, criterion, device, epoch_idx=1, split_name="Val"):
    model.eval()
    total_loss = 0.0

    all_labels = []
    all_probs = []
    all_preds = []

    pbar = tqdm(loader, desc=f"{split_name} Epoch {epoch_idx}", leave=True)

    for step, batch in enumerate(pbar, start=1):
        labels = batch["label"].float().to(device)

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)
        post_mask = batch["post_mask"].to(device)

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            post_mask=post_mask,
        )

        if torch.isnan(logits).any():
            print(f"[{split_name}][Epoch {epoch_idx}] step {step}: logits 出現 NaN")
            raise ValueError(f"{split_name} logits contains NaN")

        loss = criterion(logits, labels)

        if torch.isnan(loss):
            print(f"[{split_name}][Epoch {epoch_idx}] step {step}: loss 出現 NaN")
            raise ValueError(f"{split_name} loss is NaN")

        total_loss += loss.item()

        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs >= 0.5).astype(int)

        all_labels.extend(labels.cpu().numpy().astype(int).tolist())
        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())

        avg_loss_so_far = total_loss / step
        pbar.set_postfix(loss=f"{avg_loss_so_far:.4f}")

    avg_loss = total_loss / len(loader)

    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="binary", zero_division=0
    )

    try:
        auc = roc_auc_score(all_labels, all_probs)
    except:
        auc = np.nan

    return {
        "loss": avg_loss,
        "acc": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc
    }

In [123]:
EPOCHS = 5

best_val_f1 = -1
best_state = None

for epoch in range(1, EPOCHS + 1):
    print(f"\n========== Epoch {epoch}/{EPOCHS} ==========")

    train_metrics = train_one_epoch_text_only(
        model=text_only_model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        epoch_idx=epoch,
        log_every=5,
        grad_clip=1.0
    )

    val_metrics = evaluate_text_only(
        model=text_only_model,
        loader=val_loader,
        criterion=criterion,
        device=device,
        epoch_idx=epoch,
        split_name="Val"
    )

    print(f"[Train][Epoch {epoch}] {train_metrics}")
    print(f"[Val][Epoch {epoch}]   {val_metrics}")

    if val_metrics["f1"] > best_val_f1:
        best_val_f1 = val_metrics["f1"]
        best_state = {k: v.cpu() for k, v in text_only_model.state_dict().items()}
        print(f"*** Best model updated at epoch {epoch}, val_f1={best_val_f1:.4f}")

if best_state is not None:
    text_only_model.load_state_dict(best_state)

test_metrics = evaluate_text_only(
    model=text_only_model,
    loader=test_loader,
    criterion=criterion,
    device=device,
    epoch_idx=0,
    split_name="Test"
)
print("\n[Test]", test_metrics)


========== Epoch 1/5 ==========


Train Epoch 1:   0%|          | 0/28 [00:00<?, ?it/s]

[Train][Epoch 1] step 5/28 | avg_loss=0.6981
[Train][Epoch 1] step 10/28 | avg_loss=0.6990
[Train][Epoch 1] step 15/28 | avg_loss=0.6976
[Train][Epoch 1] step 20/28 | avg_loss=0.6974
[Train][Epoch 1] step 25/28 | avg_loss=0.6975


Val Epoch 1:   0%|          | 0/6 [00:00<?, ?it/s]

[Train][Epoch 1] {'loss': 0.697544527905328, 'acc': 0.3482142857142857, 'precision': 0.3482142857142857, 'recall': 1.0, 'f1': 0.5165562913907285, 'auc': 0.5240604144713734}
[Val][Epoch 1]   {'loss': 0.7014070252577463, 'acc': 0.20833333333333334, 'precision': 0.20833333333333334, 'recall': 1.0, 'f1': 0.3448275862068966, 'auc': 0.536842105263158}
*** Best model updated at epoch 1, val_f1=0.3448

========== Epoch 2/5 ==========


...

[Train][Epoch 5] step 5/28 | avg_loss=0.6970
[Train][Epoch 5] step 10/28 | avg_loss=0.6983
[Train][Epoch 5] step 15/28 | avg_loss=0.6974
[Train][Epoch 5] step 20/28 | avg_loss=0.6974
[Train][Epoch 5] step 25/28 | avg_loss=0.6975


Val Epoch 5:   0%|          | 0/6 [00:00<?, ?it/s]

[Train][Epoch 5] {'loss': 0.6977679474013192, 'acc': 0.3482142857142857, 'precision': 0.3482142857142857, 'recall': 1.0, 'f1': 0.5165562913907285, 'auc': 0.44783983140147526}
[Val][Epoch 5]   {'loss': 0.700010230143865, 'acc': 0.20833333333333334, 'precision': 0.20833333333333334, 'recall': 1.0, 'f1': 0.3448275862068966, 'auc': 0.536842105263158}


Test Epoch 0:   0%|          | 0/7 [00:00<?, ?it/s]


[Test] {'loss': 0.7017368078231812, 'acc': 0.2, 'precision': 0.2, 'recall': 1.0, 'f1': 0.33333333333333337, 'auc': 0.46}


In [121]:
import numpy as np

def check_loader_label_distribution(loader, name="loader"):
    all_labels = []
    for batch in loader:
        all_labels.extend(batch["label"].numpy().tolist())
    all_labels = np.array(all_labels)

    print(f"{name}:")
    print(f"  total = {len(all_labels)}")
    print(f"  pos   = {int(all_labels.sum())}")
    print(f"  neg   = {int((all_labels == 0).sum())}")
    print(f"  pos_rate = {all_labels.mean():.4f}")
    print()

check_loader_label_distribution(train_loader, "train")
check_loader_label_distribution(val_loader, "val")
check_loader_label_distribution(test_loader, "test")

train:
  total = 112
  pos   = 39
  neg   = 73
  pos_rate = 0.3482

val:
  total = 24
  pos   = 5
  neg   = 19
  pos_rate = 0.2083

test:
  total = 25
  pos   = 5
  neg   = 20
  pos_rate = 0.2000



In [122]:
@torch.no_grad()
def collect_probs(model, loader, device):
    model.eval()
    all_probs = []
    all_labels = []

    for batch in loader:
        logits = model(
            input_ids=batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device),
            token_type_ids=batch["token_type_ids"].to(device),
            post_mask=batch["post_mask"].to(device),
        )
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.extend(probs.tolist())
        all_labels.extend(batch["label"].numpy().tolist())

    return np.array(all_probs), np.array(all_labels)

test_probs, test_labels = collect_probs(text_only_model, test_loader, device)

print("min prob:", test_probs.min())
print("max prob:", test_probs.max())
print("mean prob:", test_probs.mean())
print("first 20 probs:", test_probs[:20])

min prob: 0.5025764107704163
max prob: 0.5093798637390137
mean prob: 0.5067679119110108
first 20 probs: [0.50937986 0.50782824 0.50621152 0.50587171 0.50686872 0.50614023
 0.50856465 0.50564134 0.50566858 0.50629264 0.50753647 0.50856531
 0.50769424 0.50594056 0.50257641 0.5052883  0.5067144  0.50746375
 0.50647378 0.50607353]


### 文字 pooled baseline 的完整模型 + train/eval 程式碼

In [147]:
import torch
import torch.nn as nn
from transformers import AutoModel

class TextPooledBaselineModel(nn.Module):
    def __init__(
        self,
        text_model_name="bert-base-chinese",
        text_hidden_dim=768,
        proj_dim=128,
        hidden_dim=64,
        dropout=0.2,
        freeze_text=True,
        use_token_type_ids=True,
    ):
        super().__init__()

        self.text_encoder = AutoModel.from_pretrained(text_model_name)
        self.use_token_type_ids = use_token_type_ids

        if freeze_text:
            for p in self.text_encoder.parameters():
                p.requires_grad = False

        self.text_proj = nn.Sequential(
            nn.Linear(text_hidden_dim, proj_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        self.classifier = nn.Sequential(
            nn.Linear(proj_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def masked_mean_pool(self, x, mask):
        """
        x: [B, K, D]
        mask: [B, K]
        """
        mask = mask.unsqueeze(-1)   # [B, K, 1]
        x = x * mask
        denom = mask.sum(dim=1).clamp(min=1e-6)
        return x.sum(dim=1) / denom

    def forward(
        self,
        input_ids,
        attention_mask,
        token_type_ids,
        post_mask,
    ):
        """
        input_ids:      [B, K, L]
        attention_mask: [B, K, L]
        token_type_ids: [B, K, L]
        post_mask:      [B, K]
        """
        B, K, L = input_ids.shape

        flat_input_ids = input_ids.view(B * K, L)
        flat_attention_mask = attention_mask.view(B * K, L)
        flat_token_type_ids = token_type_ids.view(B * K, L)
        flat_post_mask = post_mask.view(B * K)   # [B*K]

        valid_idx = flat_post_mask > 0

        device = input_ids.device
        text_feat_all = torch.zeros(B * K, 768, device=device)

        # 只對真實貼文跑 BERT，避免 padding post 進 BERT 造成 NaN
        if valid_idx.sum() > 0:
            valid_input_ids = flat_input_ids[valid_idx]
            valid_attention_mask = flat_attention_mask[valid_idx]
            valid_token_type_ids = flat_token_type_ids[valid_idx]

            if self.use_token_type_ids:
                text_outputs = self.text_encoder(
                    input_ids=valid_input_ids,
                    attention_mask=valid_attention_mask,
                    token_type_ids=valid_token_type_ids
                )
            else:
                text_outputs = self.text_encoder(
                    input_ids=valid_input_ids,
                    attention_mask=valid_attention_mask
                )

            valid_text_feat = text_outputs.last_hidden_state[:, 0, :]   # [N_valid, 768]
            text_feat_all[valid_idx] = valid_text_feat

        text_feat = self.text_proj(text_feat_all)   # [B*K, proj_dim]
        text_feat = text_feat.view(B, K, -1)        # [B, K, proj_dim]

        # user-level pooling
        user_feat = self.masked_mean_pool(text_feat, post_mask)   # [B, proj_dim]

        logits = self.classifier(user_feat).squeeze(-1)           # [B]
        return logits

In [148]:
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
import numpy as np
import torch

def train_one_epoch_text_pooled(
    model,
    loader,
    optimizer,
    criterion,
    device,
    epoch_idx=1,
    log_every=5,
    grad_clip=1.0
):
    model.train()
    total_loss = 0.0

    all_labels = []
    all_probs = []
    all_preds = []

    pbar = tqdm(loader, desc=f"Train Epoch {epoch_idx}", leave=True)

    for step, batch in enumerate(pbar, start=1):
        labels = batch["label"].float().to(device)

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)
        post_mask = batch["post_mask"].to(device)

        optimizer.zero_grad()

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            post_mask=post_mask,
        )

        if torch.isnan(logits).any():
            print(f"[Train][Epoch {epoch_idx}] step {step}: logits 出現 NaN")
            raise ValueError("logits contains NaN")

        loss = criterion(logits, labels)

        if torch.isnan(loss):
            print(f"[Train][Epoch {epoch_idx}] step {step}: loss 出現 NaN")
            raise ValueError("loss is NaN")

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
        optimizer.step()

        total_loss += loss.item()

        probs = torch.sigmoid(logits).detach().cpu().numpy()
        preds = (probs >= 0.5).astype(int)

        all_labels.extend(labels.cpu().numpy().astype(int).tolist())
        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())

        avg_loss_so_far = total_loss / step
        pbar.set_postfix(loss=f"{avg_loss_so_far:.4f}")

        if step % log_every == 0:
            print(f"[Train][Epoch {epoch_idx}] step {step}/{len(loader)} | avg_loss={avg_loss_so_far:.4f}")

    avg_loss = total_loss / len(loader)

    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="binary", zero_division=0
    )

    try:
        auc = roc_auc_score(all_labels, all_probs)
    except:
        auc = np.nan

    return {
        "loss": avg_loss,
        "acc": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc
    }


@torch.no_grad()
def evaluate_text_pooled(
    model,
    loader,
    criterion,
    device,
    epoch_idx=1,
    split_name="Val"
):
    model.eval()
    total_loss = 0.0

    all_labels = []
    all_probs = []
    all_preds = []

    pbar = tqdm(loader, desc=f"{split_name} Epoch {epoch_idx}", leave=True)

    for step, batch in enumerate(pbar, start=1):
        labels = batch["label"].float().to(device)

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)
        post_mask = batch["post_mask"].to(device)

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            post_mask=post_mask,
        )

        if torch.isnan(logits).any():
            print(f"[{split_name}][Epoch {epoch_idx}] step {step}: logits 出現 NaN")
            raise ValueError(f"{split_name} logits contains NaN")

        loss = criterion(logits, labels)

        if torch.isnan(loss):
            print(f"[{split_name}][Epoch {epoch_idx}] step {step}: loss 出現 NaN")
            raise ValueError(f"{split_name} loss is NaN")

        total_loss += loss.item()

        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs >= 0.5).astype(int)

        all_labels.extend(labels.cpu().numpy().astype(int).tolist())
        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())

        avg_loss_so_far = total_loss / step
        pbar.set_postfix(loss=f"{avg_loss_so_far:.4f}")

    avg_loss = total_loss / len(loader)

    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="binary", zero_division=0
    )

    try:
        auc = roc_auc_score(all_labels, all_probs)
    except:
        auc = np.nan

    return {
        "loss": avg_loss,
        "acc": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc
    }

In [149]:
import numpy as np
import torch

def compute_pos_weight_from_loader(loader):
    labels = []
    for batch in loader:
        labels.extend(batch["label"].numpy().tolist())
    labels = np.array(labels)

    pos = labels.sum()
    neg = (labels == 0).sum()

    print(f"Train labels -> pos={int(pos)}, neg={int(neg)}")

    if pos == 0:
        return torch.tensor(1.0)

    pos_weight = neg / pos
    return torch.tensor(float(pos_weight), dtype=torch.float32)

pos_weight = compute_pos_weight_from_loader(train_loader)
print("pos_weight =", pos_weight.item())

Train labels -> pos=39, neg=73
pos_weight = 1.8717948198318481


In [127]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device =", device)

text_pooled_model = TextPooledBaselineModel(
    text_model_name="bert-base-chinese",
    text_hidden_dim=768,
    proj_dim=128,
    hidden_dim=64,
    dropout=0.2,
    freeze_text=True,
    use_token_type_ids=True,
).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, text_pooled_model.parameters()),
    lr=1e-4
)

batch = next(iter(train_loader))

with torch.no_grad():
    logits = text_pooled_model(
        input_ids=batch["input_ids"].to(device),
        attention_mask=batch["attention_mask"].to(device),
        token_type_ids=batch["token_type_ids"].to(device),
        post_mask=batch["post_mask"].to(device),
    )

print("logits shape:", logits.shape)
print("logits has nan:", torch.isnan(logits).any().item())
print("logits[:5]:", logits[:5])

device = cuda
logits shape: torch.Size([4])
logits has nan: False
logits[:5]: tensor([ 0.0793, -0.0253, -0.0642, -0.1959], device='cuda:0')


In [128]:
EPOCHS = 5

best_val_f1 = -1
best_state = None

for epoch in range(1, EPOCHS + 1):
    print(f"\n========== Epoch {epoch}/{EPOCHS} ==========")

    train_metrics = train_one_epoch_text_pooled(
        model=text_pooled_model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        epoch_idx=epoch,
        log_every=5,
        grad_clip=1.0
    )

    val_metrics = evaluate_text_pooled(
        model=text_pooled_model,
        loader=val_loader,
        criterion=criterion,
        device=device,
        epoch_idx=epoch,
        split_name="Val"
    )

    print(f"[Train][Epoch {epoch}] {train_metrics}")
    print(f"[Val][Epoch {epoch}]   {val_metrics}")

    if val_metrics["f1"] > best_val_f1:
        best_val_f1 = val_metrics["f1"]
        best_state = {k: v.cpu() for k, v in text_pooled_model.state_dict().items()}
        print(f"*** Best model updated at epoch {epoch}, val_f1={best_val_f1:.4f}")

if best_state is not None:
    text_pooled_model.load_state_dict(best_state)

test_metrics = evaluate_text_pooled(
    model=text_pooled_model,
    loader=test_loader,
    criterion=criterion,
    device=device,
    epoch_idx=0,
    split_name="Test"
)
print("\n[Test]", test_metrics)


========== Epoch 1/5 ==========


Train Epoch 1:   0%|          | 0/28 [00:00<?, ?it/s]

[Train][Epoch 1] step 5/28 | avg_loss=0.9020
[Train][Epoch 1] step 10/28 | avg_loss=0.8602
[Train][Epoch 1] step 15/28 | avg_loss=0.8950
[Train][Epoch 1] step 20/28 | avg_loss=0.8939
[Train][Epoch 1] step 25/28 | avg_loss=0.8813


Val Epoch 1:   0%|          | 0/6 [00:00<?, ?it/s]

[Train][Epoch 1] {'loss': 0.8770023499216352, 'acc': 0.7232142857142857, 'precision': 0.75, 'recall': 0.3076923076923077, 'f1': 0.4363636363636364, 'auc': 0.7892518440463646}
[Val][Epoch 1]   {'loss': 0.7588011423746744, 'acc': 0.7916666666666666, 'precision': 0.5, 'recall': 0.4, 'f1': 0.4444444444444445, 'auc': 0.8842105263157894}
*** Best model updated at epoch 1, val_f1=0.4444

========== Epoch 2/5 ==========


...

[Train][Epoch 5] step 5/28 | avg_loss=0.5754
[Train][Epoch 5] step 10/28 | avg_loss=0.5692
[Train][Epoch 5] step 15/28 | avg_loss=0.6022
[Train][Epoch 5] step 20/28 | avg_loss=0.5919
[Train][Epoch 5] step 25/28 | avg_loss=0.5926


Val Epoch 5:   0%|          | 0/6 [00:00<?, ?it/s]

[Train][Epoch 5] {'loss': 0.5881928567375455, 'acc': 0.8660714285714286, 'precision': 0.7608695652173914, 'recall': 0.8974358974358975, 'f1': 0.8235294117647058, 'auc': 0.9325605900948366}
[Val][Epoch 5]   {'loss': 0.5222026854753494, 'acc': 0.7916666666666666, 'precision': 0.5, 'recall': 0.6, 'f1': 0.5454545454545454, 'auc': 0.9263157894736842}


Test Epoch 0:   0%|          | 0/7 [00:00<?, ?it/s]


[Test] {'loss': 0.7177636282784599, 'acc': 0.76, 'precision': 0.4444444444444444, 'recall': 0.8, 'f1': 0.5714285714285714, 'auc': 0.87}


In [129]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np
import pandas as pd
import torch

@torch.no_grad()
def collect_probs_and_labels(model, loader, device):
    model.eval()
    probs_all = []
    labels_all = []

    for batch in loader:
        logits = model(
            input_ids=batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device),
            token_type_ids=batch["token_type_ids"].to(device),
            post_mask=batch["post_mask"].to(device),
        )
        probs = torch.sigmoid(logits).cpu().numpy()
        labels = batch["label"].numpy()

        probs_all.extend(probs.tolist())
        labels_all.extend(labels.tolist())

    return np.array(probs_all), np.array(labels_all)

val_probs, val_labels = collect_probs_and_labels(text_pooled_model, val_loader, device)

rows = []
for th in [0.3, 0.4, 0.5, 0.6, 0.7]:
    preds = (val_probs >= th).astype(int)
    acc = accuracy_score(val_labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        val_labels, preds, average="binary", zero_division=0
    )
    rows.append({
        "threshold": th,
        "acc": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

threshold_df = pd.DataFrame(rows)
print(threshold_df)

   threshold       acc  precision  recall        f1
0        0.3  0.208333   0.208333     1.0  0.344828
1        0.4  0.458333   0.277778     1.0  0.434783
2        0.5  0.833333   0.600000     0.6  0.600000
3        0.6  0.791667   0.000000     0.0  0.000000
4        0.7  0.791667   0.000000     0.0  0.000000


In [130]:
import torch
import torch.nn as nn
from transformers import AutoModel


class Time2Vec(nn.Module):
    def __init__(self, out_dim):
        super().__init__()
        assert out_dim >= 1
        self.out_dim = out_dim

        self.w0 = nn.Parameter(torch.randn(1) * 0.01)
        self.b0 = nn.Parameter(torch.zeros(1))

        if out_dim > 1:
            self.w = nn.Parameter(torch.randn(out_dim - 1) * 0.01)
            self.b = nn.Parameter(torch.zeros(out_dim - 1))

    def forward(self, t):
        """
        t: [B, K]
        output: [B, K, out_dim]
        """
        t = t.unsqueeze(-1)  # [B, K, 1]

        v0 = self.w0 * t + self.b0
        if self.out_dim == 1:
            return v0

        vp = torch.sin(t * self.w + self.b)
        return torch.cat([v0, vp], dim=-1)


class TextTimePooledBaselineModel(nn.Module):
    def __init__(
        self,
        text_model_name="bert-base-chinese",
        text_hidden_dim=768,
        text_proj_dim=128,
        time_dim=16,
        time_proj_dim=64,
        behavior_hidden_dim=32,
        fusion_hidden_dim=128,
        classifier_hidden_dim=64,
        dropout=0.2,
        freeze_text=True,
        use_token_type_ids=True,
    ):
        super().__init__()

        self.text_encoder = AutoModel.from_pretrained(text_model_name)
        self.use_token_type_ids = use_token_type_ids

        if freeze_text:
            for p in self.text_encoder.parameters():
                p.requires_grad = False

        # 文字投影
        self.text_proj = nn.Sequential(
            nn.Linear(text_hidden_dim, text_proj_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        # 時間主特徵：relative_days -> time2vec
        self.time2vec = Time2Vec(out_dim=time_dim)
        self.time_proj = nn.Sequential(
            nn.Linear(time_dim, time_proj_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        # 行為時間特徵：hour, weekday, is_night
        self.behavior_proj = nn.Sequential(
            nn.Linear(3, behavior_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(behavior_hidden_dim, time_proj_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        # post-level fusion
        fusion_in_dim = text_proj_dim + time_proj_dim + time_proj_dim
        self.post_fusion = nn.Sequential(
            nn.Linear(fusion_in_dim, fusion_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        # user-level classifier
        self.classifier = nn.Sequential(
            nn.Linear(fusion_hidden_dim, classifier_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(classifier_hidden_dim, 1)
        )

    def masked_mean_pool(self, x, mask):
        """
        x: [B, K, D]
        mask: [B, K]
        """
        mask = mask.unsqueeze(-1)  # [B, K, 1]
        x = x * mask
        denom = mask.sum(dim=1).clamp(min=1e-6)
        return x.sum(dim=1) / denom

    def forward(
        self,
        input_ids,
        attention_mask,
        token_type_ids,
        relative_days,
        hour,
        weekday,
        is_night,
        post_mask,
    ):
        """
        input_ids:      [B, K, L]
        attention_mask: [B, K, L]
        token_type_ids: [B, K, L]
        relative_days:  [B, K]
        hour:           [B, K]
        weekday:        [B, K]
        is_night:       [B, K]
        post_mask:      [B, K]
        """
        B, K, L = input_ids.shape

        flat_input_ids = input_ids.view(B * K, L)
        flat_attention_mask = attention_mask.view(B * K, L)
        flat_token_type_ids = token_type_ids.view(B * K, L)
        flat_post_mask = post_mask.view(B * K)

        valid_idx = flat_post_mask > 0

        device = input_ids.device
        text_feat_all = torch.zeros(B * K, 768, device=device)

        # 只對真實貼文跑 BERT
        if valid_idx.sum() > 0:
            valid_input_ids = flat_input_ids[valid_idx]
            valid_attention_mask = flat_attention_mask[valid_idx]
            valid_token_type_ids = flat_token_type_ids[valid_idx]

            if self.use_token_type_ids:
                text_outputs = self.text_encoder(
                    input_ids=valid_input_ids,
                    attention_mask=valid_attention_mask,
                    token_type_ids=valid_token_type_ids
                )
            else:
                text_outputs = self.text_encoder(
                    input_ids=valid_input_ids,
                    attention_mask=valid_attention_mask
                )

            valid_text_feat = text_outputs.last_hidden_state[:, 0, :]  # [N_valid, 768]
            text_feat_all[valid_idx] = valid_text_feat

        # text feature
        text_feat = self.text_proj(text_feat_all)     # [B*K, text_proj_dim]
        text_feat = text_feat.view(B, K, -1)          # [B, K, text_proj_dim]

        # time feature
        # 先做 log 壓縮，避免尺度過大
        rel_t = torch.log1p(relative_days.clamp(min=0.0))
        time_feat = self.time2vec(rel_t)              # [B, K, time_dim]
        time_feat = self.time_proj(time_feat)         # [B, K, time_proj_dim]

        # behavior feature
        hour_norm = hour / 23.0
        weekday_norm = weekday / 6.0
        is_night_norm = is_night

        behavior_feat = torch.stack([hour_norm, weekday_norm, is_night_norm], dim=-1)  # [B, K, 3]
        behavior_feat = self.behavior_proj(behavior_feat)                               # [B, K, time_proj_dim]

        # post-level fusion
        post_feat = torch.cat([text_feat, time_feat, behavior_feat], dim=-1)
        post_feat = self.post_fusion(post_feat)   # [B, K, fusion_hidden_dim]

        # user-level masked mean pooling
        user_feat = self.masked_mean_pool(post_feat, post_mask)   # [B, fusion_hidden_dim]

        logits = self.classifier(user_feat).squeeze(-1)           # [B]
        return logits

In [131]:
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
import numpy as np
import torch


def train_one_epoch_text_time_pooled(
    model,
    loader,
    optimizer,
    criterion,
    device,
    epoch_idx=1,
    log_every=5,
    grad_clip=1.0
):
    model.train()
    total_loss = 0.0

    all_labels = []
    all_probs = []
    all_preds = []

    pbar = tqdm(loader, desc=f"Train Epoch {epoch_idx}", leave=True)

    for step, batch in enumerate(pbar, start=1):
        labels = batch["label"].float().to(device)

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)

        relative_days = batch["relative_days"].to(device)
        hour = batch["hour"].to(device)
        weekday = batch["weekday"].to(device)
        is_night = batch["is_night"].to(device)
        post_mask = batch["post_mask"].to(device)

        optimizer.zero_grad()

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            relative_days=relative_days,
            hour=hour,
            weekday=weekday,
            is_night=is_night,
            post_mask=post_mask,
        )

        if torch.isnan(logits).any():
            print(f"[Train][Epoch {epoch_idx}] step {step}: logits 出現 NaN")
            raise ValueError("logits contains NaN")

        loss = criterion(logits, labels)

        if torch.isnan(loss):
            print(f"[Train][Epoch {epoch_idx}] step {step}: loss 出現 NaN")
            raise ValueError("loss is NaN")

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
        optimizer.step()

        total_loss += loss.item()

        probs = torch.sigmoid(logits).detach().cpu().numpy()
        preds = (probs >= 0.5).astype(int)

        all_labels.extend(labels.cpu().numpy().astype(int).tolist())
        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())

        avg_loss_so_far = total_loss / step
        pbar.set_postfix(loss=f"{avg_loss_so_far:.4f}")

        if step % log_every == 0:
            print(f"[Train][Epoch {epoch_idx}] step {step}/{len(loader)} | avg_loss={avg_loss_so_far:.4f}")

    avg_loss = total_loss / len(loader)

    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="binary", zero_division=0
    )

    try:
        auc = roc_auc_score(all_labels, all_probs)
    except:
        auc = np.nan

    return {
        "loss": avg_loss,
        "acc": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc
    }


@torch.no_grad()
def evaluate_text_time_pooled(
    model,
    loader,
    criterion,
    device,
    epoch_idx=1,
    split_name="Val"
):
    model.eval()
    total_loss = 0.0

    all_labels = []
    all_probs = []
    all_preds = []

    pbar = tqdm(loader, desc=f"{split_name} Epoch {epoch_idx}", leave=True)

    for step, batch in enumerate(pbar, start=1):
        labels = batch["label"].float().to(device)

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)

        relative_days = batch["relative_days"].to(device)
        hour = batch["hour"].to(device)
        weekday = batch["weekday"].to(device)
        is_night = batch["is_night"].to(device)
        post_mask = batch["post_mask"].to(device)

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            relative_days=relative_days,
            hour=hour,
            weekday=weekday,
            is_night=is_night,
            post_mask=post_mask,
        )

        if torch.isnan(logits).any():
            print(f"[{split_name}][Epoch {epoch_idx}] step {step}: logits 出現 NaN")
            raise ValueError(f"{split_name} logits contains NaN")

        loss = criterion(logits, labels)

        if torch.isnan(loss):
            print(f"[{split_name}][Epoch {epoch_idx}] step {step}: loss 出現 NaN")
            raise ValueError(f"{split_name} loss is NaN")

        total_loss += loss.item()

        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs >= 0.5).astype(int)

        all_labels.extend(labels.cpu().numpy().astype(int).tolist())
        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())

        avg_loss_so_far = total_loss / step
        pbar.set_postfix(loss=f"{avg_loss_so_far:.4f}")

    avg_loss = total_loss / len(loader)

    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="binary", zero_division=0
    )

    try:
        auc = roc_auc_score(all_labels, all_probs)
    except:
        auc = np.nan

    return {
        "loss": avg_loss,
        "acc": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc
    }

In [133]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device =", device)

text_time_pooled_model = TextTimePooledBaselineModel(
    text_model_name="bert-base-chinese",
    text_hidden_dim=768,
    text_proj_dim=128,
    time_dim=16,
    time_proj_dim=64,
    behavior_hidden_dim=32,
    fusion_hidden_dim=128,
    classifier_hidden_dim=64,
    dropout=0.2,
    freeze_text=True,
    use_token_type_ids=True,
).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, text_time_pooled_model.parameters()),
    lr=1e-4
)

device = cuda


In [134]:
batch = next(iter(train_loader))

with torch.no_grad():
    logits = text_time_pooled_model(
        input_ids=batch["input_ids"].to(device),
        attention_mask=batch["attention_mask"].to(device),
        token_type_ids=batch["token_type_ids"].to(device),
        relative_days=batch["relative_days"].to(device),
        hour=batch["hour"].to(device),
        weekday=batch["weekday"].to(device),
        is_night=batch["is_night"].to(device),
        post_mask=batch["post_mask"].to(device),
    )

print("logits shape:", logits.shape)
print("logits has nan:", torch.isnan(logits).any().item())
print("logits[:5]:", logits[:5])

logits shape: torch.Size([4])
logits has nan: False
logits[:5]: tensor([-0.0833, -0.0778, -0.0896, -0.0740], device='cuda:0')


In [135]:
EPOCHS = 5

best_val_f1 = -1
best_state = None

for epoch in range(1, EPOCHS + 1):
    print(f"\n========== Epoch {epoch}/{EPOCHS} ==========")

    train_metrics = train_one_epoch_text_time_pooled(
        model=text_time_pooled_model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        epoch_idx=epoch,
        log_every=5,
        grad_clip=1.0
    )

    val_metrics = evaluate_text_time_pooled(
        model=text_time_pooled_model,
        loader=val_loader,
        criterion=criterion,
        device=device,
        epoch_idx=epoch,
        split_name="Val"
    )

    print(f"[Train][Epoch {epoch}] {train_metrics}")
    print(f"[Val][Epoch {epoch}]   {val_metrics}")

    if val_metrics["f1"] > best_val_f1:
        best_val_f1 = val_metrics["f1"]
        best_state = {k: v.cpu() for k, v in text_time_pooled_model.state_dict().items()}
        print(f"*** Best model updated at epoch {epoch}, val_f1={best_val_f1:.4f}")

if best_state is not None:
    text_time_pooled_model.load_state_dict(best_state)

test_metrics = evaluate_text_time_pooled(
    model=text_time_pooled_model,
    loader=test_loader,
    criterion=criterion,
    device=device,
    epoch_idx=0,
    split_name="Test"
)

print("\n[Test]", test_metrics)


========== Epoch 1/5 ==========


Train Epoch 1:   0%|          | 0/28 [00:00<?, ?it/s]

[Train][Epoch 1] step 5/28 | avg_loss=0.9131
[Train][Epoch 1] step 10/28 | avg_loss=0.8729
[Train][Epoch 1] step 15/28 | avg_loss=0.9080
[Train][Epoch 1] step 20/28 | avg_loss=0.9164
[Train][Epoch 1] step 25/28 | avg_loss=0.9070


Val Epoch 1:   0%|          | 0/6 [00:00<?, ?it/s]

[Train][Epoch 1] {'loss': 0.9052396246365139, 'acc': 0.6517857142857143, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'auc': 0.49560941341763265}
[Val][Epoch 1]   {'loss': 0.7987387875715891, 'acc': 0.7916666666666666, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'auc': 0.8526315789473685}
*** Best model updated at epoch 1, val_f1=0.0000

========== Epoch 2/5 ==========


...

[Train][Epoch 5] step 5/28 | avg_loss=0.7100
[Train][Epoch 5] step 10/28 | avg_loss=0.7037
[Train][Epoch 5] step 15/28 | avg_loss=0.7242
[Train][Epoch 5] step 20/28 | avg_loss=0.7065
[Train][Epoch 5] step 25/28 | avg_loss=0.6953


Val Epoch 5:   0%|          | 0/6 [00:00<?, ?it/s]

[Train][Epoch 5] {'loss': 0.6831440978816578, 'acc': 0.875, 'precision': 0.7777777777777778, 'recall': 0.8974358974358975, 'f1': 0.8333333333333333, 'auc': 0.9402880224798034}
[Val][Epoch 5]   {'loss': 0.5751874049504598, 'acc': 0.8333333333333334, 'precision': 0.5714285714285714, 'recall': 0.8, 'f1': 0.6666666666666666, 'auc': 0.9263157894736842}


Test Epoch 0:   0%|          | 0/7 [00:00<?, ?it/s]


[Test] {'loss': 0.7136639952659607, 'acc': 0.72, 'precision': 0.4166666666666667, 'recall': 1.0, 'f1': 0.5882352941176471, 'auc': 0.8799999999999999}


In [151]:
import torch
import torch.nn as nn
from transformers import AutoModel, CLIPVisionModel


class Time2Vec(nn.Module):
    def __init__(self, out_dim):
        super().__init__()
        assert out_dim >= 1
        self.out_dim = out_dim

        self.w0 = nn.Parameter(torch.randn(1) * 0.01)
        self.b0 = nn.Parameter(torch.zeros(1))

        if out_dim > 1:
            self.w = nn.Parameter(torch.randn(out_dim - 1) * 0.01)
            self.b = nn.Parameter(torch.zeros(out_dim - 1))

    def forward(self, t):
        """
        t: [B, K]
        output: [B, K, out_dim]
        """
        t = t.unsqueeze(-1)  # [B, K, 1]

        v0 = self.w0 * t + self.b0
        if self.out_dim == 1:
            return v0

        vp = torch.sin(t * self.w + self.b)
        return torch.cat([v0, vp], dim=-1)


class TextTimeImagePooledBaselineModel(nn.Module):
    def __init__(
        self,
        text_model_name="bert-base-chinese",
        vision_model_name="openai/clip-vit-base-patch32",
        text_hidden_dim=768,
        vision_hidden_dim=768,
        text_proj_dim=128,
        image_proj_dim=128,
        time_dim=16,
        time_proj_dim=64,
        behavior_hidden_dim=32,
        fusion_hidden_dim=128,
        classifier_hidden_dim=64,
        dropout=0.2,
        freeze_text=True,
        freeze_vision=True,
        use_token_type_ids=True,
    ):
        super().__init__()

        # ===== 文字 encoder =====
        self.text_encoder = AutoModel.from_pretrained(text_model_name)
        self.use_token_type_ids = use_token_type_ids

        if freeze_text:
            for p in self.text_encoder.parameters():
                p.requires_grad = False

        # ===== 圖片 encoder =====
        self.vision_encoder = CLIPVisionModel.from_pretrained(vision_model_name)

        if freeze_vision:
            for p in self.vision_encoder.parameters():
                p.requires_grad = False

        # ===== projection =====
        self.text_proj = nn.Sequential(
            nn.Linear(text_hidden_dim, text_proj_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        self.image_proj = nn.Sequential(
            nn.Linear(vision_hidden_dim, image_proj_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        # ===== 時間特徵 =====
        self.time2vec = Time2Vec(out_dim=time_dim)
        self.time_proj = nn.Sequential(
            nn.Linear(time_dim, time_proj_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        # ===== 行為時間特徵 =====
        self.behavior_proj = nn.Sequential(
            nn.Linear(3, behavior_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(behavior_hidden_dim, time_proj_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        # ===== post-level fusion =====
        fusion_in_dim = text_proj_dim + image_proj_dim + time_proj_dim + time_proj_dim
        self.post_fusion = nn.Sequential(
            nn.Linear(fusion_in_dim, fusion_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        # ===== user-level classifier =====
        self.classifier = nn.Sequential(
            nn.Linear(fusion_hidden_dim, classifier_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(classifier_hidden_dim, 1)
        )

    def masked_mean_pool(self, x, mask):
        """
        x: [B, K, D]
        mask: [B, K]
        """
        mask = mask.unsqueeze(-1)  # [B, K, 1]
        x = x * mask
        denom = mask.sum(dim=1).clamp(min=1e-6)
        return x.sum(dim=1) / denom

    def forward(
        self,
        input_ids,
        attention_mask,
        token_type_ids,
        vision_tensor,
        relative_days,
        hour,
        weekday,
        is_night,
        post_mask,
    ):
        """
        input_ids:      [B, K, L]
        attention_mask: [B, K, L]
        token_type_ids: [B, K, L]
        vision_tensor:  [B, K, 3, 224, 224]
        relative_days:  [B, K]
        hour:           [B, K]
        weekday:        [B, K]
        is_night:       [B, K]
        post_mask:      [B, K]
        """
        B, K, L = input_ids.shape
        device = input_ids.device

        flat_input_ids = input_ids.view(B * K, L)
        flat_attention_mask = attention_mask.view(B * K, L)
        flat_token_type_ids = token_type_ids.view(B * K, L)
        flat_vision_tensor = vision_tensor.view(B * K, *vision_tensor.shape[2:])  # [B*K, 3, 224, 224]
        flat_post_mask = post_mask.view(B * K)  # [B*K]

        valid_idx = flat_post_mask > 0

        # ===== 文字特徵容器 =====
        text_feat_all = torch.zeros(B * K, 768, device=device)

        # ===== 圖片特徵容器 =====
        image_feat_all = torch.zeros(B * K, 768, device=device)

        # 只對真實貼文做編碼
        if valid_idx.sum() > 0:
            valid_input_ids = flat_input_ids[valid_idx]
            valid_attention_mask = flat_attention_mask[valid_idx]
            valid_token_type_ids = flat_token_type_ids[valid_idx]
            valid_vision_tensor = flat_vision_tensor[valid_idx]

            # ---- 文字編碼 ----
            if self.use_token_type_ids:
                text_outputs = self.text_encoder(
                    input_ids=valid_input_ids,
                    attention_mask=valid_attention_mask,
                    token_type_ids=valid_token_type_ids
                )
            else:
                text_outputs = self.text_encoder(
                    input_ids=valid_input_ids,
                    attention_mask=valid_attention_mask
                )

            valid_text_feat = text_outputs.last_hidden_state[:, 0, :]  # [N_valid, 768]
            text_feat_all[valid_idx] = valid_text_feat

            # ---- 圖片編碼 ----
            vision_outputs = self.vision_encoder(pixel_values=valid_vision_tensor)
            valid_image_feat = vision_outputs.pooler_output  # [N_valid, 768]
            image_feat_all[valid_idx] = valid_image_feat

        # projection
        text_feat = self.text_proj(text_feat_all).view(B, K, -1)     # [B, K, text_proj_dim]
        image_feat = self.image_proj(image_feat_all).view(B, K, -1)  # [B, K, image_proj_dim]

        # ===== 時間特徵 =====
        rel_t = torch.log1p(relative_days.clamp(min=0.0))
        time_feat = self.time2vec(rel_t)           # [B, K, time_dim]
        time_feat = self.time_proj(time_feat)      # [B, K, time_proj_dim]

        # ===== 行為時間特徵 =====
        hour_norm = hour / 23.0
        weekday_norm = weekday / 6.0
        is_night_norm = is_night

        behavior_feat = torch.stack([hour_norm, weekday_norm, is_night_norm], dim=-1)  # [B, K, 3]
        behavior_feat = self.behavior_proj(behavior_feat)                               # [B, K, time_proj_dim]

        # ===== post-level fusion =====
        post_feat = torch.cat([text_feat, image_feat, time_feat, behavior_feat], dim=-1)
        post_feat = self.post_fusion(post_feat)  # [B, K, fusion_hidden_dim]

        # ===== user-level pooling =====
        user_feat = self.masked_mean_pool(post_feat, post_mask)  # [B, fusion_hidden_dim]

        logits = self.classifier(user_feat).squeeze(-1)  # [B]
        return logits

In [152]:
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
import numpy as np
import torch


def train_one_epoch_multimodal_pooled(
    model,
    loader,
    optimizer,
    criterion,
    device,
    epoch_idx=1,
    log_every=5,
    grad_clip=1.0
):
    model.train()
    total_loss = 0.0

    all_labels = []
    all_probs = []
    all_preds = []

    pbar = tqdm(loader, desc=f"Train Epoch {epoch_idx}", leave=True)

    for step, batch in enumerate(pbar, start=1):
        labels = batch["label"].float().to(device)

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)
        vision_tensor = batch["vision_tensor"].to(device)

        relative_days = batch["relative_days"].to(device)
        hour = batch["hour"].to(device)
        weekday = batch["weekday"].to(device)
        is_night = batch["is_night"].to(device)
        post_mask = batch["post_mask"].to(device)

        optimizer.zero_grad()

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            vision_tensor=vision_tensor,
            relative_days=relative_days,
            hour=hour,
            weekday=weekday,
            is_night=is_night,
            post_mask=post_mask,
        )

        if torch.isnan(logits).any():
            print(f"[Train][Epoch {epoch_idx}] step {step}: logits 出現 NaN")
            raise ValueError("logits contains NaN")

        loss = criterion(logits, labels)

        if torch.isnan(loss):
            print(f"[Train][Epoch {epoch_idx}] step {step}: loss 出現 NaN")
            raise ValueError("loss is NaN")

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
        optimizer.step()

        total_loss += loss.item()

        probs = torch.sigmoid(logits).detach().cpu().numpy()
        preds = (probs >= 0.5).astype(int)

        all_labels.extend(labels.cpu().numpy().astype(int).tolist())
        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())

        avg_loss_so_far = total_loss / step
        pbar.set_postfix(loss=f"{avg_loss_so_far:.4f}")

        if step % log_every == 0:
            print(f"[Train][Epoch {epoch_idx}] step {step}/{len(loader)} | avg_loss={avg_loss_so_far:.4f}")

    avg_loss = total_loss / len(loader)

    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="binary", zero_division=0
    )

    try:
        auc = roc_auc_score(all_labels, all_probs)
    except:
        auc = np.nan

    return {
        "loss": avg_loss,
        "acc": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc
    }


@torch.no_grad()
def evaluate_multimodal_pooled(
    model,
    loader,
    criterion,
    device,
    epoch_idx=1,
    split_name="Val"
):
    model.eval()
    total_loss = 0.0

    all_labels = []
    all_probs = []
    all_preds = []

    pbar = tqdm(loader, desc=f"{split_name} Epoch {epoch_idx}", leave=True)

    for step, batch in enumerate(pbar, start=1):
        labels = batch["label"].float().to(device)

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)
        vision_tensor = batch["vision_tensor"].to(device)

        relative_days = batch["relative_days"].to(device)
        hour = batch["hour"].to(device)
        weekday = batch["weekday"].to(device)
        is_night = batch["is_night"].to(device)
        post_mask = batch["post_mask"].to(device)

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            vision_tensor=vision_tensor,
            relative_days=relative_days,
            hour=hour,
            weekday=weekday,
            is_night=is_night,
            post_mask=post_mask,
        )

        if torch.isnan(logits).any():
            print(f"[{split_name}][Epoch {epoch_idx}] step {step}: logits 出現 NaN")
            raise ValueError(f"{split_name} logits contains NaN")

        loss = criterion(logits, labels)

        if torch.isnan(loss):
            print(f"[{split_name}][Epoch {epoch_idx}] step {step}: loss 出現 NaN")
            raise ValueError(f"{split_name} loss is NaN")

        total_loss += loss.item()

        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs >= 0.5).astype(int)

        all_labels.extend(labels.cpu().numpy().astype(int).tolist())
        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())

        avg_loss_so_far = total_loss / step
        pbar.set_postfix(loss=f"{avg_loss_so_far:.4f}")

    avg_loss = total_loss / len(loader)

    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="binary", zero_division=0
    )

    try:
        auc = roc_auc_score(all_labels, all_probs)
    except:
        auc = np.nan

    return {
        "loss": avg_loss,
        "acc": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc
    }

In [153]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device =", device)

multimodal_pooled_model = TextTimeImagePooledBaselineModel(
    text_model_name="bert-base-chinese",
    vision_model_name="openai/clip-vit-base-patch32",
    text_hidden_dim=768,
    vision_hidden_dim=768,
    text_proj_dim=128,
    image_proj_dim=128,
    time_dim=16,
    time_proj_dim=64,
    behavior_hidden_dim=32,
    fusion_hidden_dim=128,
    classifier_hidden_dim=64,
    dropout=0.2,
    freeze_text=True,
    freeze_vision=True,
    use_token_type_ids=True,
).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, multimodal_pooled_model.parameters()),
    lr=1e-4
)

device = cuda


In [154]:
batch = next(iter(train_loader))

with torch.no_grad():
    logits = multimodal_pooled_model(
        input_ids=batch["input_ids"].to(device),
        attention_mask=batch["attention_mask"].to(device),
        token_type_ids=batch["token_type_ids"].to(device),
        vision_tensor=batch["vision_tensor"].to(device),
        relative_days=batch["relative_days"].to(device),
        hour=batch["hour"].to(device),
        weekday=batch["weekday"].to(device),
        is_night=batch["is_night"].to(device),
        post_mask=batch["post_mask"].to(device),
    )

print("logits shape:", logits.shape)
print("logits has nan:", torch.isnan(logits).any().item())
print("logits[:5]:", logits[:5])

logits shape: torch.Size([4])
logits has nan: False
logits[:5]: tensor([-0.0247, -0.0024, -0.0585, -0.0006], device='cuda:0')


In [155]:
EPOCHS = 5

best_val_f1 = -1
best_state = None

for epoch in range(1, EPOCHS + 1):
    print(f"\n========== Epoch {epoch}/{EPOCHS} ==========")

    train_metrics = train_one_epoch_multimodal_pooled(
        model=multimodal_pooled_model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        epoch_idx=epoch,
        log_every=5,
        grad_clip=1.0
    )

    val_metrics = evaluate_multimodal_pooled(
        model=multimodal_pooled_model,
        loader=val_loader,
        criterion=criterion,
        device=device,
        epoch_idx=epoch,
        split_name="Val"
    )

    print(f"[Train][Epoch {epoch}] {train_metrics}")
    print(f"[Val][Epoch {epoch}]   {val_metrics}")

    if val_metrics["f1"] > best_val_f1:
        best_val_f1 = val_metrics["f1"]
        best_state = {k: v.cpu() for k, v in multimodal_pooled_model.state_dict().items()}
        print(f"*** Best model updated at epoch {epoch}, val_f1={best_val_f1:.4f}")

if best_state is not None:
    multimodal_pooled_model.load_state_dict(best_state)

test_metrics = evaluate_multimodal_pooled(
    model=multimodal_pooled_model,
    loader=test_loader,
    criterion=criterion,
    device=device,
    epoch_idx=0,
    split_name="Test"
)

print("\n[Test]", test_metrics)


========== Epoch 1/5 ==========


Train Epoch 1:   0%|          | 0/28 [00:00<?, ?it/s]

[Train][Epoch 1] step 5/28 | avg_loss=0.9086
[Train][Epoch 1] step 10/28 | avg_loss=0.8719
[Train][Epoch 1] step 15/28 | avg_loss=0.9038
[Train][Epoch 1] step 20/28 | avg_loss=0.9106
[Train][Epoch 1] step 25/28 | avg_loss=0.9011


Val Epoch 1:   0%|          | 0/6 [00:00<?, ?it/s]

[Train][Epoch 1] {'loss': 0.8982763098818916, 'acc': 0.6428571428571429, 'precision': 0.48148148148148145, 'recall': 0.3333333333333333, 'f1': 0.3939393939393939, 'auc': 0.6315419740077274}
[Val][Epoch 1]   {'loss': 0.8001432319482168, 'acc': 0.75, 'precision': 0.45454545454545453, 'recall': 1.0, 'f1': 0.625, 'auc': 0.9368421052631579}
*** Best model updated at epoch 1, val_f1=0.6250

========== Epoch 2/5 ==========


...

[Train][Epoch 5] step 5/28 | avg_loss=0.6610
[Train][Epoch 5] step 10/28 | avg_loss=0.6614
[Train][Epoch 5] step 15/28 | avg_loss=0.6703
[Train][Epoch 5] step 20/28 | avg_loss=0.6388
[Train][Epoch 5] step 25/28 | avg_loss=0.6323


Val Epoch 5:   0%|          | 0/6 [00:00<?, ?it/s]

[Train][Epoch 5] {'loss': 0.6246179959603718, 'acc': 0.8035714285714286, 'precision': 0.6491228070175439, 'recall': 0.9487179487179487, 'f1': 0.7708333333333334, 'auc': 0.9297506146821215}
[Val][Epoch 5]   {'loss': 0.4974353015422821, 'acc': 0.8333333333333334, 'precision': 0.5555555555555556, 'recall': 1.0, 'f1': 0.7142857142857143, 'auc': 0.9578947368421052}


Test Epoch 0:   0%|          | 0/7 [00:00<?, ?it/s]


[Test] {'loss': 0.7435184035982404, 'acc': 0.52, 'precision': 0.29411764705882354, 'recall': 1.0, 'f1': 0.45454545454545453, 'auc': 0.8999999999999999}


# 4/15


### 論文實作板

In [8]:
# 1. CLIP tokenizer 前處理
from transformers import AutoTokenizer

clip_tokenizer = AutoTokenizer.from_pretrained("openai/clip-vit-base-patch32")

def tokenize_clip_texts(text_list, max_length=77):
    enc = clip_tokenizer(
        text_list,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_attention_mask=True,
        return_tensors="pt"
    )
    return enc["input_ids"], enc["attention_mask"]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

c:\Users\Angle\Anaconda3\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Angle\.cache\huggingface\hub\models--openai--clip-vit-base-patch32. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

c:\Users\Angle\Anaconda3\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [9]:
# 2. 模型：ContextVecNet 對齊版
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import CLIPModel


class Time2Vec(nn.Module):
    def __init__(self, out_dim):
        super().__init__()
        assert out_dim >= 1
        self.out_dim = out_dim

        self.w0 = nn.Parameter(torch.randn(1) * 0.01)
        self.b0 = nn.Parameter(torch.zeros(1))

        if out_dim > 1:
            self.w = nn.Parameter(torch.randn(out_dim - 1) * 0.01)
            self.b = nn.Parameter(torch.zeros(out_dim - 1))

    def forward(self, tau):
        """
        tau: [B, K]
        output: [B, K, out_dim]
        """
        tau = tau.unsqueeze(-1)  # [B, K, 1]
        v0 = self.w0 * tau + self.b0

        if self.out_dim == 1:
            return v0

        vp = torch.sin(tau * self.w + self.b)
        return torch.cat([v0, vp], dim=-1)


class CrossModalBlock(nn.Module):
    """
    LXMERT-style block:
    1) language attends to vision
    2) vision attends to language
    3) self-attn on each stream
    4) FFN on each stream
    """
    def __init__(self, d_model=128, nhead=8, ff_dim=256, dropout=0.1):
        super().__init__()

        self.lang_to_vis = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=nhead, dropout=dropout, batch_first=True
        )
        self.vis_to_lang = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=nhead, dropout=dropout, batch_first=True
        )

        self.lang_self = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=nhead, dropout=dropout, batch_first=True
        )
        self.vis_self = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=nhead, dropout=dropout, batch_first=True
        )

        self.lang_ffn = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, d_model),
        )
        self.vis_ffn = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, d_model),
        )

        self.norm_l1 = nn.LayerNorm(d_model)
        self.norm_l2 = nn.LayerNorm(d_model)
        self.norm_l3 = nn.LayerNorm(d_model)

        self.norm_v1 = nn.LayerNorm(d_model)
        self.norm_v2 = nn.LayerNorm(d_model)
        self.norm_v3 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, L, V, lang_key_padding_mask=None, vis_key_padding_mask=None):
        """
        L: [B, K, d]
        V: [B, K, d]
        key_padding_mask: [B, K], True 表示 padding / ignore
        """
        # 1) cross-attention
        L_cross, _ = self.lang_to_vis(
            query=L, key=V, value=V, key_padding_mask=vis_key_padding_mask
        )
        L = self.norm_l1(L + self.dropout(L_cross))

        V_cross, _ = self.vis_to_lang(
            query=V, key=L, value=L, key_padding_mask=lang_key_padding_mask
        )
        V = self.norm_v1(V + self.dropout(V_cross))

        # 2) self-attention
        L_self, _ = self.lang_self(
            query=L, key=L, value=L, key_padding_mask=lang_key_padding_mask
        )
        L = self.norm_l2(L + self.dropout(L_self))

        V_self, _ = self.vis_self(
            query=V, key=V, value=V, key_padding_mask=vis_key_padding_mask
        )
        V = self.norm_v2(V + self.dropout(V_self))

        # 3) FFN
        L_ffn = self.lang_ffn(L)
        L = self.norm_l3(L + self.dropout(L_ffn))

        V_ffn = self.vis_ffn(V)
        V = self.norm_v3(V + self.dropout(V_ffn))

        return L, V


class ContextVecNetApprox(nn.Module):
    """
    論文對齊版（可直接落地）：
    - frozen CLIP text/image encoders
    - learnable context vectors + coupled visual context
    - time2vec positional embeddings
    - 4 x cross-modal blocks
    - 2 x transformer encoder
    - mean pooling + classifier

    與論文完全不同的唯一點：
    - learnable prompts 沒有 deep-insert 到 CLIP 每一層，而是在 CLIP post embedding 後做 coupled context adaptation
    """
    def __init__(
        self,
        clip_model_name="openai/clip-vit-base-patch32",
        clip_embed_dim=512,          # CLIP text/image projected feature dim
        d_model=128,                 # 論文 cross-modal / transformer 使用 128
        nhead=8,
        ff_dim=256,
        num_cross_layers=4,
        num_transformer_layers=2,
        time_dim=16,
        context_len=2,
        dropout=0.1,
        freeze_clip=True,
    ):
        super().__init__()

        self.clip = CLIPModel.from_pretrained(clip_model_name)

        if freeze_clip:
            for p in self.clip.parameters():
                p.requires_grad = False

        # ===== learnable context vectors =====
        # 語言 prompt/context
        self.lang_ctx = nn.Parameter(torch.randn(context_len, clip_embed_dim) * 0.02)

        # vision-language prompt coupling
        self.ctx_coupler = nn.Linear(clip_embed_dim, clip_embed_dim)

        # ===== projection to common space =====
        self.text_proj = nn.Linear(clip_embed_dim, d_model)
        self.image_proj = nn.Linear(clip_embed_dim, d_model)

        # ===== time2vec =====
        self.time2vec = Time2Vec(out_dim=time_dim)
        self.time_proj = nn.Linear(time_dim, d_model)

        # ===== cross-modal fusion =====
        self.cross_layers = nn.ModuleList([
            CrossModalBlock(d_model=d_model, nhead=nhead, ff_dim=ff_dim, dropout=dropout)
            for _ in range(num_cross_layers)
        ])

        # ===== transformer encoder =====
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=ff_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=False
        )
        self.sequence_encoder = nn.TransformerEncoder(
            encoder_layer=enc_layer,
            num_layers=num_transformer_layers
        )

        # ===== classifier =====
        self.classifier = nn.Linear(d_model, 1)

        self.dropout = nn.Dropout(dropout)

    def masked_mean_pool(self, x, mask):
        """
        x: [B, K, D]
        mask: [B, K]
        """
        mask = mask.unsqueeze(-1)  # [B, K, 1]
        x = x * mask
        denom = mask.sum(dim=1).clamp(min=1e-6)
        return x.sum(dim=1) / denom

    def forward(
        self,
        input_ids,
        attention_mask,
        vision_tensor,
        relative_days,
        post_mask,
        image_valid_mask=None,
    ):
        """
        input_ids:        [B, K, L]  (CLIP tokenizer)
        attention_mask:   [B, K, L]
        vision_tensor:    [B, K, 3, 224, 224]
        relative_days:    [B, K]
        post_mask:        [B, K]    真實貼文=1, padding=0
        image_valid_mask: [B, K]    真實圖像=1, 黑圖placeholder=0, 可選
        """
        B, K, L = input_ids.shape
        device = input_ids.device

        if image_valid_mask is None:
            image_valid_mask = post_mask

        flat_input_ids = input_ids.view(B * K, L)
        flat_attention_mask = attention_mask.view(B * K, L)
        flat_vision = vision_tensor.view(B * K, *vision_tensor.shape[2:])  # [B*K, 3, 224, 224]

        flat_post_mask = post_mask.view(B * K) > 0
        flat_img_mask = image_valid_mask.view(B * K) > 0

        # ===== CLIP features containers =====
        text_feat_all = torch.zeros(B * K, self.clip.text_projection.out_features, device=device)
        image_feat_all = torch.zeros(B * K, self.clip.visual_projection.out_features, device=device)

        # ----- text features -----
        if flat_post_mask.sum() > 0:
            valid_input_ids = flat_input_ids[flat_post_mask]
            valid_attention_mask = flat_attention_mask[flat_post_mask]

            valid_text_feat = self.clip.get_text_features(
                input_ids=valid_input_ids,
                attention_mask=valid_attention_mask
            )  # [N_valid, 512]

            text_feat_all[flat_post_mask] = valid_text_feat

        # ----- image features -----
        if flat_img_mask.sum() > 0:
            valid_vision = flat_vision[flat_img_mask]

            valid_image_feat = self.clip.get_image_features(
                pixel_values=valid_vision
            )  # [N_valid_img, 512]

            image_feat_all[flat_img_mask] = valid_image_feat

        # ===== learnable context vectors =====
        # language context
        lang_ctx_summary = self.lang_ctx.mean(dim=0)                    # [512]
        # coupled visual context
        vis_ctx = self.ctx_coupler(self.lang_ctx)                       # [b, 512]
        vis_ctx_summary = vis_ctx.mean(dim=0)                           # [512]

        text_feat_all = text_feat_all + lang_ctx_summary
        image_feat_all = image_feat_all + vis_ctx_summary

        # ===== projection =====
        L_seq = self.text_proj(text_feat_all).view(B, K, -1)            # [B, K, d_model]
        V_seq = self.image_proj(image_feat_all).view(B, K, -1)          # [B, K, d_model]

        # ===== time2vec positional embedding =====
        # 論文用 g(tau)=1/(tau+1) 緩解大 timestamp；你目前使用者內相對時間更穩定，先做 log1p 再進 time2vec
        rel_t = torch.log1p(relative_days.clamp(min=0.0))
        P_time = self.time_proj(self.time2vec(rel_t))                   # [B, K, d_model]

        L_seq = L_seq + P_time
        V_seq = V_seq + P_time

        # masks for attention
        lang_kpm = (post_mask == 0)          # True means ignore
        vis_kpm = (image_valid_mask == 0)    # 黑圖 placeholder 可忽略

        # ===== cross-modal encoder =====
        for layer in self.cross_layers:
            L_seq, V_seq = layer(
                L_seq, V_seq,
                lang_key_padding_mask=lang_kpm,
                vis_key_padding_mask=vis_kpm
            )

        # ===== transformer encoder on language branch =====
        Z = self.sequence_encoder(
            L_seq,
            src_key_padding_mask=lang_kpm
        )  # [B, K, d_model]

        # ===== mean pooling =====
        z_final = self.masked_mean_pool(Z, post_mask)   # [B, d_model]

        logits = self.classifier(self.dropout(z_final)).squeeze(-1)  # [B]
        return logits

In [10]:
# 3. pos_weight 計算
import numpy as np
import torch

def compute_pos_weight_from_loader(loader):
    labels = []
    for batch in loader:
        labels.extend(batch["label"].numpy().tolist())
    labels = np.array(labels)

    pos = labels.sum()
    neg = (labels == 0).sum()

    print(f"Train labels -> pos={int(pos)}, neg={int(neg)}")

    if pos == 0:
        return torch.tensor(1.0)

    pos_weight = neg / pos
    return torch.tensor(float(pos_weight), dtype=torch.float32)

In [13]:
# 4. train / eval
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
import numpy as np
import torch

def train_one_epoch_contextvecnet(
    model,
    loader,
    optimizer,
    criterion,
    device,
    scheduler=None,
    epoch_idx=1,
    log_every=5,
    grad_clip=1.0
):
    model.train()
    total_loss = 0.0

    all_labels = []
    all_probs = []
    all_preds = []

    pbar = tqdm(loader, desc=f"Train Epoch {epoch_idx}", leave=True)

    for step, batch in enumerate(pbar, start=1):
        labels = batch["label"].float().to(device)

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        vision_tensor = batch["vision_tensor"].to(device)
        relative_days = batch["relative_days"].to(device)
        post_mask = batch["post_mask"].to(device)

        image_valid_mask = batch["image_valid_mask"].to(device) if "image_valid_mask" in batch else post_mask

        optimizer.zero_grad()

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            vision_tensor=vision_tensor,
            relative_days=relative_days,
            post_mask=post_mask,
            image_valid_mask=image_valid_mask,
        )

        if torch.isnan(logits).any():
            print(f"[Train][Epoch {epoch_idx}] step {step}: logits 出現 NaN")
            raise ValueError("logits contains NaN")

        loss = criterion(logits, labels)

        if torch.isnan(loss):
            print(f"[Train][Epoch {epoch_idx}] step {step}: loss 出現 NaN")
            raise ValueError("loss is NaN")

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
        optimizer.step()

        if scheduler is not None:
            scheduler.step()

        total_loss += loss.item()

        probs = torch.sigmoid(logits).detach().cpu().numpy()
        preds = (probs >= 0.5).astype(int)

        all_labels.extend(labels.cpu().numpy().astype(int).tolist())
        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())

        avg_loss_so_far = total_loss / step
        pbar.set_postfix(loss=f"{avg_loss_so_far:.4f}")

        if step % log_every == 0:
            print(f"[Train][Epoch {epoch_idx}] step {step}/{len(loader)} | avg_loss={avg_loss_so_far:.4f}")

    avg_loss = total_loss / len(loader)

    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="binary", zero_division=0
    )

    try:
        auc = roc_auc_score(all_labels, all_probs)
    except:
        auc = np.nan

    return {
        "loss": avg_loss,
        "acc": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc
    }


@torch.no_grad()
def evaluate_contextvecnet(
    model,
    loader,
    criterion,
    device,
    epoch_idx=1,
    split_name="Val"
):
    model.eval()
    total_loss = 0.0

    all_labels = []
    all_probs = []
    all_preds = []

    pbar = tqdm(loader, desc=f"{split_name} Epoch {epoch_idx}", leave=True)

    for step, batch in enumerate(pbar, start=1):
        labels = batch["label"].float().to(device)

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        vision_tensor = batch["vision_tensor"].to(device)
        relative_days = batch["relative_days"].to(device)
        post_mask = batch["post_mask"].to(device)

        image_valid_mask = batch["image_valid_mask"].to(device) if "image_valid_mask" in batch else post_mask

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            vision_tensor=vision_tensor,
            relative_days=relative_days,
            post_mask=post_mask,
            image_valid_mask=image_valid_mask,
        )

        if torch.isnan(logits).any():
            print(f"[{split_name}][Epoch {epoch_idx}] step {step}: logits 出現 NaN")
            raise ValueError(f"{split_name} logits contains NaN")

        loss = criterion(logits, labels)

        if torch.isnan(loss):
            print(f"[{split_name}][Epoch {epoch_idx}] step {step}: loss 出現 NaN")
            raise ValueError(f"{split_name} loss is NaN")

        total_loss += loss.item()

        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs >= 0.5).astype(int)

        all_labels.extend(labels.cpu().numpy().astype(int).tolist())
        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())

        avg_loss_so_far = total_loss / step
        pbar.set_postfix(loss=f"{avg_loss_so_far:.4f}")

    avg_loss = total_loss / len(loader)

    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="binary", zero_division=0
    )

    try:
        auc = roc_auc_score(all_labels, all_probs)
    except:
        auc = np.nan

    return {
        "loss": avg_loss,
        "acc": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc
    }

In [32]:
import ast
import random
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
# =========================================
# 路徑設定
# =========================================
TEXT_TIME_PATH = Path(r"D:/時間序列/final_model_inputs_text_time_all/text_time_features.parquet")
VISION_NPY_PATH = Path(r"D:/時間序列/final_model_inputs_vision_all/vision_tensors.npy")
VISION_META_PATH = Path(r"D:/時間序列/final_model_inputs_vision_all/vision_metadata.csv")

# =========================================
# 參數設定
# =========================================
K = 128
BATCH_SIZE = 2
NUM_WORKERS = 0
SEED = 42
SEQUENCE_MODE = "recent"   # "recent" / "first" / "random"

CLIP_TOKENIZER_NAME = "openai/clip-vit-base-patch32"
CLIP_MAX_LENGTH = 77

In [33]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(SEED)


def ensure_list(x):
    if isinstance(x, list):
        return x
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, tuple):
        return list(x)
    if isinstance(x, str):
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, np.ndarray):
                return parsed.tolist()
            if isinstance(parsed, tuple):
                return list(parsed)
            if isinstance(parsed, list):
                return parsed
            return []
        except Exception:
            return []
    return []


def build_user_splits(usernames, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, seed=42):
    usernames = list(usernames)
    rng = random.Random(seed)
    rng.shuffle(usernames)

    n = len(usernames)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)

    train_users = usernames[:n_train]
    val_users = usernames[n_train:n_train + n_val]
    test_users = usernames[n_train + n_val:]

    return train_users, val_users, test_users

In [34]:
text_df = pd.read_parquet(TEXT_TIME_PATH)
vision_array = np.load(VISION_NPY_PATH, mmap_mode="r")
vision_meta = pd.read_csv(VISION_META_PATH)

text_df = text_df.reset_index(drop=True)
vision_meta = vision_meta.reset_index(drop=True)

assert len(text_df) == len(vision_array), "text_df 與 vision_array 筆數不一致"
assert len(text_df) == len(vision_meta), "text_df 與 vision_meta 筆數不一致"

# 若 text 欄位不存在，但 caption 在，就用 caption
if "text" not in text_df.columns:
    if "caption" in text_df.columns:
        text_df["text"] = text_df["caption"].fillna("").astype(str)
    else:
        raise ValueError("找不到 text 欄位，也沒有 caption 欄位可替代")

text_df["text"] = text_df["text"].fillna("").astype(str)

# 時間欄位
if "taken_at" in text_df.columns:
    text_df["taken_at"] = pd.to_datetime(text_df["taken_at"], utc=True, errors="coerce")

# 若沒有 relative_days，嘗試補
if "relative_days" not in text_df.columns:
    if "taken_at" not in text_df.columns:
        raise ValueError("沒有 relative_days，也沒有 taken_at 可用來計算")
    text_df["user_first_time"] = text_df.groupby("username")["taken_at"].transform("min")
    text_df["relative_days"] = (text_df["taken_at"] - text_df["user_first_time"]).dt.total_seconds() / 86400.0

# row_idx 對應 vision_array 的索引
text_df["row_idx"] = np.arange(len(text_df))

# 對齊檢查
if "post_id" in vision_meta.columns and "username" in vision_meta.columns:
    mismatch_mask = (
        (text_df["post_id"].astype(str) != vision_meta["post_id"].astype(str)) |
        (text_df["username"].astype(str) != vision_meta["username"].astype(str))
    )
    mismatch_count = int(mismatch_mask.sum())
    print(f"[對齊檢查] text_df vs vision_meta 不一致筆數: {mismatch_count}")
    if mismatch_count > 0:
        raise ValueError("text_df 與 vision_meta 的 username/post_id 對不齊，請先修正")

[對齊檢查] text_df vs vision_meta 不一致筆數: 0


In [35]:
# 使用 CLIP tokenizer 重新 tokenize 文字
clip_tokenizer = AutoTokenizer.from_pretrained(CLIP_TOKENIZER_NAME)

def clip_tokenize_text(text, max_length=77):
    enc = clip_tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_attention_mask=True
    )
    return enc["input_ids"], enc["attention_mask"]

clip_input_ids = []
clip_attention_masks = []

for text in text_df["text"].tolist():
    ids, mask = clip_tokenize_text(text, max_length=CLIP_MAX_LENGTH)
    clip_input_ids.append(ids)
    clip_attention_masks.append(mask)

text_df["clip_input_ids"] = clip_input_ids
text_df["clip_attention_mask"] = clip_attention_masks

In [36]:
def build_user_index(df):
    user_to_rows = {}
    for username, g in df.groupby("username"):
        if "taken_at" in g.columns:
            g = g.sort_values(["taken_at", "post_id"]).copy()
        else:
            g = g.sort_values(["post_id"]).copy()
        user_to_rows[username] = g["row_idx"].tolist()
    return user_to_rows

user_to_rows = build_user_index(text_df)
all_users = sorted(user_to_rows.keys())

print(f"總使用者數: {len(all_users)}")
print(f"總貼文數: {len(text_df)}")
print(f"vision_array shape: {vision_array.shape}")

總使用者數: 161
總貼文數: 76042
vision_array shape: (76042, 3, 224, 224)


In [37]:
class ContextVecNetDataset(Dataset):
    def __init__(
        self,
        text_df,
        vision_npy_path,
        vision_meta,
        user_to_rows,
        usernames,
        k=32,
        sequence_mode="recent"
    ):
        self.text_df = text_df
        self.vision_npy_path = vision_npy_path
        self.vision_array = np.load(self.vision_npy_path, mmap_mode="r")
        self.vision_meta = vision_meta
        self.user_to_rows = user_to_rows
        self.usernames = list(usernames)
        self.k = k
        self.sequence_mode = sequence_mode

        example_row = self.text_df.iloc[0]
        self.text_len = len(example_row["clip_input_ids"])
        self.vision_shape = tuple(self.vision_array[0].shape)  # (3, 224, 224)

    def __len__(self):
        return len(self.usernames)

    def _select_rows(self, rows):
        rows = list(rows)

        if len(rows) <= self.k:
            return rows

        if self.sequence_mode == "recent":
            return rows[-self.k:]
        elif self.sequence_mode == "first":
            return rows[:self.k]
        elif self.sequence_mode == "random":
            return sorted(random.sample(rows, self.k))
        else:
            raise ValueError(f"未知 sequence_mode: {self.sequence_mode}")

    def _get_image_valid_flag(self, ridx):
        if "source_type_used" in self.vision_meta.columns:
            source_type = str(self.vision_meta.iloc[ridx]["source_type_used"]).lower()
            return 0.0 if source_type == "black_image" else 1.0
        elif "load_success" in self.vision_meta.columns:
            return float(self.vision_meta.iloc[ridx]["load_success"])
        else:
            # 若沒有 metadata 可用，預設視為有效
            return 1.0

    def __getitem__(self, idx):
        username = self.usernames[idx]
        rows = self.user_to_rows[username]
        selected_rows = self._select_rows(rows)

        # 同一使用者 label 應一致
        user_label = int(self.text_df.loc[selected_rows[0], "user_label"])
        n_real = len(selected_rows)

        input_ids = []
        attention_mask = []
        vision_list = []
        relative_days = []
        image_valid_mask = []
        post_ids = []

        for ridx in selected_rows:
            row = self.text_df.iloc[ridx]

            input_ids.append(torch.tensor(row["clip_input_ids"], dtype=torch.long))
            attention_mask.append(torch.tensor(row["clip_attention_mask"], dtype=torch.long))

            vision_np = np.array(self.vision_array[ridx], dtype=np.float32)
            vision_list.append(torch.from_numpy(vision_np))

            relative_days.append(
                float(row["relative_days"]) if pd.notna(row["relative_days"]) else 0.0
            )

            image_valid_mask.append(self._get_image_valid_flag(ridx))
            post_ids.append(str(row["post_id"]))

        pad_n = self.k - n_real

        if pad_n > 0:
            pad_input_ids = torch.zeros(self.text_len, dtype=torch.long)
            pad_attention_mask = torch.zeros(self.text_len, dtype=torch.long)
            pad_vision = torch.zeros(self.vision_shape, dtype=torch.float32)

            for _ in range(pad_n):
                input_ids.append(pad_input_ids.clone())
                attention_mask.append(pad_attention_mask.clone())
                vision_list.append(pad_vision.clone())
                relative_days.append(0.0)
                image_valid_mask.append(0.0)
                post_ids.append("PAD")

        input_ids = torch.stack(input_ids, dim=0)               # [K, L]
        attention_mask = torch.stack(attention_mask, dim=0)     # [K, L]
        vision_tensor = torch.stack(vision_list, dim=0)         # [K, 3, 224, 224]

        relative_days = torch.tensor(relative_days, dtype=torch.float32)      # [K]
        post_mask = torch.tensor([1] * n_real + [0] * pad_n, dtype=torch.float32)  # [K]
        image_valid_mask = torch.tensor(image_valid_mask, dtype=torch.float32)      # [K]

        return {
            "username": username,
            "label": torch.tensor(user_label, dtype=torch.long),
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "vision_tensor": vision_tensor,
            "relative_days": relative_days,
            "post_mask": post_mask,
            "image_valid_mask": image_valid_mask,
            "num_real_posts": torch.tensor(n_real, dtype=torch.long),
            "post_ids": post_ids,
        }

In [38]:
train_users, val_users, test_users = build_user_splits(
    all_users,
    train_ratio=0.7,
    val_ratio=0.15,
    test_ratio=0.15,
    seed=SEED
)

print(f"Train users: {len(train_users)}")
print(f"Val users:   {len(val_users)}")
print(f"Test users:  {len(test_users)}")

Train users: 112
Val users:   24
Test users:  25


In [39]:
train_dataset = ContextVecNetDataset(
    text_df=text_df,
    vision_npy_path=VISION_NPY_PATH,
    vision_meta=vision_meta,
    user_to_rows=user_to_rows,
    usernames=train_users,
    k=K,
    sequence_mode=SEQUENCE_MODE
)

val_dataset = ContextVecNetDataset(
    text_df=text_df,
    vision_npy_path=VISION_NPY_PATH,
    vision_meta=vision_meta,
    user_to_rows=user_to_rows,
    usernames=val_users,
    k=K,
    sequence_mode=SEQUENCE_MODE
)

test_dataset = ContextVecNetDataset(
    text_df=text_df,
    vision_npy_path=VISION_NPY_PATH,
    vision_meta=vision_meta,
    user_to_rows=user_to_rows,
    usernames=test_users,
    k=K,
    sequence_mode=SEQUENCE_MODE
)

In [40]:
def contextvecnet_collate_fn(batch):
    return {
        "username": [x["username"] for x in batch],
        "label": torch.stack([x["label"] for x in batch], dim=0),                     # [B]
        "input_ids": torch.stack([x["input_ids"] for x in batch], dim=0),             # [B, K, L]
        "attention_mask": torch.stack([x["attention_mask"] for x in batch], dim=0),   # [B, K, L]
        "vision_tensor": torch.stack([x["vision_tensor"] for x in batch], dim=0),     # [B, K, 3, 224, 224]
        "relative_days": torch.stack([x["relative_days"] for x in batch], dim=0),     # [B, K]
        "post_mask": torch.stack([x["post_mask"] for x in batch], dim=0),             # [B, K]
        "image_valid_mask": torch.stack([x["image_valid_mask"] for x in batch], dim=0),  # [B, K]
        "num_real_posts": torch.stack([x["num_real_posts"] for x in batch], dim=0),   # [B]
        "post_ids": [x["post_ids"] for x in batch],
    }

In [41]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=contextvecnet_collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=contextvecnet_collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=contextvecnet_collate_fn
)

print("len(train_loader) =", len(train_loader))
print("len(val_loader)   =", len(val_loader))
print("len(test_loader)  =", len(test_loader))

len(train_loader) = 56
len(val_loader)   = 12
len(test_loader)  = 13


In [42]:
batch = next(iter(train_loader))

print("=== Batch keys ===")
print(batch.keys())

print("\n=== Shapes ===")
print("label:", batch["label"].shape)
print("input_ids:", batch["input_ids"].shape)
print("attention_mask:", batch["attention_mask"].shape)
print("vision_tensor:", batch["vision_tensor"].shape)
print("relative_days:", batch["relative_days"].shape)
print("post_mask:", batch["post_mask"].shape)
print("image_valid_mask:", batch["image_valid_mask"].shape)
print("num_real_posts:", batch["num_real_posts"].shape)

print("\n=== Example usernames ===")
print(batch["username"][:2])

print("\n=== Example labels ===")
print(batch["label"][:2])

print("\n=== Example post_mask ===")
print(batch["post_mask"][:2])

print("\n=== Example image_valid_mask ===")
print(batch["image_valid_mask"][:2])

=== Batch keys ===
dict_keys(['username', 'label', 'input_ids', 'attention_mask', 'vision_tensor', 'relative_days', 'post_mask', 'image_valid_mask', 'num_real_posts', 'post_ids'])

=== Shapes ===
label: torch.Size([2])
input_ids: torch.Size([2, 128, 77])
attention_mask: torch.Size([2, 128, 77])
vision_tensor: torch.Size([2, 128, 3, 224, 224])
relative_days: torch.Size([2, 128])
post_mask: torch.Size([2, 128])
image_valid_mask: torch.Size([2, 128])
num_real_posts: torch.Size([2])

=== Example usernames ===
['neko_wumiao', 'lingluo_hope']

=== Example labels ===
tensor([1, 1])

=== Example post_mask ===
tensor([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.

In [43]:
# 5. 建立模型、optimizer、scheduler
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device =", device)

pos_weight = compute_pos_weight_from_loader(train_loader)

model = ContextVecNetApprox(
    clip_model_name="openai/clip-vit-base-patch32",
    clip_embed_dim=512,
    d_model=128,
    nhead=8,
    ff_dim=256,
    num_cross_layers=4,
    num_transformer_layers=2,
    time_dim=16,
    context_len=2,
    dropout=0.1,
    freeze_clip=True,
).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-5,
    weight_decay=1e-2
)

# triangular cyclic LR, max_lr = 10 * base_lr
scheduler = torch.optim.lr_scheduler.CyclicLR(
    optimizer,
    base_lr=1e-5,
    max_lr=1e-4,
    step_size_up=max(1, len(train_loader) // 2),
    mode="triangular",
    cycle_momentum=False
)

device = cuda
Train labels -> pos=39, neg=73


In [47]:
batch = next(iter(train_loader))

with torch.no_grad():
    logits = model(
        input_ids=batch["input_ids"].to(device),
        attention_mask=batch["attention_mask"].to(device),
        vision_tensor=batch["vision_tensor"].to(device),
        relative_days=batch["relative_days"].to(device),
        post_mask=batch["post_mask"].to(device),
        image_valid_mask=batch["image_valid_mask"].to(device) if "image_valid_mask" in batch else batch["post_mask"].to(device),
    )

print("logits shape:", logits.shape)
print("logits has nan:", torch.isnan(logits).any().item())
print("logits[:5]:", logits[:5])

logits shape: torch.Size([2])
logits has nan: False
logits[:5]: tensor([-1.1332, -0.1113], device='cuda:0')


In [48]:
EPOCHS = 10

best_val_f1 = -1
best_state = None

for epoch in range(1, EPOCHS + 1):
    print(f"\n========== Epoch {epoch}/{EPOCHS} ==========")

    train_metrics = train_one_epoch_contextvecnet(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        scheduler=scheduler,
        epoch_idx=epoch,
        log_every=5,
        grad_clip=1.0
    )

    val_metrics = evaluate_contextvecnet(
        model=model,
        loader=val_loader,
        criterion=criterion,
        device=device,
        epoch_idx=epoch,
        split_name="Val"
    )

    print(f"[Train][Epoch {epoch}] {train_metrics}")
    print(f"[Val][Epoch {epoch}]   {val_metrics}")

    if val_metrics["f1"] > best_val_f1:
        best_val_f1 = val_metrics["f1"]
        best_state = {k: v.cpu() for k, v in model.state_dict().items()}
        print(f"*** Best model updated at epoch {epoch}, val_f1={best_val_f1:.4f}")

if best_state is not None:
    model.load_state_dict(best_state)

test_metrics = evaluate_contextvecnet(
    model=model,
    loader=test_loader,
    criterion=criterion,
    device=device,
    epoch_idx=0,
    split_name="Test"
)

print("\n[Test]", test_metrics)


========== Epoch 1/10 ==========


Train Epoch 1:   0%|          | 0/56 [00:00<?, ?it/s]

[Train][Epoch 1] step 1: logits 出現 NaN


ValueError: logits contains NaN

In [51]:
import pandas as pd

df = pd.read_csv(r"D:\時間序列\window_level_plus_prev_context_8\window_dataset.csv")

print(df["selected_post_count"].describe())
print(df["prev_context_count"].describe())
print(df["padding_needed"].describe())

print("\nsource_rule counts:")
print(df["source_rule"].value_counts())

print("\nwindow_label x padding_needed mean:")
print(df.groupby("window_label")["padding_needed"].mean())

count    542.000000
mean       7.433579
std        1.242433
min        3.000000
25%        8.000000
50%        8.000000
75%        8.000000
max        8.000000
Name: selected_post_count, dtype: float64
count    542.000000
mean       1.859779
std        1.977387
min        0.000000
25%        0.000000
50%        1.000000
75%        4.000000
max        5.000000
Name: prev_context_count, dtype: float64
count    542.000000
mean       0.566421
std        1.242433
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        5.000000
Name: padding_needed, dtype: float64

source_rule counts:
14d_plus_prev_context    542
Name: source_rule, dtype: int64

window_label x padding_needed mean:
window_label
0    0.592138
1    0.488889
Name: padding_needed, dtype: float64
